## Tahap 0.1 — Lingkungan

**Dasar naskah:** Bab III (lingkungan komputasi: Google Colab). GPU memakai **A100**, menggantikan T4 yang tertulis di Bab III (keputusan pengguna 21 Oktober 2026; naskah perlu disesuaikan). `CLAUDE.md` Bagian 3.1, 3.3, 3.4, dan 10.

**Package:** `os`, `sys`, `subprocess`, `importlib`, `warnings`, `pathlib`, `json`, `time`, `platform` (bawaan Python); `numpy`, `pandas`, `torch`, `transformers`, `matplotlib` (di-*import*); `scikit-learn`, `scipy`, `openpyxl`, `pyarrow` (dipasang bila belum ada, versinya dicatat); `google.colab.drive` (*mount* Google Drive).

Jalankan sel 1/2 lalu sel 2/2 **paling awal** setiap kali *runtime* dimulai atau di-*restart*, sebelum Tahap 0.2.

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 1/2: variabel lingkungan, folder kerja, pemasangan package) ══
# Package: os, sys, subprocess, importlib, warnings, pathlib (bawaan Python); google.colab.drive
import os, sys, subprocess, importlib.util, warnings
from pathlib import Path
from google.colab import drive

# 1) Variabel lingkungan, diatur SEBELUM torch/transformers di-import (3.3–3.4)
_cuda_dulu = "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized()
_cublas_dulu = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
os.environ.update({
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",                                          # determinisme cuBLAS
    "TOKENIZERS_PARALLELISM": "false",                                             # senyap: tokenizer
    "TRANSFORMERS_VERBOSITY": "error", "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",   # senyap: transformers
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_DATASETS_DISABLE_PROGRESS_BARS": "1", # senyap: HF Hub, datasets
    "HF_HUB_VERBOSITY": "error",                                                   # senyap: log HF Hub (token)
    "TQDM_DISABLE": "1",                                                           # senyap: semua tqdm
    "USE_TF": "0", "TF_CPP_MIN_LOG_LEVEL": "3",                                    # jangan muat TensorFlow
})
warnings.filterwarnings("ignore")

# 2) Google Drive dan struktur folder (3.1); folder versi lama tidak disentuh
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        raise RuntimeError("Mount gagal: pilih akun yang menyimpan folder skripsi, centang SEMUA izin, ulangi sel.") from e
AKAR = Path("/content/drive/MyDrive/skripsi_wondr_v3")
AKAR.mkdir(exist_ok=True)                         # gagal jika MyDrive belum ter-mount
for sub in ["data", "lib", "hasil", "anotasi", "bab4/tabel", "bab4/gambar", "log"]:
    (AKAR / sub).mkdir(parents=True, exist_ok=True)
(AKAR / "lib" / "__init__.py").touch()            # agar "from lib.wondr_teks import ..." jalan
os.chdir(AKAR)                                     # agar %%writefile cukup memakai lib/...
if str(AKAR) not in sys.path:
    sys.path.insert(0, str(AKAR))

# 3) Pasang hanya package yang belum ada (di-import di sel 2/2)
PAKET = {"numpy": "numpy", "pandas": "pandas", "torch": "torch", "transformers": "transformers",
         "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
         "openpyxl": "openpyxl", "pyarrow": "pyarrow"}            # nama modul -> nama pip
_dipasang = [pip for mod, pip in PAKET.items() if importlib.util.find_spec(mod) is None]
if _dipasang:
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_dipasang], capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + _r.stderr[-2000:])

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 2/2: import, GPU, dan log versi) ══
# Package: importlib, json, time, platform (bawaan Python); numpy, pandas, torch, transformers,
#          matplotlib (di-import); scikit-learn, scipy, openpyxl, pyarrow (versinya dicatat)
import importlib.metadata, json, time, platform
t0 = time.perf_counter()
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, transformers
from huggingface_hub.utils import logging as hf_logging

# 1) Matikan kebisingan yang tersisa setelah import
transformers.logging.set_verbosity_error()        # sembunyikan "Some weights ... newly initialized"
transformers.logging.disable_progress_bar()       # progress bar unduhan model
hf_logging.set_verbosity_error()                  # sembunyikan peringatan HF Hub "unauthenticated requests"
plt.rcParams.update({"savefig.dpi": 300, "savefig.bbox": "tight"})   # PNG 300 dpi (3.1)

# 2) GPU dan versi -> log/versi_package.json (acuan = sesi A100 pertama)
GPU_PENELITIAN = "A100"                            # Keputusan pengguna 21-10-2026, menggantikan T4
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada"
mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if gpu != "tidak ada" else 0.0
sesi = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "gpu": gpu,
        "gpu_memori_gb": round(mem, 2), "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "package": {p: importlib.metadata.version(p) for p in [*PAKET.values(), "tokenizers", "huggingface-hub"]}}
log = AKAR / "log" / "versi_package.json"
lama = json.loads(log.read_text()) if log.exists() else {}
pertama = lama["pertama"] if GPU_PENELITIAN in lama.get("pertama", {}).get("gpu", "") else sesi
log.write_text(json.dumps({"pertama": pertama, "terbaru": sesi}, indent=2))
beda = [k for k in ("torch", "transformers", "tokenizers") if pertama["package"][k] != sesi["package"][k]]
beda += ["GPU"] if pertama["gpu"] != gpu else []

# 3) Ringkasan + cek (judul() dan cek() baru dibuat di Tahap 0.2)
xlsx, koma = AKAR / "data" / "data_wonder_bybni_3.xlsx", lambda x: f"{x:.1f}".replace(".", ",")
cublas_ok = not _cuda_dulu or _cublas_dulu == ":4096:8"
print(f"══════════ TAHAP 0.1 — LINGKUNGAN ══════════\nMasukan : data/{xlsx.name}\nKeluaran: log/{log.name}")
print(f"{'─' * 48}\nFolder kerja : {AKAR}\nPython       : {sesi['python']}\n"
      f"GPU          : {gpu} ({koma(mem)} GB) · CUDA {sesi['cuda']} · cuDNN {sesi['cudnn']}\n\n{'Package':<17}Versi")
print("\n".join(f"{k:<17}{v}{'  (baru dipasang)' * (k in _dipasang)}" for k, v in sesi["package"].items()))
print(f"{'─' * 48}\nCek otomatis:")
for nama, ok, aktual, target in [
        ("berkas data ada", xlsx.exists(), "ya" if xlsx.exists() else "tidak", "ya"),
        ("GPU", GPU_PENELITIAN in gpu, gpu, f"memuat {GPU_PENELITIAN}"),
        ("CUBLAS_WORKSPACE_CONFIG diatur sebelum CUDA aktif", cublas_ok, "ya" if cublas_ok else "tidak, restart runtime", "ya"),
        ("versi torch/transformers/tokenizers/GPU", not beda, "berubah: " + ", ".join(beda) if beda else "sama", "sama")]:
    print(f"  {'✅' if ok else '⚠️'} {nama} = {aktual} (target {target})")
print(f"Waktu: {koma(time.perf_counter() - t0)} detik")

## Tahap 0.2 — Konfigurasi dan fungsi bantu

**Dasar naskah:** `CLAUDE.md` Bagian 4 (`CFG`, dengan asal setiap nilai di komentar) dan Bagian 6 (`TARGET`); Bagian 3.4 (`atur_seed()`). Keputusan teknis KT-1 sampai KT-4 disetujui pengguna pada 21 Oktober 2026 (lihat `CLAUDE.md` Bagian 10).

**Package:** `json`, `math`, `os`, `random`, `time`, `pathlib`, `importlib` (bawaan Python); `numpy`; `torch` dan `transformers` (di `atur_seed()`); `matplotlib` (di `simpan_gambar()`); `pandas` (lewat `DataFrame.to_csv` di `simpan_tabel()`).

Sel 1/2 menulis `lib/bantu.py`, yang isinya fungsi saja. Sel 2/2 membuat `PATH`, `CFG`, dan `TARGET`. Setelah *runtime* di-*restart*, jalankan Tahap 0.1 (2 sel) lalu Tahap 0.2 (2 sel).

In [ ]:
%%writefile lib/bantu.py
# lib/bantu.py -- fungsi bantu untuk semua subtahap (Tahap 0.2; KT-4 disetujui 21-10-2026).
# Isinya fungsi saja; PATH, CFG, dan TARGET dibuat di sel 2/2 Tahap 0.2.
import json, math, os, random, time
from pathlib import Path
import numpy as np

__all__ = ["fmt_id", "rel", "mulai", "judul", "garis", "cek", "selesai",
           "simpan_tabel", "simpan_gambar", "simpan_json", "baca_json", "atur_seed"]
_AKAR = Path(__file__).resolve().parents[1]          # folder kerja .../skripsi_wondr_v3
_STATUS = {"t0": None, "cek_mulai": False}
_LEBAR = 48                                           # lebar garis pemisah blok ringkasan


def _angka(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def fmt_id(x, d=None):
    """Format angka Indonesia: 7204 -> '7.204'; 0.836 -> '0,8360'. Selain angka -> str(x)."""
    if not _angka(x):
        return str(x)
    if d is None:
        d = 0 if isinstance(x, (int, np.integer)) else 4
    return f"{x:,.{d}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def _tampil(v, d=None):
    """Nilai untuk ringkasan: bilangan bulat tanpa desimal, pecahan d desimal, daftar dipisah ' · '."""
    if isinstance(v, np.ndarray):
        v = v.tolist()
    if isinstance(v, dict):
        return ", ".join(f"{k}: {_tampil(x, d)}" for k, x in v.items())
    if isinstance(v, (list, tuple)):
        return " · ".join(f"[{_tampil(x, d)}]" if isinstance(x, (list, tuple)) else _tampil(x, d) for x in v)
    return fmt_id(v) if isinstance(v, (int, np.integer)) else fmt_id(v, d)


def rel(p):
    """Path relatif terhadap folder kerja, untuk ditampilkan."""
    try:
        return str(Path(p).resolve().relative_to(_AKAR))
    except ValueError:
        return str(p)


def garis():
    print("─" * _LEBAR)


def mulai():
    """Dipanggil di awal sel: mulai menghitung waktu."""
    _STATUS["t0"] = time.perf_counter()


def judul(no, nama, masukan=None, keluaran=None):
    """Kepala blok ringkasan; masukan/keluaran berupa str atau list of str."""
    _STATUS["cek_mulai"] = False
    print(f"{'═' * 10} TAHAP {no} — {nama.upper()} {'═' * 10}")
    for label, isi in (("Masukan ", masukan), ("Keluaran", keluaran)):
        if isi:
            isi = [isi] if isinstance(isi, str) else list(isi)
            print(f"{label}: " + f"\n{' ' * 10}".join(isi))
    garis()


def _sama(a, t, tol):
    a = a.tolist() if isinstance(a, np.ndarray) else a
    t = t.tolist() if isinstance(t, np.ndarray) else t
    if _angka(a) and _angka(t):
        return abs(float(a) - float(t)) <= tol
    if isinstance(a, dict) and isinstance(t, dict):
        return a.keys() == t.keys() and all(_sama(a[k], t[k], tol) for k in t)
    if isinstance(a, (list, tuple)) and isinstance(t, (list, tuple)):
        return len(a) == len(t) and all(_sama(x, y, tol) for x, y in zip(a, t))
    return bool(a == t)


def cek(nama, aktual, target, toleransi=0, cara="=", d=None):
    """Cetak ✅/⚠️ beserta nilai aktual dan target, tanpa assert. cara: '=', '>', '>=', '<', '<='."""
    if not _STATUS["cek_mulai"]:
        garis()
        print("Cek otomatis:")
        _STATUS["cek_mulai"] = True
    if d is None and toleransi > 0:
        d = max(0, round(-math.log10(2 * toleransi)))  # 0,05 -> 1 desimal; 5e-5 -> 4 desimal
    if cara == "=":
        ok = _sama(aktual, target, toleransi)
    else:
        a, t = float(aktual), float(target)
        ok = {">": a > t, ">=": a >= t, "<": a < t, "<=": a <= t}[cara]
    tanda = "" if cara == "=" else cara + " "
    print(f"  {'✅' if ok else '⚠️'} {nama} = {_tampil(aktual, d)} (target {tanda}{_tampil(target, d)})")
    return bool(ok)


def selesai():
    """Baris terakhir blok ringkasan: lama sel berjalan sejak mulai()."""
    dt = time.perf_counter() - _STATUS["t0"] if _STATUS["t0"] else 0.0
    teks = (f"{fmt_id(dt, 1)} detik" if dt < 120 else
            f"{fmt_id(dt / 60, 1)} menit" if dt < 7200 else f"{fmt_id(dt / 3600, 2)} jam")
    print(f"Waktu: {teks}")


def simpan_tabel(df, nama, index=False):
    """CSV untuk Bab IV (angka mentah) -> bab4/tabel/<nama>.csv; mengembalikan path-nya."""
    p = _AKAR / "bab4" / "tabel" / f"{nama}.csv"
    df.to_csv(p, index=index)
    return p


def simpan_gambar(fig, nama):
    """PNG 300 dpi -> bab4/gambar/<nama>.png, lalu ditampilkan satu kali."""
    import matplotlib.pyplot as plt
    p = _AKAR / "bab4" / "gambar" / f"{nama}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return p


def _ke_json(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, (np.ndarray, Path)):
        return o.tolist() if isinstance(o, np.ndarray) else str(o)
    raise TypeError(f"tidak bisa disimpan ke JSON: {type(o).__name__}")


def simpan_json(obj, path):
    """Tulis JSON secara atomik (berkas .tmp lalu diganti), aman bila Colab terputus."""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_ke_json), encoding="utf-8")
    os.replace(tmp, path)
    return path


def baca_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atur_seed(s):
    """Satu seed untuk semua sumber keacakan pelatihan (3.4), ditambah KT-2 dan KT-3 (21-10-2026)."""
    import torch, transformers
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    transformers.set_seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)         # KT-2: operasi GPU tak-deterministik -> error
    torch.backends.cuda.matmul.allow_tf32 = False    # KT-3: FP32 penuh; API lama ikut mengatur API baru,
    torch.backends.cudnn.allow_tf32 = False          #       API baru saja tidak mematikan TF32 konv/RNN cuDNN


def uji_bantu(s):
    """Uji Tahap 0.2: atur_seed(s) dua kali -> angka acak sama; status mode deterministik dan TF32."""
    import torch
    atur_seed(s); a = np.random.rand(3), torch.rand(3)
    atur_seed(s); b = np.random.rand(3), torch.rand(3)
    return {"ulang_sama": all(np.array_equal(x, y) for x, y in zip(a, b)),
            "deterministik": torch.are_deterministic_algorithms_enabled(),
            "tf32": [torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32]}

In [ ]:
# ══ Tahap 0.2 — Konfigurasi dan fungsi bantu (sel 2/2: PATH, CFG, TARGET) ══
# Package: importlib (bawaan Python); lib.bantu (fungsi bantu dari sel 1/2)
import importlib, lib.bantu
importlib.reload(lib.bantu)                         # pakai isi lib/bantu.py yang terbaru
from lib.bantu import *
mulai()

# 1) Lokasi folder dan berkas (3.1)
PATH = {"akar": AKAR, **{k: AKAR / k for k in ("data", "lib", "hasil", "anotasi", "log")},
        "tabel": AKAR / "bab4/tabel", "gambar": AKAR / "bab4/gambar", "xlsx": AKAR / "data/data_wonder_bybni_3.xlsx"}

# 2) Konfigurasi tetap (Bagian 4) -- jangan diubah
CFG = {
    "gpu": "A100",                                                  # Keputusan pengguna 21-10-2026
    "checkpoint": "indobenchmark/indobert-base-p1",                 # Bab III tahap 7; Wilie dkk. (2020)
    "tau": 4, "kelas": {0: "Non-Positif", 1: "Positif"},            # pers. (2.1)-(2.2): rating >= 4 -> 1
    "proporsi": {"latih": 0.70, "validasi": 0.15, "uji": 0.15},     # Bab III tahap 6
    "seed_bagi": 42, "kelompok_panjang": ("1-2", "3-10", ">10"),    # Bab III tahap 6
    "q": 100, "b": 32,                                              # pers. (2.9): L_max dari data latih saja
    "cl_lipatan": 5, "cl_seed_lipatan": 42, "cl_seed_latih": 42,    # Bab III tahap 8; Keputusan teknis
    "seed_anotasi": 2026,                                           # Keputusan: urutan lembar anotasi (tahap 9)
    "B": 16, "E": 4, "p": 2, "v": 1.0,                              # Tabel 3.1: batch, epoch, patience, norma
    "adam_betas": (0.9, 0.999), "adam_eps": 1e-8,                   # pers. (2.18); bawaan PyTorch
    "default": {"lr": 2e-5, "wr": 0.1, "wd": 0.01},                 # Tabel 3.1; Devlin dkk.; Sun dkk.
    "ruang": {"lr": (1e-5, 5e-5), "wr": (0.0, 0.2), "wd": (0.0, 0.3)},   # Tabel 3.1; pers. (2.27); lr skala log
    "seed_fitness": (42, 43),                                       # Bab III tahap 10; pers. (2.25)
    "rs_M": 50, "rs_seed": 2024,                                    # Bab III tahap 11
    "pso": {"s": 10, "G_max": 4, "omega": 0.729, "c1": 1.49445, "c2": 1.49445, "V_max": 1.0, "seed": 7},  # Tabel 3.2
    "seed_akhir": (42, 43, 44),                                     # Bab III tahap 13
    "R": 10_000, "seed_bootstrap": 42,                              # Bab III tahap 15; pers. (2.41)-(2.43)
    "alpha": 0.05, "m_uji": 3,                                      # pers. (2.44)-(2.45): alpha' = 0,05/3
    "kembaran_dekat": (339, 595, 2854, 3946, 4128),                 # Data (temuan pembagian)
}

# 3) Angka target (Bagian 6; nomor tahap mengikuti Bab III, lihat CLAUDE.md Bagian 10)
TARGET = {
    "1.1": {"baris": 7204, "kolom": ["nama", "rating", "ulasan", "tanggal", "likes", "balasan_dev", "versi"],
            "tanggal": ("2026-07-19", "2026-09-11"), "versi": ["1.6.0"],
            "per_bulan": {"2026-07": 1747, "2026-08": 3899, "2026-09": 1558}},
    "1.2": {"kosong": {"balasan_dev": 4834}, "dibalas_pct": {"1-4": 95.0, "5": 1.0}, "likes_nol_pct": 95.1,
            "spearman": -0.2483, "r3_panjang": (237, 366), "median_kata": [15, 15, 11, 3, 2], "emoji_simbol": 404,
            "huruf_ulang": (227, 3.2), "baca_ulang": 634, "duplikat_pct": 34.8, "label_ganda": (33, 1968),
            "nama_unik": 7153},
    "2": {"baris": 7204, "kolom": ["review_id", "ulasan", "rating", "tanggal", "versi"]},
    "3": {"per_rating": {1: 1305, 2: 283, 3: 366, 4: 492, 5: 4758}, "positif": (5250, 72.88),
          "non_positif": (1954, 27.12)},
    "4": {"contoh_bab2": ["baguss bgt! tp kadang2 error, nunggu lamaa", "mantap"],
          "berubah": [55, 404, 1518, 227, 634, 403], "kosong": 63, "kosong_rating": {5: 57, 4: 6}, "hasil": 7141},
    "5": {"kunci": 4372, "kunci_seri": 8, "ulasan_seri": 16, "dokumen": 4364, "dibuang": (2777, 2688, 73, 16),
          "positif_pct": (72.6, 57.3)},
    "6.1": {"grup": 4293, "latih": (3051, 1303, 1748), "validasi": (657, 282, 375), "uji": (656, 280, 376)},
    "6.2": {"bocor": 0, "kembaran_di_uji": 5, "non_positif_pct": (42.7, 42.9, 42.7), "p_label": 0.9945,
            "p_panjang": 0.9998, "panjang_pct": {"latih": (14.0, 47.1, 38.9), "validasi": (14.3, 46.9, 38.8),
                                                 "uji": (14.0, 47.1, 38.9)}},
    "7": {"L_max": 128, "terpotong": {"latih": 0, "validasi": 0, "uji": 1}, "terpotong_id": [2741]},
    "8.1": {"T": 764, "T_w": 76},                                   # n = 3.051, rho = 0,1
    "8.3": {"t0": 0.6167, "t1": 0.6667, "C": [[2, 1], [1, 2]], "kandidat": [3, 6]},    # contoh Tabel 2.9
    "9.2": {"kappa_contoh": 0.70, "toleransi": 1e-12},              # contoh Bab II 2.3.5
    "10": {"T": 764, "T_w": 76}, "11": {"kandidat": 50, "pelatihan": 100},
    "12.1": {"iter1": (5.6237, 3.6355), "iter2": (2.6718, 3.3698), "gbest_iter2": 9.8923,
             "toleransi": 5e-4, "evaluasi": 50},                    # contoh Tabel 2.17
    "13": {"pelatihan": 9, "toleransi": 1e-4}, "14.1": {"n_uji": 656},
    "15": {"f1_contoh": 0.6703, "persentil": (0.8333, 99.1667), "toleransi": 1e-12},   # contoh Tabel 2.19
}

# 4) Uji singkat fungsi bantu, lalu ringkasan
u = lib.bantu.uji_bantu(CFG["seed_fitness"][0])
judul("0.2", "Konfigurasi dan fungsi bantu", masukan="lib/bantu.py", keluaran="PATH, CFG, TARGET (di memori)")
print(f"CFG {len(CFG)} entri · TARGET {len(TARGET)} subtahap · PATH {len(PATH)} lokasi · "
      f"fungsi bantu {len(lib.bantu.__all__)}")
cek("fmt_id(7204) · fmt_id(0.836)", [fmt_id(7204), fmt_id(0.836)], ["7.204", "0,8360"])
cek("atur_seed() dua kali, angka acak sama", "ya" if u["ulang_sama"] else "tidak", "ya")
cek("mode deterministik PyTorch (KT-2)", "aktif" if u["deterministik"] else "mati", "aktif")
cek("TF32 matmul · cuDNN (KT-3)", ["nyala" if x else "mati" for x in u["tf32"]], ["mati", "mati"])
selesai()

## Tahap 1.1 — Muat data mentah

**Dasar naskah:** Bab III tahap 1 (Pengumpulan Data) dan subbab 3.1: ulasan 19 Juli–11 September 2026, versi aplikasi 1.6.0, disimpan apa adanya dalam format Excel dan tidak diubah. `review_id` dibuat di Tahap 2 sesuai Bab III tahap 2 (`CLAUDE.md` Bagian 10, keputusan no. 7).

**Package:** `pandas` (baca .xlsx, olah tabel), `openpyxl` (mesin baca .xlsx), `pyarrow` (mesin simpan .parquet); `lib.bantu` (fungsi bantu Tahap 0.2).

In [ ]:
# ══ Tahap 1.1 — Muat data mentah ══
# Package: pandas (baca .xlsx, olah tabel), openpyxl (mesin baca .xlsx), pyarrow (mesin simpan .parquet)
import pandas as pd
mulai()
T = TARGET["1.1"]
BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
         "Agustus", "September", "Oktober", "November", "Desember"]
tgl_id = lambda t: f"{t.day} {BULAN[t.month - 1]} {t.year}"       # 2026-07-19 -> "19 Juli 2026"

# 1) Baca data mentah (.xlsx)
mentah = pd.read_excel(PATH["xlsx"], engine="openpyxl")

# 2) Buang kolom indeks bawaan Excel (Unnamed*); isi tidak diubah, review_id dibuat di Tahap 2
buang = [c for c in mentah.columns if str(c).startswith("Unnamed")]
df = mentah.drop(columns=buang)

# 3) Simpan, lalu pastikan berkas yang dibaca ulang identik
keluar = PATH["hasil"] / "tahap01_mentah.parquet"
df.to_parquet(keluar, index=False)
identik = df.equals(pd.read_parquet(keluar))

# 4) Ringkasan + cek
tgl = pd.to_datetime(df["tanggal"])                                # hanya untuk ringkasan; kolom asli tetap
per_bulan = {k: int(v) for k, v in tgl.dt.strftime("%Y-%m").value_counts().sort_index().items()}
versi = sorted(df["versi"].astype(str).unique())
judul("1.1", "Muat data mentah", masukan=f"{rel(PATH['xlsx'])} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"Kolom dibuang  : {', '.join(map(str, buang)) or '-'}")
print(f"Rentang tanggal: {tgl_id(tgl.min())} – {tgl_id(tgl.max())}")
print(f"Versi aplikasi : {', '.join(versi)}\n\n{'Bulan':<16}{'Ulasan':>7}")
for k, v in per_bulan.items():
    print(f"{BULAN[int(k[5:]) - 1] + ' ' + k[:4]:<16}{fmt_id(v):>7}")
print(f"{'Total':<16}{fmt_id(sum(per_bulan.values())):>7}")
cek("jumlah baris", len(df), T["baris"])
cek("nama dan urutan kolom", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(map(str, df.columns)), "sesuai")
cek("rentang tanggal", (str(tgl.min().date()), str(tgl.max().date())), T["tanggal"])
cek("nilai unik versi", versi, T["versi"])
cek("ulasan per bulan", per_bulan, T["per_bulan"])
cek("parquet identik dengan isi .xlsx", "ya" if identik else "tidak", "ya")
selesai()

## Tahap 1.2 — Tulis `lib/wondr_teks.py` dan audit kualitas data *(audit: program saja; data tidak diubah)*

**Dasar naskah:** Bab III tahap 1 (data mentah tidak diubah) dan tahap 2 (alasan membuang `nama`, `balasan_dev`, `likes`); pola teks sama dengan Tahap 4 (Bab II 2.2.2, 2.2.4, 2.2.5). Audit tidak dibahas di naskah.

**Package:** `json`, `importlib` (bawaan Python); `pandas` (olah tabel); `scipy.stats.spearmanr` (korelasi *likes*–*rating*); `lib.wondr_teks` (pola `EMO`, `SIMBOL`, `HURUF`, `BACA`; memakai `re`, `unicodedata`, `numpy`, `pandas`); `lib.bantu`.

Sel 1/2 menulis kode acuan Bagian 7 `CLAUDE.md` **persis** ke `lib/wondr_teks.py` (fungsi saja). Sel 2/2 mengaudit data tanpa mengubahnya.

In [ ]:
%%writefile lib/wondr_teks.py
# lib/wondr_teks.py -- kode acuan Bagian 7 CLAUDE.md, disalin persis (Tahap 1.2).
# Dipakai Tahap 1.2 (audit), 4 (pre-processing), 5 (deduplikasi), dan 6 (pembagian data).
import re, unicodedata
import numpy as np, pandas as pd

# -- Pola teks (Tahap 1.2 dan Tahap 4 memakai pola yang sama) --
EMO = re.compile("[\U0001F000-\U0001FAFF\U0001F1E6-\U0001F1FF\u2600-\u27BF\u2B00-\u2BFF"
                 "\u2300-\u23FF\u2190-\u21FF\u2900-\u297F\u3030\u303D\u3297\u3299"
                 "\u00A9\u00AE\u203C\u2049\u2122\u2139\u24C2\u25AA-\u25FE\uFE0F\u200D"
                 "\U000E0020-\U000E007F]")
SIMBOL = re.compile("[\u2600-\u27BF\u2B00-\u2BFF\uFE0F\u200D\U0001F3FB-\U0001F3FF]")
HURUF = re.compile(r"([^\W\d_])\1{2,}")      # huruf yang berulang >= 3 kali (hanya huruf)
BACA = re.compile(r"[!?.,;:]{2,}")           # tanda baca berurutan >= 2

LANGKAH = [  # (nama, fungsi) -- urutan Tahap 4, jangan diubah
    ("1. Normalisasi Unicode NFKC", lambda s: unicodedata.normalize("NFKC", s)),
    ("2. Hapus emoji dan simbol", lambda s: SIMBOL.sub(" ", EMO.sub(" ", s))),
    ("3. Case folding", lambda s: s.lower()),
    ("4. Huruf berulang >=3 -> 2", lambda s: HURUF.sub(r"\1\1", s)),
    ("5. Tanda baca berulang -> 1", lambda s: BACA.sub(lambda m: m.group(0)[0], s)),
    ("6. Rapikan spasi", lambda s: re.sub(r"\s+", " ", s).strip()),
]

def bersihkan(s):
    s = str(s)
    for _, f in LANGKAH:
        s = f(s)
    return s

# -- Tahap 5: deduplikasi --
def kunci_dedup(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s))   # sisakan a-z, 0-9, spasi
    s = re.sub(r"(.)\1+", r"\1", s)          # karakter berurutan yang sama -> satu
    return re.sub(r"\s+", " ", s).strip()

def deduplikasi(d):
    d = d.copy()
    d["dedup_key"] = d["text_clean"].map(kunci_dedup)
    d = d[d["dedup_key"] != ""]
    suara = (d.groupby(["dedup_key", "label"]).size().unstack(fill_value=0)
               .reindex(columns=[0, 1], fill_value=0))
    suara.columns = ["n_np", "n_p"]
    seri = suara["n_np"] == suara["n_p"]
    suara["mayor"] = (suara["n_p"] > suara["n_np"]).astype(int)
    sah = suara[~seri]
    x = d[d["dedup_key"].isin(sah.index)].copy()
    x = x[x["label"] == x["dedup_key"].map(sah["mayor"])]          # anggota berlabel mayoritas
    frek = x.groupby(["dedup_key", "text_clean"]).size().rename("_f").reset_index()
    x = x.merge(frek, on=["dedup_key", "text_clean"])
    wakil = (x.sort_values(["dedup_key", "_f", "review_id"], ascending=[True, False, True])
               .groupby("dedup_key").head(1)["review_id"])           # teks tersering, lalu id terkecil
    hasil = d[d["review_id"].isin(set(wakil))].copy()
    hasil["label"] = hasil["dedup_key"].map(sah["mayor"]).astype(int)
    return hasil.sort_values("review_id").reset_index(drop=True), suara, seri

# -- Tahap 6: pembagian data berbasis grup --
KATA_FUNGSI = {"dan", "yang", "nya", "ini", "itu", "ya", "sih", "deh", "juga", "aja", "saja", "nih",
               "kok", "dong", "lah", "di", "ke", "dari", "untuk", "pada", "dengan", "yg", "jg", "utk", "dgn"}

def kunci_grup(s):                      # hanya untuk pengelompokan; teks tidak diubah
    k = set(str(s).split())
    inti = k - KATA_FUNGSI
    return " ".join(sorted(inti)) if inti else " ".join(sorted(k))

def kelompok_panjang(n_kata):
    return "1-2" if n_kata <= 2 else ("3-10" if n_kata <= 10 else ">10")

PROPORSI = {"latih": 0.70, "validasi": 0.15, "uji": 0.15}   # urutan kunci menentukan pemecah seri

def bagi_data(dd, seed=42):
    dd = dd.copy()
    dd["group_key"] = dd["text_clean"].map(kunci_grup)
    dd["n_kata"] = dd["text_clean"].str.split().str.len()
    label_id_kecil = dd.sort_values("review_id").groupby("group_key")["label"].first()
    g = (dd.groupby("group_key")
           .agg(n=("review_id", "size"), n_p=("label", "sum"), kata=("n_kata", "median"))
           .reset_index())
    mayor = np.where(g.n_p * 2 > g.n, 1, np.where(g.n_p * 2 < g.n, 0, -1))
    g["label_grup"] = np.where(mayor == -1, g.group_key.map(label_id_kecil).to_numpy(), mayor).astype(int)
    g["kelompok_panjang"] = g["kata"].map(kelompok_panjang)
    rng = np.random.default_rng(seed)                     # satu generator untuk semua strata
    alokasi = {}
    for _, sub in g.groupby(["label_grup", "kelompok_panjang"], sort=True):
        sub = sub.iloc[rng.permutation(len(sub))]
        total = int(sub.n.sum())
        target = {b: p * total for b, p in PROPORSI.items()}
        terisi = {b: 0 for b in PROPORSI}
        for kunci, n in zip(sub.group_key, sub.n):
            b = min(terisi, key=lambda b: terisi[b] / target[b])   # keterisian terkecil
            alokasi[kunci] = b
            terisi[b] += int(n)
    dd["split"] = dd["group_key"].map(alokasi)
    dd["kelompok_panjang"] = dd["n_kata"].map(kelompok_panjang)
    return dd, g

In [ ]:
# ══ Tahap 1.2 — Audit kualitas data (sel 2/2; program saja, data tidak diubah) ══
# Package: json, importlib (bawaan Python); pandas; scipy.stats.spearmanr; lib.wondr_teks (pola teks)
import json, importlib
import pandas as pd
from scipy.stats import spearmanr
import lib.wondr_teks as wt
from lib.bantu import _tampil                              # format angka Indonesia untuk tabel
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["1.2"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap01_mentah.parquet"
df = pd.read_parquet(masuk)
u, r = df["ulasan"].astype(str), df["rating"]
n_kata, teks = u.str.split().str.len(), u.str.lower().str.strip()    # kata = dipisah spasi

# 2) Pemeriksaan (data hanya dibaca); label sementara = rating >= tau, hanya untuk audit
dibalas, huruf = df["balasan_dev"].notna(), u.str.lower().map(lambda s: bool(wt.HURUF.search(s)))
n_lbl = (r >= CFG["tau"]).astype(int).groupby(teks).nunique()
med = [float(n_kata[r == k].median()) for k in range(1, 6)]
emoji = int(u.map(lambda s: bool(wt.EMO.search(s) or wt.SIMBOL.search(s))).sum())
AUDIT = [  # (temuan, nilai aktual, kunci TARGET, toleransi, tahap pemakai)
    ("Nilai kosong per kolom", {k: int(v) for k, v in df.isna().sum().items() if v}, "kosong", 0, "2"),
    ("Dibalas pengembang (%), rating 1–4 · 5",
     {"1-4": dibalas[r <= 4].mean() * 100, "5": dibalas[r == 5].mean() * 100}, "dibalas_pct", 0.05, "2"),
    ("Likes bernilai nol (%)", (df["likes"] == 0).mean() * 100, "likes_nol_pct", 0.05, "2"),
    ("Spearman likes–rating", spearmanr(df["likes"], r).statistic, "spearman", 5e-5, "2"),
    ("Nama unik", int(df["nama"].nunique()), "nama_unik", 0, "2"),
    ("Rating 3 >= 8 kata (jumlah · total)", (int(((r == 3) & (n_kata >= 8)).sum()), int((r == 3).sum())),
     "r3_panjang", 0, "3"),
    ("Median kata per rating 1–5", [int(m) if m == int(m) else m for m in med], "median_kata", 0, "6.1"),
    ("Baris berisi emoji atau simbol", emoji, "emoji_simbol", 0, "4"),
    ("Huruf berulang >= 3 (baris · %)", (int(huruf.sum()), huruf.mean() * 100), "huruf_ulang", 0.05, "4"),
    ("Tanda baca berulang (baris)", int(u.map(lambda s: bool(wt.BACA.search(s))).sum()), "baca_ulang", 0, "4"),
    ("Duplikat persis (% baris)", df["ulasan"].duplicated().mean() * 100, "duplikat_pct", 0.05, "5"),
    ("Teks berlabel ganda (teks · baris)", (int((n_lbl == 2).sum()), int(teks.isin(n_lbl[n_lbl == 2].index).sum())),
     "label_ganda", 0, "5"),
]

# 3) Simpan tabel (angka mentah, JSON), tampilkan temuan -> angka -> tahap pemakai, lalu cek
keluar = simpan_tabel(pd.DataFrame([{"temuan": t, "nilai": json.dumps(v), "target": json.dumps(T[k]), "tahap": th}
                                    for t, v, k, _, th in AUDIT]), "tahap01_audit")
judul("1.2", "Audit kualitas data", masukan=[f"{rel(masuk)} ({fmt_id(len(df))} baris)", "lib/wondr_teks.py"],
      keluaran=f"{rel(keluar)} ({len(AUDIT)} baris)")
print(f"{'Temuan':<40}{'Angka':<28}Tahap")
for t, v, _, tol, th in AUDIT:
    print(f"{t:<40}{_tampil(v, 1 if tol == 0.05 else 4):<28}{th}")
cek("jumlah baris tetap", len(df), TARGET["1.1"]["baris"])
for t, v, k, tol, _ in AUDIT:
    cek(t, v, T[k], toleransi=tol)
selesai()

## Tahap 2 — Seleksi atribut

**Dasar naskah:** Bab III tahap 2 (nomor urut sebagai identitas; atribut yang digunakan hanya teks ulasan dan *rating*; nama, balasan pengembang, dan *likes* dihapus; tanggal dan versi hanya informasi pendukung); Bab II subbab 2.4.2. `review_id` dibuat di sini (`CLAUDE.md` Bagian 10, keputusan no. 7).

**Package:** `pandas` (olah tabel), `pyarrow` (mesin baca/simpan .parquet); `lib.bantu`.

In [ ]:
# ══ Tahap 2 — Seleksi atribut ══
# Package: pandas (olah tabel), pyarrow (mesin baca/simpan .parquet)
import pandas as pd
mulai()
T = TARGET["2"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap01_mentah.parquet"
mentah = pd.read_parquet(masuk)

# 2) Nomor urut sebagai identitas, lalu pilih atribut (ringkasan kalimat Bab III tahap 2)
ATRIBUT = [  # (atribut, peran, alasan)
    ("review_id", "dibuat", "nomor urut sesuai urutan baris data mentah, sebagai identitas"),
    ("ulasan", "dipakai", "teks ulasan: satu-satunya masukan model"),
    ("rating", "dipakai", "sumber label awal (Tahap 3), bukan fitur"),
    ("tanggal", "disimpan", "informasi pendukung (subbab 2.4.2)"),
    ("versi", "disimpan", "informasi pendukung (subbab 2.4.2)"),
    ("nama", "dibuang", "nama pengguna; dihapus untuk menjaga privasi"),
    ("balasan_dev", "dibuang", "ditulis setelah ulasan; isinya dapat membocorkan label"),
    ("likes", "dibuang", "tidak tersedia pada saat prediksi"),
]
pakai = [a for a, peran, _ in ATRIBUT if peran != "dibuang"]
df = mentah.assign(review_id=range(1, len(mentah) + 1))[pakai]

# 3) Simpan
keluar = PATH["hasil"] / "tahap02_atribut.parquet"
df.to_parquet(keluar, index=False)

# 4) Ringkasan + cek
judul("2", "Seleksi atribut", masukan=f"{rel(masuk)} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"{'Atribut':<13}{'Peran':<10}Alasan")
for a, peran, alasan in ATRIBUT:
    print(f"{a:<13}{peran:<10}{alasan}")
tercatat = {a for a, _, _ in ATRIBUT} - {"review_id"} == set(mentah.columns)
ya = lambda b: "ya" if b else "tidak"
cek("jumlah baris", len(df), T["baris"])
cek("kolom dan urutannya", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(df.columns), "sesuai")
cek("semua kolom data mentah tercatat di tabel", ya(tercatat), "ya")
cek("review_id = 1, 2, ..., n (unik, urut)", ya(df["review_id"].tolist() == list(range(1, len(df) + 1))), "ya")
cek("isi ulasan, rating, tanggal, versi tidak berubah", ya(df[pakai[1:]].equals(mentah[pakai[1:]])), "ya")
selesai()

## Tahap 3 — Pelabelan awal (*top-2-box*)

**Dasar naskah:** Bab III tahap 3; pers. (2.1)–(2.2) dengan τ = 4; Tabel 2.8; Morgan dan Rego (2006). Kolom bernama `label` agar kode acuan `lib/wondr_teks.py` tidak diubah; namanya menjadi `label_awal` di Tahap 9.3.

**Package:** `pandas` (olah tabel), `matplotlib` (grafik), `pyarrow` (mesin baca/simpan .parquet); `lib.bantu`. Warna kelas: slot 1–2 palet kategori yang lolos validasi buta warna (Positif biru `#2a78d6`, Non-Positif oranye `#eb6834`), dipakai tetap di semua gambar.

In [ ]:
# ══ Tahap 3 — Pelabelan awal (top-2-box) ══
# Package: pandas (olah tabel), matplotlib (grafik), pyarrow (mesin baca/simpan .parquet)
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
mulai()
T = TARGET["3"]
WARNA = {1: "#2a78d6", 0: "#eb6834"}          # slot kategori 1-2 palet tervalidasi (CVD aman); tetap per kelas

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap02_atribut.parquet"
df = pd.read_parquet(masuk)

# 2) Label awal pers. (2.1)-(2.2): rating >= tau -> 1 (Positif), selain itu 0 (Non-Positif); tidak ada baris dihapus
df["label"] = (df["rating"] >= CFG["tau"]).astype(int)
keluar = PATH["hasil"] / "tahap03_label.parquet"
df.to_parquet(keluar, index=False)

# 3) Gambar: jumlah ulasan per rating, diwarnai per kelas
per = df.groupby("rating").agg(n=("review_id", "size"), kelas=("label", "first"))
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(per.index, per.n, width=0.6, color=[WARNA[k] for k in per.kelas], zorder=2)
for r, n in per.n.items():
    ax.text(r, n + 60, fmt_id(int(n)), ha="center", va="bottom", fontsize=9, color="#52514e")
ax.set(xlabel="Rating", ylabel="Jumlah ulasan", xticks=per.index, ylim=(0, per.n.max() * 1.12))
ax.yaxis.set_major_formatter(lambda y, _: fmt_id(int(y)))
ax.grid(axis="y", color="#e4e3df", linewidth=0.8, zorder=0)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(colors="#52514e", length=0)
ax.legend(handles=[Patch(color=WARNA[k], label=CFG["kelas"][k]) for k in (1, 0)], frameon=False, loc="upper left")
gambar = simpan_gambar(fig, "tahap03_distribusi_rating")

# 4) Ringkasan + cek
kelas = {CFG["kelas"][k]: (int((df.label == k).sum()), (df.label == k).mean() * 100) for k in (1, 0)}
judul("3", "Pelabelan awal (top-2-box)", masukan=f"{rel(masuk)} ({fmt_id(len(df))} baris)",
      keluaran=[f"{rel(keluar)} ({fmt_id(len(df))} baris, kolom baru: label)", rel(gambar)])
print(f"{'Rating':<8}{'Jumlah':>8}  Kelas")
for r, (n, k) in per.iterrows():
    print(f"{r:<8}{fmt_id(int(n)):>8}  {CFG['kelas'][k]}")
for nama, (n, pct) in kelas.items():
    print(f"Total {nama:<12}{fmt_id(n):>7}  ({fmt_id(pct, 2)}%)")
cek("jumlah baris tetap (tidak ada yang dihapus)", len(df), TARGET["2"]["baris"])
cek("jumlah per rating", {int(r): int(n) for r, n in per.n.items()}, T["per_rating"])
cek("Positif (jumlah · %)", kelas["Positif"], T["positif"], toleransi=0.005)
cek("Non-Positif (jumlah · %)", kelas["Non-Positif"], T["non_positif"], toleransi=0.005)
cek("gambar tersimpan", "ya" if gambar.exists() else "tidak", "ya")
selesai()

## Tahap 4 — *Pre-processing*

**Dasar naskah:** Bab III tahap 4; Bab II subbab 2.2 (Tabel 2.1–2.7). Enam langkah `LANGKAH` dari `lib/wondr_teks.py` dijalankan berurutan, lalu teks kosong dibuang. Tanpa *stemming*, tanpa *stopword removal*, dan tanpa normalisasi kamus.

**Package:** `importlib` (bawaan Python); `pandas`; `lib.wondr_teks` (memakai `re`, `unicodedata`); `lib.bantu`. Masukan uji contoh Bab II ditulis sebagai *escape* Unicode.

In [ ]:
# ══ Tahap 4 — Pre-processing ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (LANGKAH, bersihkan; memakai re, unicodedata)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["4"]

# 1) Uji bersihkan() dengan contoh Bab II (Tabel 2.1-2.7); masukan ditulis sebagai escape Unicode
CONTOH = ["\U0001D401\U0001D400\U0001D406\U0001D414\U0001D412\U0001D412\U0001D412 bgt!!! \u2605\u2605 tp "
          "kadang\u00b2 error, nunggu lamaaa", "\uff4d\uff41\uff4e\uff54\uff41\uff50"]
uji = [wt.bersihkan(x) for x in CONTOH]

# 2) Baca masukan, jalankan 6 langkah berurutan sambil menghitung ulasan yang berubah
masuk = PATH["hasil"] / "tahap03_label.parquet"
df = pd.read_parquet(masuk)
s, berubah = df["ulasan"].astype(str), []
for i, (_, f) in enumerate(wt.LANGKAH):
    t = s.map(f)
    berubah.append(int((t != s).sum()))
    if i == 3:                                              # langkah 4: simpan contoh sebelum -> sesudah
        contoh4 = pd.DataFrame({"id": df["review_id"], "a": s, "b": t})[t != s].head(3)
    s = t
df["text_clean"] = s

# 3) Buang teks kosong (jumlah sebelum dan sesudah ditampilkan), lalu simpan
kosong, bersih = df[df["text_clean"] == ""], df[df["text_clean"] != ""].reset_index(drop=True)
keluar = PATH["hasil"] / "tahap04_bersih.parquet"
bersih.to_parquet(keluar, index=False)

# 4) Ringkasan + cek
judul("4", "Pre-processing", masukan=f"{rel(masuk)} ({fmt_id(len(df))} baris)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(bersih))} baris, kolom baru: text_clean)")
print(f"{'Langkah':<32}{'Ulasan berubah':>15}")
for (nama, _), n in zip(wt.LANGKAH, berubah):
    print(f"{nama:<32}{fmt_id(n):>15}")
r_kosong = {int(k): int(v) for k, v in kosong["rating"].value_counts().items()}
print(f"\nTeks kosong dibuang: {fmt_id(len(kosong))} ({', '.join(f'rating {k}: {v}' for k, v in r_kosong.items())})"
      f"  →  {fmt_id(len(df))} menjadi {fmt_id(len(bersih))} ulasan")
print("Contoh langkah 4 saja (huruf berulang >= 3 -> 2); tanda baca berulang baru dipadatkan di langkah 5:")
for i, a, b in contoh4.itertuples(index=False):                # teks utuh, tanpa pemotongan
    kata = "; ".join(f'"{x}" → "{y}"' for x, y in zip(a.split(), b.split()) if x != y)
    print(f"  [{i}] kata yang berubah : {kata}")
    print(f"       hasil 6 langkah  : {bersih.loc[bersih['review_id'] == i, 'text_clean'].iloc[0]}")
cek("uji contoh Bab II (Tabel 2.1-2.7)", uji, T["contoh_bab2"])
cek("ulasan berubah per langkah", berubah, T["berubah"])
cek("teks kosong dibuang", len(kosong), T["kosong"])
cek("rating teks kosong", r_kosong, T["kosong_rating"])
cek("jumlah setelah pre-processing", len(bersih), T["hasil"])
tc = bersih["text_clean"]                                      # invarian: aturan berlaku di seluruh teks
cek("sisa huruf berulang · tanda baca berulang · emoji", [int(tc.map(lambda x: bool(p.search(x))).sum())
    for p in (wt.HURUF, wt.BACA, wt.EMO)], [0, 0, 0])
sama = (bersih["ulasan"].map(wt.bersihkan) == tc).all()
cek("text_clean = bersihkan(ulasan) di semua baris", "ya" if sama else "tidak", "ya")
selesai()

## Tahap 5 — Deduplikasi

**Dasar naskah:** Bab III tahap 5; Bab II subbab 2.4.1 dan Tabel 2.11 (label mayoritas; kunci seri dikeluarkan; wakil dipilih berdasarkan kesesuaian dengan label mayoritas, teks tersering, lalu nomor urut terkecil). Tabel kunci seri ditampilkan sebagai *program saja* (`CLAUDE.md` Bagian 10, keputusan no. 10).

**Package:** `importlib` (bawaan Python); `pandas`; `lib.wondr_teks` (`kunci_dedup`, `deduplikasi`); `lib.bantu`.

In [ ]:
# ══ Tahap 5 — Deduplikasi ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (kunci_dedup, deduplikasi)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["5"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap04_bersih.parquet"
d = pd.read_parquet(masuk)

# 2) Deduplikasi (kode acuan): label mayoritas, kunci seri dikeluarkan, wakil = teks tersering lalu id terkecil
hasil, suara, seri = wt.deduplikasi(d)
keluar = PATH["hasil"] / "tahap05_dedup.parquet"
hasil.to_parquet(keluar, index=False)

# 3) Rincian alur, dihitung dari tabel suara per kunci; simpan CSV (angka mentah)
kunci, sah = d["text_clean"].map(wt.kunci_dedup), suara[~seri]
ALUR = {"ulasan masuk": len(d), "kunci kosong (dibuang)": int((kunci == "").sum()),
        "kunci deduplikasi": len(suara), "kunci seri": int(seri.sum()),
        "dibuang: salinan berlabel sama": int((sah[["n_np", "n_p"]].max(axis=1) - 1).sum()),
        "dibuang: anggota kalah suara": int(sah[["n_np", "n_p"]].min(axis=1).sum()),
        "dibuang: anggota kunci seri": int(suara.loc[seri, ["n_np", "n_p"]].to_numpy().sum()),
        "dokumen hasil": len(hasil)}
alur = simpan_tabel(pd.DataFrame({"langkah": list(ALUR), "jumlah": list(ALUR.values())}), "tahap05_alur")

# 4) Ringkasan + cek
dibuang = [len(d) - ALUR["kunci kosong (dibuang)"] - len(hasil)] + [v for k, v in ALUR.items() if k.startswith("dibuang")]
pos = (d["label"].mean() * 100, hasil["label"].mean() * 100)
judul("5", "Deduplikasi", masukan=f"{rel(masuk)} ({fmt_id(len(d))} baris)",
      keluaran=[f"{rel(keluar)} ({fmt_id(len(hasil))} baris, kolom baru: dedup_key)", rel(alur)])
for k, v in ALUR.items():
    print(f"{k:<34}{fmt_id(v):>7}")
print(f"Proporsi Positif                  {fmt_id(pos[0], 1)}% → {fmt_id(pos[1], 1)}%")
print("\nKunci seri yang dikeluarkan (program saja; rating anggotanya saling bertentangan):")
anggota = d.assign(k=kunci)[kunci.isin(suara[seri].index)].sort_values("review_id")
for k, g in anggota.groupby("k"):
    print(f"  {k:<20}" + " · ".join(f"[{i}] rating {r}" for i, r in zip(g["review_id"], g["rating"])))
cek("kunci kosong dibuang", ALUR["kunci kosong (dibuang)"], 0)
cek("kunci · kunci seri · ulasan seri", (len(suara), int(seri.sum()), dibuang[3]),
    (T["kunci"], T["kunci_seri"], T["ulasan_seri"]))
cek("jumlah dokumen", len(hasil), T["dokumen"])
cek("dibuang (total · salinan · kalah suara · seri)", dibuang, T["dibuang"])
cek("total dibuang = jumlah ketiga rincian", "ya" if dibuang[0] == sum(dibuang[1:]) else "tidak", "ya")
cek("Positif (%) sebelum · sesudah", pos, T["positif_pct"], toleransi=0.05)
cek("satu dokumen per kunci", "ya" if hasil["dedup_key"].is_unique else "tidak", "ya")
selesai()

## Tahap 6.1 — Pembagian data

**Dasar naskah:** Bab III tahap 6; pers. (2.6); Tabel 2.12–2.13. Latih 70%, validasi 15%, uji 15%; grup berkunci sama ditempatkan pada himpunan yang sama; stratifikasi pada tingkat grup berdasarkan label mayoritas grup dan kelompok panjang (1–2, 3–10, >10 kata); grup setiap strata diacak dengan *seed* 42 lalu dialokasikan ke himpunan dengan tingkat pengisian terendah.

**Package:** `importlib` (bawaan Python); `pandas`; `lib.wondr_teks` (`bagi_data`, `kunci_grup`, `kelompok_panjang`; memakai `numpy`); `lib.bantu`.

In [ ]:
# ══ Tahap 6.1 — Pembagian data ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (bagi_data, kunci_grup, kelompok_panjang)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["6.1"]
KOLOM = ["review_id", "ulasan", "text_clean", "rating", "label", "group_key", "dedup_key", "kelompok_panjang"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap05_dedup.parquet"
dd = pd.read_parquet(masuk)

# 2) Bagi berbasis grup pers. (2.6), stratifikasi label grup x kelompok panjang (kode acuan; seed dari CFG)
bagi, grup = wt.bagi_data(dd, seed=CFG["seed_bagi"])

# 3) Simpan tiga bagian
keluar = {b: PATH["hasil"] / f"tahap06_{b}.parquet" for b in CFG["proporsi"]}
for b, p in keluar.items():
    bagi.loc[bagi["split"] == b, KOLOM].reset_index(drop=True).to_parquet(p, index=False)

# 4) Ringkasan + cek
hit = {b: (int((bagi.split == b).sum()), int(((bagi.split == b) & (bagi.label == 0)).sum()),
           int(((bagi.split == b) & (bagi.label == 1)).sum())) for b in CFG["proporsi"]}
judul("6.1", "Pembagian data", masukan=f"{rel(masuk)} ({fmt_id(len(dd))} dokumen)",
      keluaran=[f"{rel(p)} ({fmt_id(hit[b][0])} dokumen)" for b, p in keluar.items()])
print(f"Grup: {fmt_id(len(grup))} dari {fmt_id(len(dd))} dokumen "
      f"({fmt_id(int((grup.n > 1).sum()))} grup berisi lebih dari satu dokumen)\n")
print(f"{'Bagian':<10}{'Dokumen':>8}{'Porsi':>8}{'Target':>8}{'Non-Positif':>13}{'Positif':>9}")
for b, p in CFG["proporsi"].items():
    n, n0, n1 = hit[b]
    print(f"{b:<10}{fmt_id(n):>8}{fmt_id(n / len(bagi) * 100, 1) + '%':>8}{fmt_id(p * 100, 0) + '%':>8}"
          f"{fmt_id(n0):>13}{fmt_id(n1):>9}")
cek("PROPORSI di lib/wondr_teks.py = CFG", wt.PROPORSI, CFG["proporsi"])
cek("jumlah grup", len(grup), T["grup"])
for b in CFG["proporsi"]:
    cek(f"{b} (dokumen · Non-Positif · Positif)", hit[b], T[b])
cek("semua dokumen mendapat satu bagian", "ya" if bagi["split"].notna().all() and len(bagi) == len(dd) else "tidak", "ya")
selesai()

## Tahap 6.2 — Cek kebocoran dan keseimbangan

**Dasar naskah:** Bab III tahap 6 (setelah pembagian dipastikan tidak ada kunci pengelompokan, teks, dan kunci deduplikasi yang muncul pada lebih dari satu himpunan); Bab II subbab 2.4.2–2.4.4. Kedua sel hanya **membaca** berkas Tahap 6.1; *seed*, proporsi, dan algoritma pembagian tidak diubah.

**Package:** `pandas` (olah tabel); `scipy.stats.chi2_contingency` (uji khi-kuadrat); `lib.bantu`.

Sel 1/2: kebocoran `group_key`, `dedup_key`, `text_clean`, kelengkapan dokumen, dan posisi kembaran-dekat. Sel 2/2: proporsi label dan kelompok panjang per bagian, uji khi-kuadrat, dan tabel strata. **Mulai di sini data uji dikunci.**

In [ ]:
# ══ Tahap 6.2 — Cek kebocoran dan keseimbangan (sel 1/2: kebocoran dan kelengkapan) ══
# Package: pandas (olah tabel, baca .parquet)
import pandas as pd
mulai()
T, BAGIAN = TARGET["6.2"], list(CFG["proporsi"])

# 1) Baca ketiga bagian dari berkas Tahap 6.1 (pembagian tidak diubah) dan hasil deduplikasi
masuk = {b: PATH["hasil"] / f"tahap06_{b}.parquet" for b in BAGIAN}
df = pd.concat([pd.read_parquet(p).assign(bagian=b) for b, p in masuk.items()], ignore_index=True)
dedup = pd.read_parquet(PATH["hasil"] / "tahap05_dedup.parquet")

# 2) Kebocoran: kunci yang muncul di lebih dari satu bagian
KUNCI = ("group_key", "dedup_key", "text_clean")
lintas = {k: df.groupby(k)["bagian"].nunique() for k in KUNCI}
bocor = {k: s[s > 1] for k, s in lintas.items()}

# 3) Ringkasan + cek
judul("6.2", "Cek kebocoran (sel 1/2)", masukan=[f"{rel(p)} ({fmt_id(int((df.bagian == b).sum()))} dokumen)"
                                                for b, p in masuk.items()])
print(f"{'Kunci':<12}{'Nilai unik':>11}{'Di > 1 bagian':>15}")
for k in KUNCI:
    print(f"{k:<12}{fmt_id(len(lintas[k])):>11}{fmt_id(len(bocor[k])):>15}")
for k in KUNCI:                                             # contoh hanya dicetak bila ada kebocoran
    for kunci in bocor[k].index[:5]:
        print(f"  BOCOR {k} {kunci!r}: {sorted(df.loc[df[k] == kunci, 'bagian'].unique())}")
grup_per_bagian = int(df.groupby("bagian")["group_key"].nunique().sum())
for k in KUNCI:
    cek(f"{k} di lebih dari satu bagian", len(bocor[k]), T["bocor"])
cek("jumlah grup per bagian dijumlahkan", grup_per_bagian, TARGET["6.1"]["grup"])
cek("total dokumen · review_id unik", (len(df), int(df["review_id"].nunique())),
    (TARGET["5"]["dokumen"], TARGET["5"]["dokumen"]))
sama = set(df["review_id"]) == set(dedup["review_id"])
cek("review_id = review_id hasil deduplikasi", "ya" if sama else "tidak", "ya")
kd = df.loc[df["review_id"].isin(CFG["kembaran_dekat"]), ["review_id", "bagian"]]
cek("kembaran-dekat berada di uji", int((kd["bagian"] == "uji").sum()), T["kembaran_di_uji"])
selesai()

In [ ]:
# ══ Tahap 6.2 — Cek kebocoran dan keseimbangan (sel 2/2: keseimbangan dan strata) ══
# Package: pandas (olah tabel); scipy.stats.chi2_contingency (uji khi-kuadrat)
import pandas as pd
from scipy.stats import chi2_contingency
mulai()
T, BAGIAN, PANJANG = TARGET["6.2"], list(CFG["proporsi"]), list(CFG["kelompok_panjang"])

# 1) Baca ketiga bagian (pembagian tidak diubah)
df = pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet").assign(bagian=b) for b in BAGIAN],
               ignore_index=True)

# 2) Tabel silang bagian x label dan bagian x kelompok panjang, lalu uji khi-kuadrat
x_label = pd.crosstab(df["bagian"], df["label"]).loc[BAGIAN]
x_panjang = pd.crosstab(df["bagian"], df["kelompok_panjang"]).loc[BAGIAN, PANJANG]
p_label, p_panjang = chi2_contingency(x_label)[1], chi2_contingency(x_panjang)[1]
pct_np = x_label[0] / x_label.sum(axis=1) * 100
pct_pj = x_panjang.div(x_panjang.sum(axis=1), axis=0) * 100

# 3) Simpan tabel strata (label x kelompok panjang) per bagian, angka mentah
strata = pd.crosstab([df["label"].map(CFG["kelas"]), df["kelompok_panjang"]], df["bagian"])[BAGIAN]
keluar = simpan_tabel(strata.reset_index().rename(columns={"label": "kelas"}), "tahap06_strata")

# 4) Ringkasan + cek
judul("6.2", "Cek keseimbangan (sel 2/2)", masukan="hasil/tahap06_{latih,validasi,uji}.parquet", keluaran=rel(keluar))
print(f"{'Bagian':<10}{'Non-Positif':>12}" + "".join(f"{'kata ' + k:>12}" for k in PANJANG))
for b in BAGIAN:
    sel = [fmt_id(pct_np[b], 1)] + [fmt_id(pct_pj.loc[b, k], 1) for k in PANJANG]
    print(f"{b:<10}" + "".join(f"{x + '%':>12}" for x in sel))
print(f"\nKhi-kuadrat: bagian x label p = {fmt_id(p_label)} · bagian x kelompok panjang p = {fmt_id(p_panjang)}")
print(f"\nDokumen per strata\n{strata.to_string()}")
cek("Non-Positif (%) latih · validasi · uji", [pct_np[b] for b in BAGIAN], T["non_positif_pct"], toleransi=0.05)
for b in BAGIAN:
    cek(f"kelompok panjang (%) {b}", list(pct_pj.loc[b]), T["panjang_pct"][b], toleransi=0.05)
cek("p khi-kuadrat label · panjang", (p_label, p_panjang), (T["p_label"], T["p_panjang"]), toleransi=5e-5)
print("\nMulai di sini data uji dikunci: label dan metriknya tidak dipakai sampai Tahap 14 "
      "(kecuali teksnya diekspor ke lembar anotasi di Tahap 9.1).")
selesai()

## Tahap 7 — Tokenisasi

**Dasar naskah:** Bab III tahap 7; Bab II subbab 2.5 dan 2.7; pers. (2.7)–(2.9); Tabel 2.14. *Tokenizer* bawaan `indobenchmark/indobert-base-p1`; *fertility*, r_UNK, dan L_max dihitung **hanya dari data latih** (q = 100, b = 32); panjang dokumen = token subkata + [CLS] + [SEP]; dokumen validasi/uji yang lebih panjang dari L_max hanya dideskripsikan (akan terpotong).

**Package:** `math` (bawaan Python); `numpy`; `pandas`; `transformers.AutoTokenizer`; `matplotlib`; `lib.bantu`.

Sel 1/2: token, statistik, dan L_max → `hasil/tahap07_tokenisasi.json`. Sel 2/2: histogram panjang token data latih → `bab4/gambar/tahap07_panjang_token.png`.

In [ ]:
# ══ Tahap 7 — Tokenisasi (sel 1/2: token subkata, statistik data latih, L_max) ══
# Package: math (bawaan Python); numpy; pandas; transformers.AutoTokenizer (tokenizer IndoBERT)
import math
import numpy as np, pandas as pd
from transformers import AutoTokenizer
mulai()
T, BAGIAN = TARGET["7"], list(CFG["proporsi"])

# 1) Baca ketiga bagian dan tokenizer bawaan checkpoint (harus sama dengan tahap prapelatihan)
data = {b: pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet") for b in BAGIAN}
tok = AutoTokenizer.from_pretrained(CFG["checkpoint"])

# 2) Token subkata per dokumen (tanpa [CLS]/[SEP]); panjang dokumen = subkata + 2
sub = {b: [tok.tokenize(t) for t in d["text_clean"]] for b, d in data.items()}
panjang = {b: np.array([len(s) + 2 for s in sub[b]]) for b in BAGIAN}
cocok = all((panjang[b] == [len(x) for x in tok(list(data[b]["text_clean"]))["input_ids"]]).all() for b in BAGIAN)

# 3) Statistik HANYA dari data latih: fertility (2.7), r_UNK (2.8), L_max (2.9); validasi dan uji hanya deskripsi
n_sub, n_kata = sum(map(len, sub["latih"])), int(data["latih"]["text_clean"].str.split().str.len().sum())
n_unk = sum(s.count(tok.unk_token) for s in sub["latih"])
F, r_unk, P_q = n_sub / n_kata, n_unk / n_sub, float(np.percentile(panjang["latih"], CFG["q"]))
L_max = CFG["b"] * math.ceil(P_q / CFG["b"])
potong = {b: data[b].loc[panjang[b] > L_max, "review_id"].tolist() for b in BAGIAN}
stat = {b: {"n": len(p), "median": float(np.median(p)), "p95": float(np.percentile(p, 95)), "maks": int(p.max()),
            "terpotong": len(potong[b])} for b, p in panjang.items()}
keluar = simpan_json({"checkpoint": CFG["checkpoint"], "kosakata": tok.vocab_size, "n_subkata": n_sub,
                      "n_kata": n_kata, "n_unk": n_unk, "F": F, "r_UNK": r_unk, "P_q": P_q, "L_max": L_max,
                      "statistik": stat, "terpotong_id": potong, "panjang_latih": panjang["latih"].tolist()},
                     PATH["hasil"] / "tahap07_tokenisasi.json")

# 4) Ringkasan + cek
judul("7", "Tokenisasi (sel 1/2)", masukan=["hasil/tahap06_{latih,validasi,uji}.parquet", f"tokenizer {CFG['checkpoint']}"],
      keluaran=rel(keluar))
print(f"{'Bagian':<10}{'n':>7}{'Median':>8}{'P95':>7}{'Maks':>6}{'Terpotong':>11}")
for b, s in stat.items():
    print(f"{b:<10}{fmt_id(s['n']):>7}{fmt_id(s['median'], 0):>8}{fmt_id(s['p95'], 0):>7}{s['maks']:>6}{s['terpotong']:>11}")
print(f"\nData latih: F = {fmt_id(F)} subkata/kata · r_UNK = {fmt_id(r_unk, 6)} ({fmt_id(n_unk)} dari {fmt_id(n_sub)} token)")
print(f"L_max = {CFG['b']}·⌈P{CFG['q']} / {CFG['b']}⌉ = {CFG['b']}·⌈{fmt_id(P_q, 0)} / {CFG['b']}⌉ = {L_max}")
cek("panjang = subkata + 2 (sesuai tokenizer)", "ya" if cocok else "tidak", "ya")
cek("L_max (naskah: 128)", L_max, T["L_max"])
cek("dokumen terpotong", {b: len(potong[b]) for b in BAGIAN}, T["terpotong"])
cek("review_id yang terpotong", potong["latih"] + potong["validasi"] + potong["uji"], T["terpotong_id"])
selesai()

In [ ]:
# ══ Tahap 7 — Tokenisasi (sel 2/2: histogram panjang token data latih) ══
# Package: numpy; matplotlib (grafik); lib.bantu (baca_json, simpan_gambar)
import numpy as np
import matplotlib.pyplot as plt
mulai()

# 1) Baca hasil sel 1/2
masuk = PATH["hasil"] / "tahap07_tokenisasi.json"
tk = baca_json(masuk)
p, L_max = np.array(tk["panjang_latih"]), tk["L_max"]

# 2) Histogram (satu seri: tanpa legenda) dengan garis L_max
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(p, bins=range(0, L_max + 9, 4), color="#2a78d6", rwidth=0.85, zorder=2)
ax.axvline(L_max, color="#52514e", linewidth=1.2, zorder=3)
ax.text(L_max - 2, ax.get_ylim()[1] * 0.92, f"L_max = {L_max}", ha="right", fontsize=9, color="#52514e")
ax.set(xlabel="Panjang token per dokumen latih (termasuk [CLS] dan [SEP])", ylabel="Jumlah dokumen")
ax.yaxis.set_major_formatter(lambda y, _: fmt_id(int(y)))
ax.grid(axis="y", color="#e4e3df", linewidth=0.8, zorder=0)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(colors="#52514e", length=0)
gambar = simpan_gambar(fig, "tahap07_panjang_token")

# 3) Ringkasan + cek
judul("7", "Tokenisasi (sel 2/2)", masukan=rel(masuk), keluaran=rel(gambar))
cek("dokumen latih pada histogram", len(p), TARGET["6.1"]["latih"][0])
cek("semua dokumen latih <= L_max", int((p > L_max).sum()), TARGET["7"]["terpotong"]["latih"])
cek("gambar tersimpan", "ya" if gambar.exists() else "tidak", "ya")
selesai()

### Tahap 7 — Pemeriksaan akhir tokenisasi (tambahan, diminta pengguna)

Hanya membaca hasil Tahap 6.1 dan 7; konfigurasi tokenisasi tidak diubah. Memeriksa: L_max dihitung ulang dari data latih saja; identitas *tokenizer* terhadap Tabel 2.14 dan contoh subbab 2.5.2 naskah; pemotongan `truncation=True, max_length=L_max` beserta token khusus; dokumen yang terpotong dan panjangnya sebelum/sesudah; *padding* per *batch*.

**Package:** `math` (bawaan Python); `numpy`; `pandas`; `transformers.AutoTokenizer`; `huggingface_hub`; `lib.bantu`.

In [ ]:
# ══ Tahap 7 — Pemeriksaan akhir tokenisasi (tambahan; konfigurasi tidak diubah) ══
# Package: math (bawaan Python); numpy; pandas; transformers.AutoTokenizer; huggingface_hub (versi)
import math
import numpy as np, pandas as pd
import huggingface_hub
from transformers import AutoTokenizer
mulai()
masuk = PATH["hasil"] / "tahap07_tokenisasi.json"
tk, BAGIAN = baca_json(masuk), list(CFG["proporsi"])
data = {b: pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet") for b in BAGIAN}
tok, L = AutoTokenizer.from_pretrained(CFG["checkpoint"]), tk["L_max"]     # dari cache; tidak mengunduh ulang

# 1) L_max dihitung ulang dari panjang token data latih saja
pl = np.array(tk["panjang_latih"])
L_ulang = CFG["b"] * math.ceil(np.percentile(pl, CFG["q"]) / CFG["b"])
# 2) Identitas tokenizer: contoh Tabel 2.14 dan contoh subbab 2.5.2 naskah
T214 = {"aplikasinya": ["aplikasinya"], "mantapp": ["mantap", "##p"], "gabisa": ["gab", "##isa"],
        "tiba-tiba": ["tiba", "-", "tiba"], "bagus!": ["bagus", "!"]}
s252 = [len(tok.tokenize(w)) for w in "aplikasinya error bgt gabisa buka".split()]
# 3) Pemotongan seperti saat pelatihan (truncation=True, max_length=L_max) dan padding per batch
penuh = {b: tok(list(d["text_clean"]))["input_ids"] for b, d in data.items()}
potong = {b: tok(list(d["text_clean"]), truncation=True, max_length=L)["input_ids"] for b, d in data.items()}
kena = {b: [int(i) for i, a in zip(data[b]["review_id"], penuh[b]) if len(a) > L] for b in BAGIAN}
utuh = all(a == c for b in BAGIAN for a, c in zip(penuh[b], potong[b]) if len(a) <= L)
ujung = all(c[0] == tok.cls_token_id and c[-1] == tok.sep_token_id for b in BAGIAN for c in potong[b])
bt = tok(list(data["latih"]["text_clean"][:16]), truncation=True, max_length=L, padding=True, return_tensors="np")
nyata = [len(x) for x in potong["latih"][:16]]

judul("7", "Pemeriksaan akhir tokenisasi", masukan=[rel(masuk), "hasil/tahap06_{latih,validasi,uji}.parquet"])
mml = "tidak diatur" if tok.model_max_length > 1e6 else tok.model_max_length
print(f"Tokenizer {type(tok).__name__} · kosakata {fmt_id(tok.vocab_size)} · potong sisi {tok.truncation_side} · "
      f"padding sisi {tok.padding_side} · model_max_length bawaan {mml} · huggingface_hub {huggingface_hub.__version__}")
print(f"Token khusus: [CLS]={tok.cls_token_id} [SEP]={tok.sep_token_id} [PAD]={tok.pad_token_id} [UNK]={tok.unk_token_id}")
print(f"(1) n latih {fmt_id(len(pl))} · P{CFG['q']} = maks latih = {int(pl.max())} → L_max = {L_ulang} (JSON {L})")
print("(2) Tabel 2.14: " + "; ".join(f"{k} → {' '.join(tok.tokenize(k))}" for k in T214) + f" | 2.5.2: {s252}")
print(f"(3) {'Bagian':<9}{'Maks sebelum':>13}{'Maks sesudah':>13}  Terpotong")
for b in BAGIAN:
    print(f"    {b:<9}{max(map(len, penuh[b])):>13}{max(map(len, potong[b])):>13}  {kena[b] or '-'}")
for i in kena["uji"]:
    n = len(penuh["uji"][data["uji"]["review_id"].tolist().index(i)])
    print(f"    review_id {i}: {n} → {L} token ({n - L} subkata terakhir dibuang; {fmt_id(L / n * 100, 1)}% dipertahankan)")
print(f"(4) padding per batch (16 dokumen latih pertama): lebar {bt['input_ids'].shape[1]} = terpanjang {max(nyata)}")
ya = lambda x: "ya" if x else "tidak"
cek("L_max ulang · JSON · n latih", (L_ulang, L, len(pl)), (TARGET["7"]["L_max"],) * 2 + (TARGET["6.1"]["latih"][0],))
cek("Tabel 2.14 cocok (kata)", sum(tok.tokenize(k) == v for k, v in T214.items()), len(T214))
cek("subbab 2.5.2: subkata per kata", s252, [1, 1, 1, 2, 1])
cek("terpotong latih · validasi · uji", [len(kena[b]) for b in BAGIAN], list(TARGET["7"]["terpotong"].values()))
cek("review_id terpotong", sum(kena.values(), []), TARGET["7"]["terpotong_id"])
cek("panjang sesudah potong <= L_max", ya(all(len(c) <= L for b in BAGIAN for c in potong[b])), "ya")
cek("tak terpotong identik · diawali [CLS], diakhiri [SEP]", [ya(utuh), ya(ujung)], ["ya", "ya"])
cek("attention_mask = panjang asli", ya(bt["attention_mask"].sum(1).tolist() == nyata), "ya")
selesai()

## Tahap 8.1 — Tulis `lib/wondr_latih.py` dan uji cepat

**Dasar naskah:** Bab II subbab 2.6.2 dan 2.8; pers. (2.14), (2.15), (2.18)–(2.22); Tabel 3.1. Keputusan teknis KT-5 (*weight decay* tidak dikenakan pada bias dan `LayerNorm.weight`), KT-6 (kepala klasifikasi bawaan dengan *pooler* hasil prapelatihan), KT-7 (konfigurasi lewat `pakai_konfig(CFG)`, *batch* evaluasi = B, FP32, `num_workers=0`, `fitness()` memakai `label_akhir`, uji determinisme) disetujui 21-10-2026.

**Package:** `gc`, `json`, `math`, `time`, `pathlib`, `importlib` (bawaan Python); `numpy`; `pandas`; `torch`; `scikit-learn` (`f1_score`); `transformers` (`AutoModelForSequenceClassification`, `AutoTokenizer`, `get_linear_schedule_with_warmup`); `lib.bantu` (`atur_seed`).

Sel 1/2 menulis `lib/wondr_latih.py` (fungsi saja). Sel 2/2 menguji: T dan T_w, parameter baru, dan pelatihan kecil (64 dokumen, 1 *epoch*) dua kali dengan *seed* sama. Hasil uji tidak disimpan.

In [ ]:
%%writefile lib/wondr_latih.py
# lib/wondr_latih.py -- fine-tuning IndoBERT (Tahap 8.1; KT-5, KT-6, KT-7 disetujui 21-10-2026).
# Isinya fungsi saja. Konfigurasi diambil dari CFG lewat pakai_konfig(CFG) di setiap sel pelatihan.
import gc, json, math, time
from pathlib import Path
import numpy as np, pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from lib.bantu import atur_seed

_AKAR = Path(__file__).resolve().parents[1]          # folder kerja .../skripsi_wondr_v3
_K = {}                                              # konfigurasi aktif (salinan CFG + L_max)
_CACHE = {}                                          # tokenizer dan input_ids per review_id, per sesi
TANPA_WD = ("bias", "LayerNorm.weight")              # KT-5: tanpa weight decay, seperti BERT asli


def pakai_konfig(cfg):
    """Salin CFG dan ambil L_max dari hasil Tahap 7; cache tokenisasi dikosongkan bila L_max/checkpoint berubah."""
    L = json.loads((_AKAR / "hasil" / "tahap07_tokenisasi.json").read_text(encoding="utf-8"))["L_max"]
    if (_K.get("checkpoint"), _K.get("L_max")) != (cfg["checkpoint"], L):
        _CACHE.clear()
    _K.clear(); _K.update(cfg); _K["L_max"] = L
    return L


def jadwal(n_latih, wr):
    """Pers. (2.21)-(2.22): T = ceil(n_latih / B) * E dan T_w = floor(wr * T)."""
    T = math.ceil(n_latih / _K["B"]) * _K["E"]
    return T, math.floor(wr * T)


def _tok():
    if "tok" not in _CACHE:
        _CACHE["tok"] = AutoTokenizer.from_pretrained(_K["checkpoint"])
    return _CACHE["tok"]


def _kode(df):
    """input_ids per dokumen dengan truncation=True, max_length=L_max; dihitung sekali per review_id."""
    ids = _CACHE.setdefault("ids", {})
    baru = [(r, t) for r, t in zip(df["review_id"], df["text_clean"]) if r not in ids]
    if baru:
        hasil = _tok()([t for _, t in baru], truncation=True, max_length=_K["L_max"])["input_ids"]
        ids.update({r: x for (r, _), x in zip(baru, hasil)})
    return [ids[r] for r in df["review_id"]]


def _muat(ids, y, acak, seed=None):
    """DataLoader dengan padding dinamis per batch (kanan, token [PAD]); batch = B; num_workers = 0."""
    pad = _tok().pad_token_id

    def kumpul(batch):
        n = max(len(a) for a, _ in batch)
        x = torch.full((len(batch), n), pad, dtype=torch.long)
        m = torch.zeros((len(batch), n), dtype=torch.long)
        for i, (a, _) in enumerate(batch):
            x[i, :len(a)] = torch.tensor(a)
            m[i, :len(a)] = 1
        return x, m, torch.tensor([b for _, b in batch], dtype=torch.long)

    g = torch.Generator().manual_seed(seed) if acak else None
    return DataLoader(list(zip(ids, y)), batch_size=_K["B"], shuffle=acak, generator=g, collate_fn=kumpul, num_workers=0)


@torch.no_grad()
def _peluang(model, ids, dev):
    """Peluang kelas Positif (softmax) per dokumen, urutan tetap."""
    model.eval()
    p = [torch.softmax(model(input_ids=x.to(dev), attention_mask=m.to(dev)).logits, dim=-1)[:, 1].cpu()
         for x, m, _ in _muat(ids, [0] * len(ids), acak=False)]
    return torch.cat(p).double().numpy()


def tebak(p1):
    """argmax dua kelas dari peluang Positif; seri (0,5) -> kelas 0."""
    return (np.asarray(p1) > 0.5).astype(int)


def parameter_baru():
    """Parameter yang tidak ada di checkpoint saat model klasifikasi dibuat (KT-6: harus hanya lapisan klasifikasi)."""
    _, info = AutoModelForSequenceClassification.from_pretrained(_K["checkpoint"], num_labels=2, output_loading_info=True)
    return sorted(info["missing_keys"]), sorted(info["unexpected_keys"])


def latih(df_latih, lr, wr, wd, seed, df_val=None, early_stopping=True, prediksi=None, maks_epoch=None):
    """Satu kali fine-tuning (Bab II 2.8). maks_epoch hanya untuk uji cepat; bawaan = E."""
    if not _K:
        raise RuntimeError("Panggil wl.pakai_konfig(CFG) lebih dulu.")
    if early_stopping and df_val is None:
        raise ValueError("early_stopping=True memerlukan df_val.")
    t0, dev = time.perf_counter(), torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ids_latih, prediksi = _kode(df_latih), prediksi or {}
    ids_val = _kode(df_val) if df_val is not None else None
    ids_pred = {k: _kode(d) for k, d in prediksi.items()}

    atur_seed(seed)                                  # sebelum model dibuat: bobot awal W_c, b_c bergantung seed
    model = AutoModelForSequenceClassification.from_pretrained(_K["checkpoint"], num_labels=2).to(dev)
    grup = [{"params": [p for n, p in model.named_parameters() if not n.endswith(TANPA_WD)], "weight_decay": wd},
            {"params": [p for n, p in model.named_parameters() if n.endswith(TANPA_WD)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(grup, lr=lr, betas=tuple(_K["adam_betas"]), eps=_K["adam_eps"])     # pers. (2.18)
    T, T_w = jadwal(len(df_latih), wr)
    sch = get_linear_schedule_with_warmup(opt, T_w, T)                  # pers. (2.20); T tetap walau berhenti dini
    muat = _muat(ids_latih, df_latih["label"].astype(int).tolist(), acak=True, seed=seed)

    f1_epoch, terbaik, ep_terbaik, tanpa_naik, prob = [], -1.0, None, 0, {}
    for ep in range(1, (maks_epoch or _K["E"]) + 1):
        model.train()
        for x, m, y in muat:
            loss = model(input_ids=x.to(dev), attention_mask=m.to(dev), labels=y.to(dev)).loss   # (2.15), tanpa bobot kelas
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), _K["v"])                         # pers. (2.19)
            opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        if early_stopping:
            f1 = float(f1_score(df_val["label"].astype(int), tebak(_peluang(model, ids_val, dev)),
                                average="macro", zero_division=0))
            f1_epoch.append(f1)
            if f1 > terbaik:                         # diperbarui hanya bila lebih besar
                terbaik, ep_terbaik, tanpa_naik = f1, ep, 0
                prob = {k: _peluang(model, i, dev) for k, i in ids_pred.items()}   # peluang di epoch terbaik
            else:
                tanpa_naik += 1
                if tanpa_naik >= _K["p"]:
                    break
    if not early_stopping:                           # CL: peluang dari bobot epoch terakhir
        prob = {k: _peluang(model, i, dev) for k, i in ids_pred.items()}

    del model, opt, sch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"f1_terbaik": terbaik if early_stopping else None, "epoch_terbaik": ep_terbaik, "f1_per_epoch": f1_epoch,
            "detik": time.perf_counter() - t0, "prob": prob, "T": T, "T_w": T_w, "perangkat": dev.type}


def fitness(lr, wr, wd):
    """Pers. (2.25): rata-rata f1_terbaik seed 42 dan 43; data Tahap 9.3, label = label_akhir (KT-7)."""
    baca = lambda b: pd.read_parquet(_AKAR / "hasil" / f"tahap09_{b}.parquet").assign(label=lambda d: d["label_akhir"])
    df_latih, df_val = baca("latih"), baca("validasi")
    hasil = {s: latih(df_latih, lr, wr, wd, s, df_val=df_val) for s in _K["seed_fitness"]}
    skor = [hasil[s]["f1_terbaik"] for s in _K["seed_fitness"]]
    return {"fitness": float(np.mean(skor)), "skor": skor, "hasil": hasil}

In [ ]:
# ══ Tahap 8.1 — Uji cepat lib/wondr_latih.py (sel 2/2; hasil tidak disimpan) ══
# Package: importlib (bawaan Python); numpy; pandas; torch; lib.wondr_latih (uji memori GPU: sel tambahan)
import importlib
import numpy as np, pandas as pd, torch
import lib.wondr_latih as wl
importlib.reload(wl)                                        # pakai isi lib/wondr_latih.py yang terbaru
mulai()
T = TARGET["8.1"]
L_max = wl.pakai_konfig(CFG)

# 1) Jadwal untuk n = 3.051 dokumen latih dan skema default (pers. 2.21-2.22)
T_, Tw_ = wl.jadwal(TARGET["6.1"]["latih"][0], CFG["default"]["wr"])
# 2) Parameter baru saat model klasifikasi dibuat (KT-6: hanya lapisan klasifikasi W_c, b_c)
baru, sisa = wl.parameter_baru()
# 3) Pelatihan kecil: 64 dokumen latih + 64 validasi, 1 epoch, skema default, dua kali dengan seed yang sama
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet").head(64)
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet").head(64)
d, s = CFG["default"], CFG["seed_fitness"][0]
r = [wl.latih(lat, d["lr"], d["wr"], d["wd"], s, df_val=val, prediksi={"val": val}, maks_epoch=1) for _ in range(2)]

judul("8.1", "Uji cepat lib/wondr_latih.py", keluaran="(tidak ada; hasil uji tidak disimpan)",
      masukan=["lib/wondr_latih.py", "64 dokumen tahap06_latih + 64 tahap06_validasi"])
print(f"L_max = {L_max} · jadwal n = {fmt_id(TARGET['6.1']['latih'][0])}: T = {T_} · T_w = {Tw_}")
print(f"Parameter baru  : {', '.join(baru)} ({len(baru)} tensor)")
print(f"Tidak terpakai  : {len(sisa)} tensor checkpoint (kepala prapelatihan)")
for i, x in enumerate(r, 1):
    print(f"Pelatihan {i}: lr {d['lr']} · wr {fmt_id(d['wr'], 2)} · wd {fmt_id(d['wd'], 2)} · seed {s} · epoch terbaik "
          f"{x['epoch_terbaik']} · Macro F1 val {fmt_id(x['f1_terbaik'])} · {fmt_id(x['detik'], 1)} detik · {x['perangkat']}")
cek("T · T_w (n = 3.051, rho = 0,1)", (T_, Tw_), (T["T"], T["T_w"]))
cek("parameter baru hanya lapisan klasifikasi", baru, ["classifier.bias", "classifier.weight"])
ya, p = (lambda x: "ya" if x else "tidak"), r[0]["prob"]["val"]
cek("dua pelatihan identik (F1 · peluang)", [ya(r[0]["f1_terbaik"] == r[1]["f1_terbaik"]),
    ya(np.array_equal(p, r[1]["prob"]["val"]))], ["ya", "ya"])
cek("peluang di [0, 1] untuk 64 dokumen", ya(len(p) == 64 and ((p >= 0) & (p <= 1)).all()), "ya")
cek("pelatihan di GPU", r[0]["perangkat"], "cuda")
selesai()

### Tahap 8.1 — Uji memori GPU (tambahan, diminta pengguna)

Model, parameter pelatihan, dan *seed* tidak diubah. Mengukur empat metrik di empat titik (awal setelah ruang kerja dilepas, setelah pelatihan 1, setelah pelatihan 2, setelah ruang kerja cuBLAS dilepas): `torch.cuda.memory_allocated()` (blok *allocator* yang sedang dipakai, termasuk ruang kerja cuBLAS), `torch.cuda.memory_reserved()` (blok yang ditahan *allocator*, termasuk *cache* kosong), jumlah tensor CUDA yang masih hidup di Python, dan memori terpakai menurut *driver* (`torch.cuda.mem_get_info()`, termasuk konteks CUDA). Ruang kerja cuBLAS dilepas dengan `torch._C._cuda_clearCublasWorkspaces()` lalu `torch.cuda.empty_cache()`.

**Package:** `gc`, `os` (bawaan Python); `pandas`; `torch`; `lib.wondr_latih`; `lib.bantu`.

In [ ]:
# ══ Tahap 8.1 — Uji memori GPU (tambahan; model, parameter, dan seed tidak diubah) ══
# Package: gc, os (bawaan Python); pandas; torch; lib.wondr_latih
import gc, os
import pandas as pd, torch
import lib.wondr_latih as wl
mulai()
wl.pakai_konfig(CFG)
MiB = 2**20

def ukur():
    """[teralokasi, dicadangkan, tensor CUDA hidup di Python, terpakai di GPU menurut driver]."""
    gc.collect()
    hidup = sum(1 for o in gc.get_objects() if torch.is_tensor(o) and o.is_cuda)
    bebas, total = torch.cuda.mem_get_info()
    return [torch.cuda.memory_allocated() / MiB, torch.cuda.memory_reserved() / MiB, hidup, (total - bebas) / MiB]

def lepas():                                       # ruang kerja cuBLAS -> allocator, lalu cache allocator -> driver
    torch._C._cuda_clearCublasWorkspaces()
    torch.cuda.empty_cache()

# 1) Ukur di 4 titik dengan pelatihan kecil yang sama seperti sel 2/2 (64 dokumen, 1 epoch, skema default)
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet").head(64)
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet").head(64)
d, s = CFG["default"], CFG["seed_fitness"][0]
lepas(); M = {"awal (setelah dilepas)": ukur()}
for i in (1, 2):
    wl.latih(lat, d["lr"], d["wr"], d["wd"], s, df_val=val, prediksi={"val": val}, maks_epoch=1)
    M[f"setelah pelatihan {i}"] = ukur()
lepas(); M["setelah cuBLAS dilepas"] = ukur()
uk, jml = map(int, os.environ["CUBLAS_WORKSPACE_CONFIG"].strip(":").split(":"))
ws = 2 * uk * jml / 1024 + 1                        # Hitungan: 2 handle x (4.096 KiB x 8) + 1 MiB cuBLASLt (bawaan)

# 2) Ringkasan + cek
judul("8.1", "Uji memori GPU (tambahan)", masukan="lib/wondr_latih.py · 2 pelatihan kecil", keluaran="(tidak ada)")
print(f"{'Titik ukur':<24}{'Teralokasi':>11}{'Dicadangkan':>12}{'Tensor CUDA':>12}{'Terpakai GPU':>13}")
for k, (a, r, n, g) in M.items():
    print(f"{k:<24}{fmt_id(a, 1):>11}{fmt_id(r, 1):>12}{n:>12}{fmt_id(g, 0):>13}")
print("(MB; teralokasi/dicadangkan = allocator PyTorch; terpakai GPU = driver, termasuk konteks CUDA)")
v = list(M.values())
cek("tensor CUDA hidup setelah pelatihan 1 · 2", [v[1][2], v[2][2]], [0, 0])
cek("teralokasi setelah pelatihan 1 · 2 = ruang kerja cuBLAS (MB)", [v[1][0], v[2][0]], [ws, ws], toleransi=0.5)
cek("perubahan pelatihan 1 → 2: teralokasi · dicadangkan · terpakai GPU (MB)",
    [v[2][j] - v[1][j] for j in (0, 1, 3)], [0, 0, 0], toleransi=1)
cek("teralokasi · dicadangkan, awal dan setelah dilepas (MB, nilai aktual)",
    [v[0][0], v[0][1], v[3][0], v[3][1]], [0, 0, 0, 0], toleransi=0.5)
cek("penurunan terpakai GPU saat dilepas = cadangan yang dilepas (MB)", v[2][3] - v[3][3], v[2][1] - v[3][1], toleransi=2)
selesai()

## Tahap 8.2 — *Confident learning*: pelatihan 5 lipatan

**Dasar naskah:** Bab III tahap 8; Bab II subbab 2.3.4, pers. (2.6); Northcutt dkk. (2021). Lipatan: `StratifiedGroupKFold(5, shuffle=True, random_state=42)` dengan `groups=group_key` dan `y=label` (label awal); setiap model memakai skema *default* (lr 2e-5, *warmup ratio* 0,1, *weight decay* 0,01), *seed* 42, 4 *epoch* tanpa *early stopping*. Data uji tidak dibaca.

**Package:** `os`, `importlib` (bawaan Python); `numpy`; `pandas`; `scikit-learn` (`StratifiedGroupKFold`, `f1_score`); `lib.wondr_latih`.

Sel 1/2 melatih 5 model dan menyimpan peluang Positif untuk lipatan yang ditahan (di luar sampel) dan untuk seluruh data validasi ke `log/cl_lipatan_k.parquet`. Lipatan yang sudah tersimpan dilewati, sehingga sel bisa dijalankan ulang bila Colab terputus. Sel 2/2 membaca kelima berkas, menyusun ulang lipatan, lalu memeriksa kelengkapan peluang (berdiri sendiri, tanpa GPU).

In [ ]:
# ══ Tahap 8.2 — Confident learning: pelatihan 5 lipatan (sel 1/2) ══
# Package: os, importlib (bawaan Python); numpy; pandas; scikit-learn (StratifiedGroupKFold, f1_score); lib.wondr_latih
import os, importlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
import lib.wondr_latih as wl
importlib.reload(wl)                                        # pakai isi lib/wondr_latih.py yang terbaru
mulai()
wl.pakai_konfig(CFG)
d, K, s = CFG["default"], CFG["cl_lipatan"], CFG["cl_seed_latih"]

# 1) Baca data latih dan validasi (label awal); data uji tidak dibaca
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet")
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet")

# 2) Lipatan berbasis grup (pers. 2.6), terstratifikasi label awal
lipatan = np.empty(len(lat), dtype=int)
sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=CFG["cl_seed_lipatan"])
for k, (_, idx) in enumerate(sgkf.split(lat, lat["label"], groups=lat["group_key"])):
    lipatan[idx] = k

# 3) Setiap lipatan k: latih pada 4 lipatan lain (skema default, E epoch, tanpa early stopping), lalu prediksi
#    lipatan k dan seluruh data validasi. Satu berkas per lipatan; lipatan yang sudah tersimpan dilewati.
for k in range(K):
    berkas = PATH["log"] / f"cl_lipatan_{k}.parquet"
    tr, te = lat[lipatan != k], lat[lipatan == k]
    if berkas.exists():
        print(f"Lipatan {k}: sudah tersimpan di {rel(berkas)}, dilewati")
        continue
    r = wl.latih(tr, d["lr"], d["wr"], d["wd"], s, early_stopping=False, prediksi={"latih": te, "validasi": val})
    out = pd.concat([pd.DataFrame({"review_id": x["review_id"].to_numpy(), "bagian": b, "p1": r["prob"][b]})
                     for b, x in (("latih", te), ("validasi", val))], ignore_index=True)
    out.assign(lipatan=k, n_latih=len(tr), detik=r["detik"]).to_parquet(berkas.with_suffix(".tmp"), index=False)
    os.replace(berkas.with_suffix(".tmp"), berkas)            # tulis atomik: aman bila Colab terputus
    f1 = f1_score(te["label"], wl.tebak(r["prob"]["latih"]), average="macro", zero_division=0)
    print(f"Lipatan {k}: lr {d['lr']} · wr {fmt_id(d['wr'], 2)} · wd {fmt_id(d['wd'], 2)} · seed {s} · {CFG['E']} epoch · "
          f"latih {fmt_id(len(tr))} · prediksi {fmt_id(len(te))} + {fmt_id(len(val))} · Macro F1 lipatan {fmt_id(f1)} · "
          f"{fmt_id(r['detik'] / 60, 1)} menit")
selesai()

In [ ]:
# ══ Tahap 8.2 — Confident learning: ringkasan dan cek 5 lipatan (sel 2/2) ══
# Package: numpy; pandas; scikit-learn (StratifiedGroupKFold, f1_score); lib.wondr_latih (tebak)
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
import lib.wondr_latih as wl
mulai()
K = CFG["cl_lipatan"]

# 1) Baca data latih/validasi dan kelima berkas lipatan; lipatan disusun ulang untuk mencocokkan berkas
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet")
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet")
cl = pd.concat([pd.read_parquet(PATH["log"] / f"cl_lipatan_{k}.parquet") for k in range(K)], ignore_index=True)
lipatan = np.empty(len(lat), dtype=int)
sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=CFG["cl_seed_lipatan"])
for k, (_, idx) in enumerate(sgkf.split(lat, lat["label"], groups=lat["group_key"])):
    lipatan[idx] = k
oof = cl[cl.bagian == "latih"].merge(lat[["review_id", "label", "group_key"]], on="review_id", how="left")
f1 = lambda g: f1_score(g["label"], wl.tebak(g["p1"]), average="macro", zero_division=0)

# 2) Ringkasan per lipatan (Macro F1 terhadap label awal, deskriptif) + cek
judul("8.2", "Confident learning: 5 lipatan", masukan="hasil/tahap06_latih.parquet, hasil/tahap06_validasi.parquet",
      keluaran=f"log/cl_lipatan_0..{K - 1}.parquet ({fmt_id(len(cl))} baris)")
print(f"{'k':<3}{'n latih':>9}{'n lipatan':>11}{'n validasi':>12}{'Non-Positif':>13}{'Macro F1':>10}{'Durasi':>11}")
for k, g in oof.groupby("lipatan"):
    m = cl[cl.lipatan == k]
    print(f"{k:<3}{fmt_id(int(m.n_latih.iloc[0])):>9}{fmt_id(len(g)):>11}{fmt_id(int((m.bagian == 'validasi').sum())):>12}"
          f"{fmt_id((g.label == 0).mean() * 100, 1) + '%':>13}{fmt_id(f1(g)):>10}{fmt_id(m.detik.iloc[0] / 60, 1) + ' mnt':>11}")
garis()
print(f"Macro F1 di luar sampel, seluruh data latih (label awal, deskriptif): {fmt_id(f1(oof))} · "
      f"total durasi {fmt_id(cl.groupby('lipatan').detik.first().sum() / 60, 1)} menit")
ya = lambda x: "ya" if x else "tidak"
cek("dokumen latih dengan tepat 1 peluang di luar sampel", [int(oof.review_id.nunique()), ya(oof.review_id.is_unique)],
    [len(lat), "ya"])
cek("lipatan di berkas = lipatan yang disusun ulang", ya(oof[["review_id", "lipatan"]].merge(pd.DataFrame(
    {"review_id": lat.review_id, "k": lipatan}), on="review_id", how="outer").eval("lipatan == k").all()), "ya")
cek("group_key terbelah antarlipatan", int((oof.groupby("group_key").lipatan.nunique() > 1).sum()), 0)
cek("dokumen validasi dengan tepat 5 peluang", int((cl[cl.bagian == "validasi"].groupby("review_id").lipatan.nunique()
    == K).sum()), len(val))
cek("peluang di [0, 1] · tanpa NaN", [ya(cl.p1.between(0, 1).all()), ya(cl.p1.notna().all())], ["ya", "ya"])
cek("review_id di luar latih/validasi (data uji tidak disentuh)",
    len(set(cl.review_id) - set(lat.review_id) - set(val.review_id)), 0)
selesai()

## Tahap 8.3 — Ambang, *confident joint*, dan kandidat

**Dasar naskah:** Bab III tahap 8; Bab II subbab 2.3.4, pers. (2.3)–(2.4), Tabel 2.9; Northcutt dkk. (2021). Rumus ditulis sendiri; pustaka `cleanlab` tidak dipakai. Ambang dan *confident joint* dihitung terpisah untuk data latih (peluang di luar sampel) dan data validasi (rata-rata peluang 5 model). Data uji tidak dibaca.

**Package:** `math`, `importlib` (bawaan Python); `numpy`; `pandas`; `lib.wondr_label` (`confident_learning`, `per_bagian`).

Sel 1/2 menulis `lib/wondr_label.py` (fungsi saja). Sel 2/2 menguji fungsi dengan contoh Tabel 2.9, menerapkannya pada latih dan validasi, lalu menyimpan `hasil/tahap08_kandidat.parquet` dan `bab4/tabel/tahap08_cj.csv`. Dokumen yang tidak memenuhi ambang kelas mana pun tidak dihitung (`kelas_taksiran` kosong) dan bukan kandidat.

In [ ]:
%%writefile lib/wondr_label.py
# lib/wondr_label.py -- confident learning dua kelas (Tahap 8.3). Rumus ditulis sendiri sesuai pers. (2.3)-(2.4),
# Bab II subbab 2.3.4; pustaka cleanlab tidak dipakai. Isinya fungsi saja.
import math
import numpy as np, pandas as pd


def confident_learning(p1, y):
    """Ambang kelas, kelas taksiran, confident joint, dan kandidat label salah; p0 = 1 - p1.

    t_j      = rata-rata p_j pada dokumen berlabel j                                   (pers. 2.3)
    taksiran = argmax p_l di antara kelas l dengan p_l >= t_l; -1 bila tidak ada yang memenuhi ambang
               (dokumen itu tidak dihitung); bila p0 = p1 dan keduanya lolos, dipilih kelas 0
    C[i, j]  = banyak dokumen berlabel i dengan kelas taksiran j; baris = label, kolom = taksiran (pers. 2.4)
    kandidat = dokumen yang dihitung dengan kelas taksiran != label (unsur di luar diagonal C)
    """
    p1, y = np.asarray(p1, dtype=float), np.asarray(y)
    if not (set(np.unique(y)) == {0, 1} and np.all((p1 >= 0) & (p1 <= 1))):
        raise ValueError("y harus memuat kelas 0 dan 1 saja, dan p1 harus di [0, 1] tanpa NaN.")
    y = y.astype(int)
    P = np.column_stack([1.0 - p1, p1])                       # kolom j = peluang kelas j
    t = np.array([math.fsum(P[y == j, j]) / (y == j).sum() for j in (0, 1)])   # pers. (2.3); jumlah dibulatkan tepat
    lolos = P >= t
    taksiran = np.where(lolos.any(axis=1), np.where(lolos, P, -np.inf).argmax(axis=1), -1)
    dihitung = taksiran >= 0
    C = np.zeros((2, 2), dtype=int)
    np.add.at(C, (y[dihitung], taksiran[dihitung]), 1)        # pers. (2.4)
    return {"t": t, "taksiran": taksiran, "dihitung": dihitung, "C": C, "kandidat": dihitung & (taksiran != y)}


def per_bagian(d):
    """confident_learning() terpisah untuk setiap bagian (Bab III tahap 8); d berkolom review_id, bagian, label, p1.

    Mengembalikan (tabel per dokumen: review_id, bagian, label, p1, kelas_taksiran [<NA> = tidak dihitung], kandidat;
    tabel per bagian: n, t0, t1, tidak_dihitung, C00, C01, C10, C11, kandidat, kandidat_pct)."""
    dok, cj = [], []
    for b, x in d.groupby("bagian", sort=False):
        r = confident_learning(x["p1"], x["label"])
        taksiran = pd.array(np.where(r["dihitung"], r["taksiran"], pd.NA), dtype="Int64")
        dok.append(x[["review_id", "bagian", "label", "p1"]].assign(kelas_taksiran=taksiran, kandidat=r["kandidat"]))
        cj.append({"bagian": b, "n": len(x), "t0": r["t"][0], "t1": r["t"][1], "tidak_dihitung": int((~r["dihitung"]).sum()),
                   **{f"C{i}{j}": int(r["C"][i, j]) for i in (0, 1) for j in (0, 1)}, "kandidat": int(r["kandidat"].sum())})
    cj = pd.DataFrame(cj)
    cj["kandidat_pct"] = 100 * cj["kandidat"] / cj["n"]
    return pd.concat(dok, ignore_index=True), cj

In [ ]:
# ══ Tahap 8.3 — Ambang, confident joint, dan kandidat (sel 2/2) ══
# Package: importlib (bawaan Python); numpy; pandas; lib.wondr_label (confident_learning, per_bagian)
import importlib
import numpy as np, pandas as pd
import lib.wondr_label as wlab
importlib.reload(wlab)                                      # pakai isi lib/wondr_label.py yang terbaru
mulai()
T, K, n_uji, ya = TARGET["8.3"], CFG["cl_lipatan"], TARGET["6.1"]["uji"][0], (lambda x: "ya" if x else "tidak")

# 1) Peluang Positif: latih = di luar sampel (1 model); validasi = rata-rata 5 model. Data uji tidak dibaca.
cl = pd.concat([pd.read_parquet(PATH["log"] / f"cl_lipatan_{k}.parquet") for k in range(K)])
p = cl.groupby(["bagian", "review_id"], as_index=False).agg(p1=("p1", "mean"), n_model=("p1", "size"))
d = pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet", columns=["review_id", "label"]).assign(bagian=b)
               for b in ("latih", "validasi")]).merge(p, on=["bagian", "review_id"], how="left")

# 2) Ambang (2.3), confident joint (2.4), dan kandidat, terpisah untuk latih dan validasi; lalu simpan
dok, cj = wlab.per_bagian(d)
dok.to_parquet(PATH["hasil"] / "tahap08_kandidat.parquet", index=False)
simpan_tabel(cj, "tahap08_cj")

# 3) Ringkasan dan cek; fungsi diuji dengan contoh Tabel 2.9 (d1-d6)
u = wlab.confident_learning([0.95, 0.90, 0.15, 0.10, 0.20, 0.85], [1, 1, 1, 0, 0, 0])
judul("8.3", "Ambang, confident joint, dan kandidat", masukan=f"log/cl_lipatan_0..{K - 1}.parquet, tahap06 latih/validasi",
      keluaran=[f"hasil/tahap08_kandidat.parquet ({fmt_id(len(dok))} baris)", "bab4/tabel/tahap08_cj.csv"])
print(f"{'Bagian':<10}{'n':>6}{'t0':>8}{'t1':>8}{'C00':>7}{'C01 NP→P':>10}{'C10 P→NP':>10}{'C11':>7}{'Kandidat':>15}"
      f"{'Tdk dihitung':>14}")
for c in cj.itertuples():
    print(f"{c.bagian:<10}{fmt_id(c.n):>6}{fmt_id(c.t0):>8}{fmt_id(c.t1):>8}{fmt_id(c.C00):>7}{fmt_id(c.C01):>10}"
          f"{fmt_id(c.C10):>10}{fmt_id(c.C11):>7}{fmt_id(c.kandidat) + ' (' + fmt_id(c.kandidat_pct, 1) + '%)':>15}"
          f"{fmt_id(c.tidak_dihitung):>14}")
print(f"C: baris = label awal, kolom = kelas taksiran · dokumen untuk lembar anotasi (Tahap 9.1): "
      f"{' + '.join(fmt_id(k) for k in cj.kandidat)} + uji {fmt_id(n_uji)} = {fmt_id(int(cj.kandidat.sum()) + n_uji)}")
cek("contoh Tabel 2.9: t0 · t1", u["t"].tolist(), [T["t0"], T["t1"]], toleransi=5e-5)
cek("contoh Tabel 2.9: C · kandidat (d)", [u["C"].tolist(), (np.flatnonzero(u["kandidat"]) + 1).tolist()], [T["C"], T["kandidat"]])
cek("peluang per dokumen: latih 1 model · validasi 5 model", [ya(d.query("bagian == 'latih'").n_model.eq(1).all()),
    ya(d.query("bagian == 'validasi'").n_model.eq(K).all())], ["ya", "ya"])
for c in cj.itertuples():
    cek(f"{c.bagian}: jumlah kandidat", c.kandidat, 0, cara=">")
    cek(f"{c.bagian}: kandidat (%)", c.kandidat_pct, 40, cara="<=", d=1)
    cek(f"{c.bagian}: diagonal C dominan (C00 > C01 · C11 > C10)", [ya(c.C00 > c.C01), ya(c.C11 > c.C10)], ["ya", "ya"])
selesai()

### Tahap 8.3 — Audit (tambahan, diminta pengguna)

Hanya membaca berkas; tidak ada yang ditulis atau diubah. Semua angka dihitung ulang secara mandiri tanpa `lib.wondr_label`. **Audit A** memeriksa sumber peluang (latih: 1 model di luar sampel; validasi: rata-rata 5 model), ambang pers. (2.3), kecocokan p₁ dan label dengan log CL dan tahap06, kolom berkas, serta data uji (hanya `review_id`: jumlah 656 dan tidak beririsan). **Audit B** memeriksa aturan ambang pers. (2.4): pita dokumen yang tidak dihitung (1 − t₀ < p₁ < t₁), kelas taksiran, C, kandidat = C₀₁ + C₁₀, penyebut persentase, dan perbandingan dengan argmax biasa.

**Package:** `numpy`; `pandas`.

In [ ]:
# ══ Tahap 8.3 — Audit A: sumber peluang, ambang, dan isi berkas (tambahan; hanya membaca berkas) ══
# Package: pandas  (dihitung ulang secara mandiri, tanpa lib.wondr_label)
import pandas as pd
mulai()
K, n_lat, ya = CFG["cl_lipatan"], TARGET["6.1"]["latih"][0], (lambda x: "ya" if x else "tidak")
e = lambda v: f"{v:.1e}".replace(".", ",")

# 1) Baca keluaran 8.3, log lipatan CL, label awal tahap06, dan review_id data uji (tanpa teks dan label)
kd = pd.read_parquet(PATH["hasil"] / "tahap08_kandidat.parquet")
cj = pd.read_csv(PATH["tabel"] / "tahap08_cj.csv").set_index("bagian")
cl = pd.concat([pd.read_parquet(PATH["log"] / f"cl_lipatan_{k}.parquet") for k in range(K)])
lab = pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet", columns=["review_id", "label"]).assign(bagian=b)
                 for b in ("latih", "validasi")])
uji = pd.read_parquet(PATH["hasil"] / "tahap06_uji.parquet", columns=["review_id"]).review_id
# 2) Peluang dari log: latih = 1 model (lipatan yang tidak melatihnya), validasi = rata-rata 5 model; lalu ambang (2.3)
sumber = cl.groupby(["bagian", "review_id"]).agg(p1=("p1", "mean"), n_model=("p1", "size")).reset_index()
u = (lab.merge(sumber, on=["bagian", "review_id"], how="left")
        .merge(kd.set_index(["bagian", "review_id"]).add_suffix("_b").reset_index(), on=["bagian", "review_id"],
               how="outer", indicator=True))
lip = cl[cl.bagian == "latih"].groupby("lipatan").agg(n_latih=("n_latih", "first"), n_pred=("review_id", "size"))
judul("8.3", "Audit A: sumber peluang dan ambang", masukan=["hasil/tahap08_kandidat.parquet + bab4/tabel/tahap08_cj.csv",
      f"log/cl_lipatan_0..{K - 1}.parquet + tahap06 (uji: review_id saja)"])
print("Kolom berkas 8.3:", " · ".join(f"{c} ({t})" for c, t in kd.dtypes.astype(str).items()))
print("Lipatan 0-4, n latih + n diprediksi:", " · ".join(f"{fmt_id(int(r.n_latih))} + {fmt_id(int(r.n_pred))}" for _, r in lip.iterrows()))
print(f"{'Bagian':<9}{'n':>6}{'model/dok':>10}{'t0 ulang':>10}{'t0 CSV':>9}{'t1 ulang':>10}{'t1 CSV':>9}{'|Δt| maks':>11}"
      f"{'|Δp1| maks':>12}")
R = []
for b, x in u.groupby("bagian", sort=False):
    c, y = cj.loc[b], x.label
    t0, t1 = (1 - x.p1[y == 0]).mean(), x.p1[y == 1].mean()          # pers. (2.3), p0 = 1 - p1
    dt, dp = max(abs(t0 - c.t0), abs(t1 - c.t1)), float((x.p1 - x.p1_b).abs().max())
    print(f"{b:<9}{fmt_id(len(x)):>6}{'/'.join(map(str, sorted(x.n_model.unique()))):>10}{fmt_id(t0):>10}{fmt_id(c.t0):>9}"
          f"{fmt_id(t1):>10}{fmt_id(c.t1):>9}{e(dt):>11}{e(dp):>12}")
    R.append((b, [int((x._merge == "both").sum()), int(x.p1.isna().sum()), ya((y == x.label_b).all()),
                  ya(x.n_model.eq(1 if b == "latih" else K).all()), ya(max(dt, dp) <= 1e-12)], [len(x), 0, "ya", "ya", "ya"]))
for b, aktual, target in R:
    cek(f"{b}: baris berpasangan · p1 NaN · label = tahap06 · model per dokumen · |Δt|, |Δp1| ≤ 1e-12", aktual, target)
cek("setiap lipatan: n latih + n diprediksi = n data latih", ya((lip.n_latih + lip.n_pred == n_lat).all()), "ya")
cek("kolom berkas = spesifikasi 8.3", list(kd.columns), ["review_id", "bagian", "label", "p1", "kelas_taksiran", "kandidat"])
cek("uji: n dari berkas · irisan dengan berkas 8.3 · irisan dengan log CL", [len(uji), len(set(uji) & set(kd.review_id)),
    len(set(uji) & set(cl.review_id))], [TARGET["6.1"]["uji"][0], 0, 0])
selesai()

In [ ]:
# ══ Tahap 8.3 — Audit B: aturan ambang, dokumen tidak dihitung, C, dan kandidat (tambahan; hanya membaca berkas) ══
# Package: numpy; pandas  (dihitung ulang secara mandiri, tanpa lib.wondr_label)
import numpy as np, pandas as pd
mulai()
ya = lambda x: "ya" if x else "tidak"

# 1) Baca keluaran 8.3 (p1 dan label sudah dicocokkan dengan log CL dan tahap06 di Audit A)
kd = pd.read_parquet(PATH["hasil"] / "tahap08_kandidat.parquet")
cj = pd.read_csv(PATH["tabel"] / "tahap08_cj.csv").set_index("bagian")

# 2) Hitung ulang per bagian dengan aturan pers. (2.4): kelas j lolos bila p_j >= t_j; argmax hanya di antara yang lolos
judul("8.3", "Audit B: aturan ambang, C, dan kandidat", masukan="hasil/tahap08_kandidat.parquet + bab4/tabel/tahap08_cj.csv")
print(f"{'Bagian':<9}{'pita tidak dihitung':>24}{'p1 tdk dihitung':>17}{'tdk NP/P':>10}{'dua lolos':>10}{'salah argmax':>13}"
      f"{'kandidat':>9}{'⊆ argmax':>9}{'salah argmax di pita':>21}")
R = []
for b, x in kd.groupby("bagian", sort=False):
    p1, y, c = x.p1.to_numpy(), x.label.to_numpy(), cj.loc[b]
    t0, t1 = (1 - p1[y == 0]).mean(), p1[y == 1].mean()
    l0, l1 = (1 - p1) >= t0, p1 >= t1
    tak = np.where(l0 & l1, (p1 > 1 - p1).astype(int), np.where(l1, 1, np.where(l0, 0, -1)))
    kand, salah, tdk = (tak >= 0) & (tak != y), (p1 > 0.5).astype(int) != y, tak < 0     # salah = argmax biasa != label
    C = [[int(((y == i) & (tak == j)).sum()) for j in (0, 1)] for i in (0, 1)]
    print(f"{b:<9}{fmt_id(1 - t0) + ' < p1 < ' + fmt_id(t1):>24}{fmt_id(p1[tdk].min()) + '–' + fmt_id(p1[tdk].max()):>17}"
          f"{fmt_id(int((tdk & (y == 0)).sum())) + '/' + fmt_id(int((tdk & (y == 1)).sum())):>10}{fmt_id(int((l0 & l1).sum())):>10}"
          f"{fmt_id(int(salah.sum())):>13}{fmt_id(int(kand.sum())):>9}{ya(not (kand & ~salah).any()):>9}"
          f"{fmt_id(int((salah & tdk).sum())):>21}")
    R.append((b, x, c, tak, kand, C, int(((p1 > 1 - t0) & (p1 < t1)).sum())))
for b, x, c, tak, kand, C, pita in R:
    cek(f"{b}: tidak dihitung = dokumen di pita · kelas_taksiran kosong", [int((tak < 0).sum()), int(x.kelas_taksiran.isna()
        .sum())], [pita, int(c.tidak_dihitung)])
    cek(f"{b}: taksiran · C · kandidat = berkas/CSV", [ya((x.kelas_taksiran.fillna(-1).to_numpy() == tak).all()),
        ya(C == [[c.C00, c.C01], [c.C10, c.C11]]), ya((x.kandidat.to_numpy() == kand).all())], ["ya"] * 3)
    cek(f"{b}: kandidat = C01 + C10 · % kandidat (penyebut n = {fmt_id(len(x))})", [int(kand.sum()),
        round(100 * kand.sum() / len(x), 4)], [int(c.C01 + c.C10), round(c.kandidat_pct, 4)])
selesai()

### Tahap 8.3 — Audit C: kelas taksiran, pita tidak dihitung, dan batas ambang (tambahan, diminta pengguna)

Hanya membaca berkas. Menampilkan tabel silang status CL (kandidat / dihitung dan sesuai label / tidak dihitung) × argmax biasa (p₁ > 0,5) terhadap label awal, contoh dokumen yang salah menurut argmax tetapi berada di pita tidak dihitung (1 − t₀ < p₁ < t₁), dokumen terdekat ke kedua batas ambang, dan uji kecil bahwa p tepat sama dengan ambang ikut dihitung (batas inklusif ≥, pers. 2.4).

**Package:** `numpy`; `pandas`; `lib.wondr_label` (`confident_learning`, hanya untuk uji batas).

In [ ]:
# ══ Tahap 8.3 — Audit C: kelas taksiran, pita tidak dihitung, dan batas ambang (tambahan; hanya membaca berkas) ══
# Package: numpy; pandas; lib.wondr_label (confident_learning, hanya untuk uji batas)
import numpy as np, pandas as pd
import lib.wondr_label as wlab
mulai()
ya = lambda x: "ya" if x else "tidak"

# 1) Baca keluaran 8.3; ambang per bagian dari CSV; argmax biasa = tebakan p1 > 0,5 (KT-7)
x = pd.read_parquet(PATH["hasil"] / "tahap08_kandidat.parquet").join(
    pd.read_csv(PATH["tabel"] / "tahap08_cj.csv").set_index("bagian")[["t0", "t1"]], on="bagian")
x = x.assign(p0=1 - x.p1, argmax=(x.p1 > 0.5).astype(int), lolos0=(1 - x.p1) >= x.t0, lolos1=x.p1 >= x.t1)
x["status"] = np.select([x.kandidat, x.kelas_taksiran.notna()], ["kandidat CL", "dihitung, = label"], "tidak dihitung")
x["argmax_vs_label"] = np.where(x.argmax == x.label, "argmax benar", "argmax salah")

# 2) Status CL x argmax; contoh dokumen salah-argmax di pita (review_id terkecil per label) + 1 kandidat pembanding
judul("8.3", "Audit C: kelas taksiran dan pita", masukan="hasil/tahap08_kandidat.parquet + bab4/tabel/tahap08_cj.csv")
print(pd.crosstab([x.bagian, x.status], x.argmax_vs_label, margins=True, margins_name="total").map(fmt_id).to_string(), "\n")
pita = x[x.kelas_taksiran.isna() & (x.argmax != x.label)]
contoh = pd.concat([pita.sort_values("review_id").groupby(["bagian", "label"]).head(1),
                    x[x.kandidat].sort_values("review_id").groupby("bagian").head(1)]).sort_values(["bagian", "kandidat"])
kol = ["bagian", "review_id", "label", "p1", "p0", "t0", "t1", "lolos0", "lolos1", "argmax", "kelas_taksiran", "kandidat"]
print(contoh[kol].to_string(index=False, formatters={k: (lambda v: fmt_id(v)) for k in ("p1", "p0", "t0", "t1")}), "\n")
# 3) Dokumen terdekat ke batas ambang: selisih terkecil di kedua sisi; jumlah yang tepat sama dengan ambang
for b, g in x.groupby("bagian", sort=False):
    d1, d0 = g.p1 - g.t1, g.p0 - g.t0                         # >= 0 berarti lolos (batas inklusif)
    print(f"{b:<9} p1 − t1: terdekat di bawah {fmt_id(d1[d1 < 0].max(), 6)} · di atas/sama {fmt_id(d1[d1 >= 0].min(), 6)} · "
          f"tepat sama {int((d1 == 0).sum())} | p0 − t0: terdekat di bawah {fmt_id(d0[d0 < 0].max(), 6)} · "
          f"di atas/sama {fmt_id(d0[d0 >= 0].min(), 6)} · tepat sama {int((d0 == 0).sum())}")
u = wlab.confident_learning([0.75, 0.75, 0.25, 0.25], [1, 1, 0, 0])   # t0 = t1 = 0,75; semua p tepat = ambang
for b, g in x.groupby("bagian", sort=False):
    cek(f"{b}: kosong ⇔ tidak lolos t0 maupun t1 · kandidat ⊆ argmax salah · dihitung & = label ⇒ argmax benar",
        [ya((g.kelas_taksiran.isna() == ~(g.lolos0 | g.lolos1)).all()), ya((g.argmax[g.kandidat] != g.label[g.kandidat]).all()),
         ya((g.argmax == g.label)[g.status == "dihitung, = label"].all())], ["ya"] * 3)
cek("uji batas: p tepat = ambang ikut dihitung (inklusif ≥) · taksiran", [ya(u["dihitung"].all()), u["taksiran"].tolist()],
    ["ya", [1, 1, 0, 0]])
selesai()

## Tahap 9.1 — Ekspor lembar anotasi

**Dasar naskah:** Bab III tahap 9; Bab II subbab 2.3.2 (makna kelas) dan 2.3.5. Kandidat latih dan validasi dari Tahap 8.3 digabung dengan **seluruh** data uji, diacak dengan `default_rng(2026)`, lalu diberi `id_anotasi` D0001, D0002, dan seterusnya. Lembar anotator A dan B identik dan hanya memuat `id_anotasi, teks, label`, tanpa *rating*, label awal, prediksi, atau asal bagian. Teks = `text_clean` (KT-10). Dari data uji hanya `review_id` dan `text_clean` yang dibaca.

**Package:** `importlib` (bawaan Python); `numpy`; `pandas`; `openpyxl` (`lib.wondr_anotasi`).

Sel 1/2 menulis `lib/wondr_anotasi.py`: teks petunjuk, `tulis_lembar()` (validasi Excel 0/1, teks selalu disimpan sebagai string, dan **tidak menimpa lembar yang sudah diisi anotator**), serta `baca_lembar()` untuk Tahap 9.2. Sel 2/2 membuat `anotasi/lembar_anotator_A.xlsx`, `anotasi/lembar_anotator_B.xlsx`, dan `anotasi/kunci_anotasi.csv` (`id_anotasi, review_id, bagian, kandidat`; **tidak** diberikan ke anotator). Setelah ini program berhenti sampai kedua lembar selesai dianotasi.

In [ ]:
%%writefile lib/wondr_anotasi.py
# lib/wondr_anotasi.py -- lembar anotasi Excel (Tahap 9). Isinya teks petunjuk dan fungsi saja.
import pandas as pd
from openpyxl.styles import Alignment, Font
from openpyxl.worksheet.datavalidation import DataValidation

PETUNJUK = [  # Bab III tahap 9; makna kelas Bab II subbab 2.3.2; contoh teks dari Tabel 2.8 (tanpa rating)
    "PETUNJUK ANOTASI SENTIMEN ULASAN APLIKASI WONDR BY BNI",
    "",
    "Tugas: baca setiap teks pada sheet 'Anotasi', lalu isi kolom 'label' dengan 1 atau 0.",
    "  1 = Positif     : ulasan menyatakan kepuasan atau pujian tanpa keluhan.",
    "  0 = Non-Positif : keluhan, kritik, ulasan netral, atau ulasan bernada campuran (memuat pujian sekaligus keluhan).",
    "",
    "Contoh:",
    "  'aplikasi ini sangat membantu, transaksi jadi cepat'  -> 1",
    "  'aplikasi bagus, tetapi notifikasi sering terlambat'  -> 0 (campuran)",
    "  'verifikasi gagal terus padahal data sudah benar'     -> 0 (keluhan)",
    "",
    "Aturan:",
    "  1. Nilai hanya dari isi teks. Teks sudah dibersihkan: huruf kecil, emoji dan simbol dihapus, huruf dan tanda baca",
    "     yang berulang dipendekkan.",
    "  2. Kerjakan sendiri; jangan berdiskusi dengan anotator lain sampai seluruh lembar selesai.",
    "  3. Isi setiap baris dengan 1 atau 0 saja; jangan ada baris yang kosong.",
    "  4. Jangan mengubah kolom id_anotasi dan teks, urutan baris, maupun nama sheet.",
    "  5. Simpan dengan nama berkas yang sama, lalu serahkan kepada peneliti.",
]


def tulis_lembar(path, data, kolom_isian, petunjuk=None):
    """Tulis xlsx: sheet 'Petunjuk' (opsional) lalu sheet 'Anotasi' = data. Kolom isian dibiarkan seperti di data
    dan diberi validasi Excel 0/1; teks ditulis sebagai string (tidak pernah ditafsirkan sebagai rumus).
    Berkas yang kolom isiannya sudah terisi (hasil kerja anotator) tidak pernah ditimpa."""
    if path.exists() and baca_lembar(path)[kolom_isian].notna().any():
        raise FileExistsError(f"{path.name} sudah berisi isian '{kolom_isian}'; berkas tidak ditimpa.")
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        if petunjuk is not None:
            pd.DataFrame({"p": petunjuk}).to_excel(w, sheet_name="Petunjuk", index=False, header=False)
            w.book["Petunjuk"].column_dimensions["A"].width = 120
        data.to_excel(w, sheet_name="Anotasi", index=False)
        ws, n = w.book["Anotasi"], len(data) + 1
        for j, nama in enumerate(data.columns, start=1):
            huruf = ws.cell(1, j).column_letter
            ws.column_dimensions[huruf].width = 90 if nama == "teks" else 12
            ws.cell(1, j).font = Font(bold=True)
            for i in range(2, n + 1):
                c = ws.cell(i, j)
                if isinstance(c.value, str):
                    c.data_type = "s"                          # cegah teks berawalan '=' menjadi rumus
                if nama == "teks":
                    c.alignment = Alignment(wrap_text=True, vertical="top")
            if nama == kolom_isian:
                dv = DataValidation(type="list", formula1='"0,1"', allow_blank=True, showErrorMessage=True,
                                    errorTitle="Nilai tidak sah", error="Isi dengan 1 (Positif) atau 0 (Non-Positif).")
                dv.add(f"{huruf}2:{huruf}{n}")
                ws.add_data_validation(dv)
        ws.freeze_panes = "A2"


def baca_lembar(path):
    """Baca sheet 'Anotasi' apa adanya: teks tidak diubah menjadi NaN; sel kosong tetap kosong (NaN)."""
    return pd.read_excel(path, sheet_name="Anotasi", dtype={"id_anotasi": str, "teks": str}, keep_default_na=False,
                         na_values=[""])

In [ ]:
# ══ Tahap 9.1 — Ekspor lembar anotasi (sel 2/2) ══
# Package: importlib (bawaan Python); numpy; pandas; openpyxl (lewat lib.wondr_anotasi)
import importlib
import numpy as np, pandas as pd
import lib.wondr_anotasi as wa
importlib.reload(wa)                                        # pakai isi lib/wondr_anotasi.py yang terbaru
mulai()
ya, A = (lambda x: "ya" if x else "tidak"), PATH["anotasi"]

# 1) Kandidat latih dan validasi (Tahap 8.3) + seluruh data uji. Teks = text_clean (KT-10). Dari data uji hanya
#    review_id dan text_clean yang dibaca (tanpa rating dan label).
kd = pd.read_parquet(PATH["hasil"] / "tahap08_kandidat.parquet", columns=["review_id", "bagian", "kandidat"])
teks = {b: pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet", columns=["review_id", "text_clean"])
        for b in ("latih", "validasi", "uji")}
sumber = pd.concat([kd[kd.kandidat & (kd.bagian == b)].merge(teks[b], on="review_id").sort_values("review_id")
                    for b in ("latih", "validasi")] + [teks["uji"].assign(bagian="uji", kandidat=False)
                    .sort_values("review_id")], ignore_index=True)        # urutan dasar: bagian, lalu review_id

# 2) Acak dengan default_rng(2026), beri id_anotasi D0001..., simpan kunci (bukan untuk anotator) dan 2 lembar identik
lembar = sumber.iloc[np.random.default_rng(CFG["seed_anotasi"]).permutation(len(sumber))].reset_index(drop=True)
lembar.insert(0, "id_anotasi", [f"D{i:04d}" for i in range(1, len(lembar) + 1)])
lembar[["id_anotasi", "review_id", "bagian", "kandidat"]].to_csv(A / "kunci_anotasi.csv", index=False)
isi = pd.DataFrame({"id_anotasi": lembar.id_anotasi, "teks": lembar.text_clean, "label": None})
for a in "AB":
    wa.tulis_lembar(A / f"lembar_anotator_{a}.xlsx", isi, kolom_isian="label", petunjuk=wa.PETUNJUK)

# 3) Baca ulang kedua lembar dan kunci untuk cek; ringkasan per sumber
L, k = {a: wa.baca_lembar(A / f"lembar_anotator_{a}.xlsx") for a in "AB"}, pd.read_csv(A / "kunci_anotasi.csv")
cj, n = pd.read_csv(PATH["tabel"] / "tahap08_cj.csv").set_index("bagian"), k.bagian.value_counts()[["latih", "validasi", "uji"]]
judul("9.1", "Ekspor lembar anotasi", masukan=["hasil/tahap08_kandidat.parquet", "hasil/tahap06_{latih,validasi,uji}.parquet"],
      keluaran=["anotasi/lembar_anotator_A.xlsx · lembar_anotator_B.xlsx (identik)", "anotasi/kunci_anotasi.csv (tidak untuk anotator)"])
for nama, v in zip(["kandidat latih", "kandidat validasi", "uji (seluruhnya)", "total"], [*n, n.sum()]):
    print(f"{nama:<20}{fmt_id(int(v)):>7}")
cek("jumlah: kandidat latih · kandidat validasi · uji", n.tolist(), [*cj.kandidat[["latih", "validasi"]], TARGET["6.1"]["uji"][0]])
cek("sheet · kolom lembar", [pd.ExcelFile(A / "lembar_anotator_A.xlsx").sheet_names, list(L["A"].columns)],
    [["Petunjuk", "Anotasi"], ["id_anotasi", "teks", "label"]])
cek("lembar A = lembar B · teks = text_clean · label kosong", [ya(L["A"].equals(L["B"])), ya(L["A"].teks.equals(isi.teks)),
    int(L["A"].label.notna().sum())], ["ya", "ya", 0])
cek("id_anotasi = kunci · review_id unik · teks unik", [ya(L["A"].id_anotasi.equals(k.id_anotasi)), int(k.review_id.nunique()),
    int(L["A"].teks.nunique())], ["ya", len(k), len(k)])
selesai()

## Tahap 9.2 — Impor anotasi dan kesepakatan

**Dasar naskah:** Bab III tahap 9; Bab II subbab 2.3.5, pers. (2.5), Tabel 2.10 (Landis dan Koch). Kedua lembar yang sudah diisi divalidasi terhadap `kunci_anotasi.csv`: id lengkap, unik, dan sama; teks tidak diubah; isian hanya 0/1 tanpa sel kosong. Baris diselaraskan menurut `id_anotasi`. Kappa dihitung dengan rumus (2.5) dan dengan `sklearn.metrics.cohen_kappa_score` (harus sama sampai 1e-12), secara keseluruhan dan per sumber (kandidat latih, kandidat validasi, uji). Rumus diuji dengan contoh Bab II 2.3.5 (κ = 0,70). Untuk data uji hanya p_o, p_e, κ, dan jumlah selisih yang ditampilkan; sebaran labelnya tidak ditampilkan.

**Package:** `importlib` (bawaan Python); `numpy`; `pandas`; `scikit-learn` (`cohen_kappa_score`); `openpyxl`; `lib.wondr_anotasi`; `lib.wondr_kesepakatan`.

Sel 1/2 menulis `lib/wondr_kesepakatan.py` (validasi isian, kappa, tafsiran, tabel kappa, ekspor selisih; dipakai lagi di 9.3 dan 14.3). Sel 2/2 menyimpan `bab4/tabel/tahap09_kappa.csv` dan `anotasi/selisih.xlsx` (`id_anotasi, teks, label_A, label_B, label_final` kosong). Bila lembar tidak valid, sel berhenti dan menampilkan id yang bermasalah. Setelah ini program berhenti sampai selisih diselesaikan lewat diskusi.

In [ ]:
%%writefile lib/wondr_kesepakatan.py
# lib/wondr_kesepakatan.py -- validasi isian lembar anotasi, selisih, dan kesepakatan dua penilai (Tahap 9.2, 9.3, 14.3).
# Kappa Cohen pers. (2.5); tafsiran Tabel 2.10 (Landis dan Koch, 1977). Isinya teks petunjuk dan fungsi saja.
import numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score
from lib.wondr_anotasi import tulis_lembar

PETUNJUK_SELISIH = [
    "PETUNJUK PENYELESAIAN SELISIH ANOTASI",
    "",
    "Baris pada sheet 'Anotasi' adalah ulasan yang diberi label berbeda oleh anotator A dan B.",
    "Diskusikan setiap baris bersama kedua anotator, lalu isi kolom 'label_final' dengan 1 atau 0:",
    "  1 = Positif     : ulasan menyatakan kepuasan atau pujian tanpa keluhan.",
    "  0 = Non-Positif : keluhan, kritik, ulasan netral, atau ulasan bernada campuran (memuat pujian sekaligus keluhan).",
    "",
    "Jangan mengubah kolom lain, urutan baris, maupun nama sheet; jangan ada label_final yang kosong.",
]
BATAS_KAPPA = [(0.20, "Sedikit (slight)"), (0.40, "Cukup (fair)"), (0.60, "Sedang (moderate)"),
               (0.80, "Substansial (substantial)"), (1.00, "Hampir sempurna (almost perfect)")]


def periksa_isian(x, id_kunci, teks_kunci, kolom):
    """Validasi lembar terisi terhadap kunci: id lengkap, unik, dan sama; teks tidak diubah; isian hanya 0/1 tanpa sel
    kosong. Mengembalikan (ringkasan, isian numerik dalam urutan id_kunci, daftar id bermasalah)."""
    acuan = pd.DataFrame({"id_anotasi": list(id_kunci), "teks_kunci": list(teks_kunci)})
    m = acuan.merge(x, on="id_anotasi", how="left", indicator=True)
    nilai = pd.to_numeric(m[kolom], errors="coerce")
    sah = nilai.isin([0, 1])
    r = {"id_lengkap": bool(x["id_anotasi"].is_unique and len(x) == len(acuan) and (m["_merge"] == "both").all()),
         "teks_sama": bool((m["teks"] == m["teks_kunci"]).all()),
         "kosong": int(m[kolom].isna().sum()),
         "tidak_sah": int((m[kolom].notna() & ~sah).sum())}
    r["valid"] = r["id_lengkap"] and r["teks_sama"] and r["kosong"] == 0 and r["tidak_sah"] == 0
    masalah = m.loc[~sah | (m["teks"] != m["teks_kunci"]) | (m["_merge"] != "both"), "id_anotasi"].tolist()
    return r, nilai.astype("Int64"), masalah


def kappa(a, b):
    """Pers. (2.5): p_o = proporsi label sama; p_e = sum_k p_A,k * p_B,k; kappa = (p_o - p_e) / (1 - p_e)."""
    a, b = np.asarray(a).astype(int), np.asarray(b).astype(int)
    if a.shape != b.shape or a.size == 0 or not np.isin(np.r_[a, b], [0, 1]).all():
        raise ValueError("a dan b harus sepanjang, tidak kosong, dan hanya berisi 0/1.")
    p_o = float(np.mean(a == b))
    p_e = float(sum(np.mean(a == k) * np.mean(b == k) for k in (0, 1)))
    return {"n": int(a.size), "p_o": p_o, "p_e": p_e, "kappa": (p_o - p_e) / (1 - p_e) if p_e < 1 else float("nan")}


def tafsir_kappa(k):
    """Tabel 2.10: kappa < 0 buruk; selang lain dibaca dengan batas atas inklusif (0,20; 0,40; 0,60; 0,80; 1,00)."""
    if np.isnan(k):
        return "tidak terdefinisi"
    if k < 0:
        return "Buruk (poor)"
    return next((nama for batas, nama in BATAS_KAPPA if k <= batas), BATAS_KAPPA[-1][1])


def tabel_kappa(a, b, kelompok):
    """Kappa per kelompok (dict nama -> mask boolean): rumus (2.5) dan sklearn, tafsiran Tabel 2.10, jumlah selisih."""
    a, b, baris = np.asarray(a).astype(int), np.asarray(b).astype(int), []
    for nama, m in kelompok.items():
        m = np.asarray(m, dtype=bool)
        r = kappa(a[m], b[m])
        baris.append({"sumber": nama, **r, "kappa_sklearn": float(cohen_kappa_score(a[m], b[m])),
                      "selisih": int((a[m] != b[m]).sum()), "tafsiran": tafsir_kappa(r["kappa"])})
    return pd.DataFrame(baris)


def ekspor_selisih(path, d):
    """Tulis selisih.xlsx (id_anotasi, teks, label_A, label_B, label_final kosong) untuk baris label_A != label_B, urut
    id_anotasi. Berkas yang label_final-nya sudah diisi tidak ditimpa (hanya dicatat). Mengembalikan baris selisih."""
    s = d[d["label_A"] != d["label_B"]].sort_values("id_anotasi")
    data = pd.DataFrame({"id_anotasi": s["id_anotasi"], "teks": s["text_clean"], "label_A": s["label_A"],
                         "label_B": s["label_B"], "label_final": None})
    try:
        tulis_lembar(path, data, kolom_isian="label_final", petunjuk=PETUNJUK_SELISIH)
    except FileExistsError as galat:
        print(f"Catatan: {galat}")
    return s

In [ ]:
# ══ Tahap 9.2 — Impor anotasi dan kesepakatan (sel 2/2) ══
# Package: importlib (bawaan Python); numpy; pandas; lib.wondr_anotasi; lib.wondr_kesepakatan (scikit-learn cohen_kappa_score)
import importlib
import numpy as np, pandas as pd
import lib.wondr_anotasi as wa, lib.wondr_kesepakatan as wk
importlib.reload(wk)                                        # pakai isi lib/wondr_kesepakatan.py yang terbaru
mulai()
ya, A, T, e = (lambda x: "ya" if x else "tidak"), PATH["anotasi"], TARGET["9.2"], (lambda v: f"{v:.1e}".replace(".", ","))

# 1) Kunci + text_clean (dari data uji hanya review_id dan text_clean), lalu validasi kedua lembar yang sudah diisi
k = pd.read_csv(A / "kunci_anotasi.csv").merge(pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet",
        columns=["review_id", "text_clean"]) for b in ("latih", "validasi", "uji")]), on="review_id", how="left")
V = {a: wk.periksa_isian(wa.baca_lembar(A / f"lembar_anotator_{a}.xlsx"), k.id_anotasi, k.text_clean, "label") for a in "AB"}
judul("9.2", "Impor anotasi dan kesepakatan", masukan=["anotasi/lembar_anotator_A.xlsx · B.xlsx", "anotasi/kunci_anotasi.csv"],
      keluaran=["bab4/tabel/tahap09_kappa.csv", "anotasi/selisih.xlsx (label_final kosong)"])
cek_lembar = lambda: [cek(f"lembar {a}: id lengkap · teks tidak diubah · sel kosong · isian selain 0/1", [ya(r["id_lengkap"]),
                      ya(r["teks_sama"]), r["kosong"], r["tidak_sah"]], ["ya", "ya", 0, 0]) for a, (r, _, _) in V.items()]
if not all(r["valid"] for r, _, _ in V.values()):
    cek_lembar(); [print(f"  lembar {a}, id bermasalah ({len(m)}): {', '.join(m[:20])}") for a, (_, _, m) in V.items() if m]
    raise ValueError("Lembar anotasi belum valid: perbaiki id di atas pada lembarnya, unggah ulang, lalu jalankan sel ini lagi.")

# 2) Kappa pers. (2.5) dan sklearn per sumber + keseluruhan; jumlah selisih
d = k.assign(label_A=V["A"][1].astype(int).to_numpy(), label_B=V["B"][1].astype(int).to_numpy())
tab = wk.tabel_kappa(d.label_A, d.label_B, {"kandidat latih": d.bagian == "latih", "kandidat validasi": d.bagian == "validasi",
                                            "uji": d.bagian == "uji", "keseluruhan": d.bagian.notna()})
simpan_tabel(tab, "tahap09_kappa")

# 3) Ekspor selisih untuk diskusi (label_final kosong); berkas yang sudah diisi tidak ditimpa
s = wk.ekspor_selisih(A / "selisih.xlsx", d)
sel, u = wa.baca_lembar(A / "selisih.xlsx"), wk.kappa([1] * 45 + [0] * 55, [1] * 40 + [0] * 5 + [1] * 10 + [0] * 45)  # contoh 2.3.5
print(f"{'Sumber':<19}{'n':>6}{'p_o':>8}{'p_e':>8}{'κ':>8}  {'Tafsiran':<33}{'Selisih':>8}")
for t in tab.itertuples():
    print(f"{t.sumber:<19}{fmt_id(t.n):>6}{fmt_id(t.p_o):>8}{fmt_id(t.p_e):>8}{fmt_id(t.kappa):>8}  {t.tafsiran:<33}{fmt_id(t.selisih):>8}")
cek_lembar()
cek("contoh Bab II 2.3.5: p_o · p_e · κ", [u["p_o"], u["p_e"], u["kappa"]], [0.85, 0.50, T["kappa_contoh"]], toleransi=T["toleransi"], d=4)
dk = float((tab.kappa - tab.kappa_sklearn).abs().max())
cek(f"κ rumus (2.5) = sklearn (|Δ| maks {e(dk)} ≤ 1e-12)", ya(dk <= T["toleransi"]), "ya")
cek("selisih.xlsx: baris = selisih keseluruhan · id sama", [len(sel), ya(sel.id_anotasi.tolist() == s.id_anotasi.tolist())],
    [int(tab.selisih.iloc[-1]), "ya"])
selesai()

### Tahap 9.2 — Pemeriksaan kemandirian anotasi (tambahan)

Hanya membaca berkas. Kesepakatan kedua anotator sangat tinggi (κ keseluruhan 0,9929; 3 selisih dari 1.051). Sel ini menampilkan sidik SHA-256 dan metadata Excel kedua lembar (waktu dibuat dan diubah; nama pengguna tidak dicetak, hanya sama/beda), serta proporsi kesepakatan per kelompok panjang teks dan per paruh urutan lembar. Tidak membaca *rating* maupun label awal data uji. Hasilnya bersifat petunjuk; kemandirian anotasi hanya bisa dipastikan dari prosedur kerja anotator.

**Package:** `hashlib` (bawaan Python); `pandas`; `openpyxl`; `lib.wondr_anotasi`.

In [ ]:
# ══ Tahap 9.2 — Pemeriksaan kemandirian anotasi (tambahan; hanya membaca berkas) ══
# Package: hashlib (bawaan Python); pandas; openpyxl; lib.wondr_anotasi
import hashlib
import pandas as pd, openpyxl
import lib.wondr_anotasi as wa
mulai()
A = PATH["anotasi"]
f = {a: A / f"lembar_anotator_{a}.xlsx" for a in "AB"}

# 1) Identitas berkas: sidik SHA-256 dan metadata Excel (nama pengguna tidak dicetak, hanya sama/beda)
P = {a: openpyxl.load_workbook(p, read_only=True).properties for a, p in f.items()}
# 2) Kesepakatan per kelompok panjang teks; dari data uji hanya review_id dan kelompok_panjang (tanpa rating dan label)
k = pd.read_csv(A / "kunci_anotasi.csv").merge(pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet",
        columns=["review_id", "kelompok_panjang"]) for b in ("latih", "validasi", "uji")]), on="review_id")
L = {a: wa.baca_lembar(p).set_index("id_anotasi").label for a, p in f.items()}
k["sama"] = k.id_anotasi.map(L["A"]).to_numpy() == k.id_anotasi.map(L["B"]).to_numpy()
k["paruh"] = (k.id_anotasi.str[1:].astype(int) > len(k) // 2).map({False: "id paruh 1", True: "id paruh 2"})

judul("9.2", "Pemeriksaan kemandirian anotasi", masukan=["anotasi/lembar_anotator_A.xlsx · B.xlsx", "anotasi/kunci_anotasi.csv"])
for a in "AB":
    print(f"lembar {a}: SHA-256 {hashlib.sha256(f[a].read_bytes()).hexdigest()[:16]} · dibuat {P[a].created} · "
          f"terakhir diubah {P[a].modified}")
sama = lambda x: "sama" if getattr(P["A"], x) == getattr(P["B"], x) else "beda"
print(f"metadata pembuat (creator): {sama('creator')} · terakhir diubah oleh (lastModifiedBy): {sama('lastModifiedBy')}")
garis()
print(f"{'Kelompok':<16}{'n':>7}{'sepakat':>9}{'p_o':>9}")
for kol, urut in (("kelompok_panjang", ["1-2", "3-10", ">10"]), ("paruh", ["id paruh 1", "id paruh 2"])):
    for g in urut:
        x = k[k[kol] == g]
        print(f"{(g + ' kata') if kol == 'kelompok_panjang' else g:<16}{fmt_id(len(x)):>7}{fmt_id(int(x.sama.sum())):>9}"
              f"{fmt_id(x.sama.mean()):>9}")
selesai()

## Tahap 9.3 — Terapkan label akhir

**Dasar naskah:** Bab III tahap 9; Bab II subbab 2.3.5. Label manual = label anotator A bila A dan B sepakat, atau `label_final` hasil diskusi bila berbeda. Data latih dan validasi: `label_akhir` = label manual untuk kandidat, label awal untuk dokumen lain. Data uji: `label_akhir` = label manual untuk seluruh dokumen. Kolom `label` diganti nama menjadi `label_awal`; ditambah `kandidat` dan `berubah`. Sebaran label uji **belum** ditampilkan.

**Package:** `importlib` (bawaan Python); `numpy`; `pandas`; `lib.wondr_anotasi`; `lib.wondr_kesepakatan`.

Sel 1/2 memvalidasi ulang lembar A, B, dan `selisih.xlsx` (semua `label_final` harus 0/1; kolom lain tidak berubah), lalu menyimpan `anotasi/label_manual.csv` dan `hasil/tahap09_anotasi.json` (kappa dan tafsirannya). Sel 2/2 menyimpan `hasil/tahap09_latih.parquet`, `hasil/tahap09_validasi.parquet`, dan `hasil/tahap09_uji.parquet`.

In [ ]:
# ══ Tahap 9.3 — Terapkan label akhir (sel 1/2: label manual dari dua anotator dan diskusi) ══
# Package: importlib (bawaan Python); pandas; lib.wondr_anotasi; lib.wondr_kesepakatan
import importlib
import pandas as pd
import lib.wondr_anotasi as wa, lib.wondr_kesepakatan as wk
importlib.reload(wk)                                        # pakai isi lib/wondr_kesepakatan.py yang terbaru
mulai()
ya, A = (lambda x: "ya" if x else "tidak"), PATH["anotasi"]

# 1) Kunci + text_clean; validasi ulang lembar A dan B, lalu selisih.xlsx (label_final) dengan aturan yang sama
k = pd.read_csv(A / "kunci_anotasi.csv").merge(pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet",
        columns=["review_id", "text_clean"]) for b in ("latih", "validasi", "uji")]), on="review_id", how="left")
V = {a: wk.periksa_isian(wa.baca_lembar(A / f"lembar_anotator_{a}.xlsx"), k.id_anotasi, k.text_clean, "label") for a in "AB"}
if not all(r["valid"] for r, _, _ in V.values()):
    raise ValueError("Lembar A/B tidak valid lagi; jalankan sel Tahap 9.2 untuk melihat id yang bermasalah.")
k = k.assign(label_A=V["A"][1].astype(int).to_numpy(), label_B=V["B"][1].astype(int).to_numpy())
s, sel = k[k.label_A != k.label_B], wa.baca_lembar(A / "selisih.xlsx")
rs, final, masalah = wk.periksa_isian(sel, s.id_anotasi, s.text_clean, "label_final")
ab = sel.set_index("id_anotasi").reindex(s.id_anotasi)[["label_A", "label_B"]].to_numpy()
ab_tetap = bool((ab == s[["label_A", "label_B"]].to_numpy()).all())
if not (rs["valid"] and ab_tetap):
    print(f"selisih.xlsx bermasalah ({len(masalah)} id): {', '.join(masalah[:20]) or '-'} · label_A/label_B tetap: {ya(ab_tetap)}")
    raise ValueError("Isi label_final 1/0 untuk semua baris selisih.xlsx tanpa mengubah kolom lain, lalu jalankan lagi.")

# 2) Label manual = label A bila A = B, label_final bila berbeda; simpan bersama kappa (dari Tahap 9.2) ke json
k["label_final"] = pd.Series(final.to_numpy(), index=s.index).reindex(k.index).astype("Int64")
k["label_manual"] = k.label_final.fillna(k.label_A).astype(int)
k[["id_anotasi", "review_id", "bagian", "kandidat", "label_A", "label_B", "label_final", "label_manual"]].to_csv(
    A / "label_manual.csv", index=False)
tab = pd.read_csv(PATH["tabel"] / "tahap09_kappa.csv")
simpan_json({"n_anotasi": len(k), "n_selisih": len(s), "kappa": tab.set_index("sumber")[["n", "p_o", "p_e", "kappa", "tafsiran",
             "selisih"]].to_dict("index")}, PATH["hasil"] / "tahap09_anotasi.json")
judul("9.3", "Label manual", masukan=["anotasi/lembar_anotator_A.xlsx · B.xlsx", "anotasi/selisih.xlsx"],
      keluaran=["anotasi/label_manual.csv", "hasil/tahap09_anotasi.json"])
print(f"dokumen dianotasi {fmt_id(len(k))} · sepakat {fmt_id(len(k) - len(s))} · selisih {fmt_id(len(s))} (diselesaikan lewat "
      f"diskusi: {fmt_id(int(k.label_final.notna().sum()))}) · κ keseluruhan {fmt_id(tab.kappa.iloc[-1])} ({tab.tafsiran.iloc[-1]})")
cek("lembar A · B valid · selisih.xlsx valid · label_A/label_B di selisih tidak berubah",
    [ya(V["A"][0]["valid"]), ya(V["B"][0]["valid"]), ya(rs["valid"]), ya(ab_tetap)], ["ya"] * 4)
cek("label manual lengkap · hanya 0/1 · selisih di kappa CSV = selisih lembar", [int(k.label_manual.notna().sum()),
    ya(k.label_manual.isin([0, 1]).all()), int(tab.selisih.iloc[-1])], [len(k), "ya", len(s)])
selesai()

In [ ]:
# ══ Tahap 9.3 — Terapkan label akhir (sel 2/2: label_awal, label_akhir, kandidat, berubah) ══
# Package: numpy; pandas
import numpy as np, pandas as pd
mulai()
ya, T6 = (lambda x: "ya" if x else "tidak"), TARGET["6.1"]

# 1) Data Tahap 6.1, kandidat Tahap 8.3, dan label manual (sel 1/2)
m = pd.read_csv(PATH["anotasi"] / "label_manual.csv")
kd = pd.read_parquet(PATH["hasil"] / "tahap08_kandidat.parquet", columns=["review_id", "kandidat"])
# 2) latih/validasi: label_akhir = label manual untuk kandidat, label awal untuk lainnya; uji: label manual seluruhnya
D = {}
for b in ("latih", "validasi", "uji"):
    d = pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet").rename(columns={"label": "label_awal"})
    d = d.merge(kd, on="review_id", how="left") if b != "uji" else d.assign(kandidat=False)
    d = d.merge(m.loc[m.bagian == b, ["review_id", "id_anotasi", "label_manual"]], on="review_id", how="left")
    d["label_manual"] = d.label_manual.astype("Int64")
    pakai = d.kandidat if b != "uji" else pd.Series(True, index=d.index)
    d["label_akhir"] = d.label_manual.where(pakai, d.label_awal).astype("int64")   # galat bila ada label manual yang hilang
    d["berubah"] = (d.label_akhir != d.label_awal).astype(bool)
    d.to_parquet(PATH["hasil"] / f"tahap09_{b}.parquet", index=False)
    D[b] = d

judul("9.3", "Terapkan label akhir", masukan=["hasil/tahap06_{latih,validasi,uji}.parquet", "hasil/tahap08_kandidat.parquet",
      "anotasi/label_manual.csv"], keluaran="hasil/tahap09_{latih,validasi,uji}.parquet")
print(f"{'Bagian':<10}{'n':>7}{'kandidat':>10}{'berubah P→NP':>14}{'berubah NP→P':>14}{'Non-Positif sebelum → sesudah':>32}")
for b in ("latih", "validasi"):
    d = D[b]
    print(f"{b:<10}{fmt_id(len(d)):>7}{fmt_id(int(d.kandidat.sum())):>10}{fmt_id(int((d.berubah & (d.label_awal == 1)).sum())):>14}"
          f"{fmt_id(int((d.berubah & (d.label_awal == 0)).sum())):>14}"
          f"{fmt_id((d.label_awal == 0).mean() * 100, 1) + '% → ' + fmt_id((d.label_akhir == 0).mean() * 100, 1) + '%':>32}")
print(f"{'uji':<10}{fmt_id(len(D['uji'])):>7}  dokumen berlabel manual (sebaran label uji belum ditampilkan)")
cek("n latih · validasi · uji", [len(D[b]) for b in D], [T6["latih"][0], T6["validasi"][0], T6["uji"][0]])
cek("label manual tersedia: kandidat latih · kandidat validasi · seluruh uji", [int(D["latih"].label_manual.notna().sum()),
    int(D["validasi"].label_manual.notna().sum()), int(D["uji"].label_manual.notna().sum())],
    [int(D["latih"].kandidat.sum()), int(D["validasi"].kandidat.sum()), T6["uji"][0]])
cek("bukan kandidat tidak berubah (latih · validasi) · label_akhir hanya 0/1 · kolom label → label_awal",
    [int((D["latih"].berubah & ~D["latih"].kandidat).sum()), int((D["validasi"].berubah & ~D["validasi"].kandidat).sum()),
     ya(all(D[b].label_akhir.isin([0, 1]).all() for b in D)), ya(all("label" not in D[b] and "label_awal" in D[b] for b in D))],
    [0, 0, "ya", "ya"])
selesai()

### Tahap 9.3 — Audit (tambahan, diminta pengguna)

Hanya membaca berkas. **Audit A:** tiga selisih beserta `label_final` (berkas tidak memuat catatan diskusi), sebaran `label_akhir` per bagian, dan perubahan kandidat per arah CL; bukan kandidat dicek tetap memakai label awal. Sebaran label uji ditampilkan atas permintaan pengguna (spesifikasi 9.3 semula menundanya); ubah `SEBARAN_UJI = False` untuk menyembunyikannya. **Audit B:** seluruh label uji = label manual; daftar berkas di `hasil/`, `log/`, dan `anotasi/` yang memuat `review_id` uji; `fitness()` hanya membaca `tahap09_latih` dan `tahap09_validasi`; jumlah baris, urutan `review_id`, dan kolom Tahap 6 tidak berubah.

**Package:** `inspect` (bawaan Python); `pandas`; `lib.wondr_anotasi`; `lib.wondr_latih`.

In [ ]:
# ══ Tahap 9.3 — Audit A: selisih, sebaran label akhir, dan perubahan kandidat (tambahan; hanya membaca berkas) ══
# Package: pandas; lib.wondr_anotasi
import pandas as pd
import lib.wondr_anotasi as wa
mulai()
H, A, ya = PATH["hasil"], PATH["anotasi"], (lambda x: "ya" if x else "tidak")
SEBARAN_UJI = True   # spesifikasi 9.3 menunda sebaran label uji; ditampilkan atas permintaan pengguna (audit 22-10-2026)

# 1) Label manual, selisih.xlsx, hasil Tahap 9.3, dan kelas taksiran Tahap 8.3
m, n_sel = pd.read_csv(A / "label_manual.csv"), baca_json(H / "tahap09_anotasi.json")["n_selisih"]
sel = wa.baca_lembar(A / "selisih.xlsx").merge(m[["id_anotasi", "bagian", "label_manual"]], on="id_anotasi", how="left")
t9 = {b: pd.read_parquet(H / f"tahap09_{b}.parquet") for b in ("latih", "validasi", "uji")}
kd = pd.read_parquet(H / "tahap08_kandidat.parquet", columns=["review_id", "kelas_taksiran"])
judul("9.3", "Audit A: selisih, sebaran, kandidat", masukan=["anotasi/selisih.xlsx + label_manual.csv",
      "hasil/tahap09_{latih,validasi,uji}.parquet + tahap08_kandidat.parquet"])
print("(1) Selisih A ≠ B. label_final dibaca apa adanya dari selisih.xlsx; berkas tidak memuat catatan atau alasan diskusi.")
print(sel[["id_anotasi", "bagian", "label_A", "label_B", "label_final", "label_manual"]].to_string(index=False))
garis(); print(f"(2) Sebaran label_akhir\n{'Bagian':<10}{'n':>7}{'label 0 (Non-Positif)':>24}{'label 1 (Positif)':>20}")
for b, d in t9.items():
    if b == "uji" and not SEBARAN_UJI:
        print(f"{b:<10}{fmt_id(len(d)):>7}  (tidak ditampilkan)"); continue
    n0, n1 = int((d.label_akhir == 0).sum()), int((d.label_akhir == 1).sum())
    print(f"{b:<10}{fmt_id(len(d)):>7}{fmt_id(n0) + ' (' + fmt_id(100 * n0 / len(d), 1) + '%)':>24}"
          f"{fmt_id(n1) + ' (' + fmt_id(100 * n1 / len(d), 1) + '%)':>20}")
garis(); print(f"(5) Kandidat per arah CL\n{'Bagian':<10}{'kelompok':<24}{'n':>6}{'berubah':>9}{'tetap':>7}")
X = {b: t9[b].merge(kd, on="review_id", how="left") for b in ("latih", "validasi")}
for b, x in X.items():
    for nama, g in (("P → NP (label 1, C10)", x[x.kandidat & (x.label_awal == 1)]),
                    ("NP → P (label 0, C01)", x[x.kandidat & (x.label_awal == 0)]), ("bukan kandidat", x[~x.kandidat])):
        print(f"{b:<10}{nama:<24}{fmt_id(len(g)):>6}{fmt_id(int(g.berubah.sum())):>9}{fmt_id(int((~g.berubah).sum())):>7}")
for b, x in X.items():
    nk, kk = x[~x.kandidat], x[x.kandidat]
    cek(f"{b}: kandidat taksiran ≠ label awal · bukan kandidat: label_akhir = label_awal, label_manual kosong",
        [ya((kk.kelas_taksiran != kk.label_awal).all()), ya((nk.label_akhir == nk.label_awal).all()),
         int(nk.label_manual.notna().sum())], ["ya", "ya", 0])
cek("selisih: baris · A ≠ B · label_final terisi · label_manual = label_final", [len(sel), ya((sel.label_A != sel.label_B).all()),
    int(sel.label_final.notna().sum()), ya((sel.label_manual == sel.label_final).all())], [n_sel, "ya", n_sel, "ya"])
selesai()

In [ ]:
# ══ Tahap 9.3 — Audit B: label uji, pemakaian data uji, dan konsistensi dengan Tahap 6 (tambahan; hanya membaca berkas) ══
# Package: inspect (bawaan Python); pandas; lib.wondr_latih
import inspect
import pandas as pd
import lib.wondr_latih as wl
mulai()
H, ya, B = PATH["hasil"], (lambda x: "ya" if x else "tidak"), ("latih", "validasi", "uji")

# 1) Tahap 6, Tahap 9.3, label manual, dan kandidat Tahap 8.3
t6, t9 = ({b: pd.read_parquet(H / f"tahap{t}_{b}.parquet") for b in B} for t in ("06", "09"))
m, kd = pd.read_csv(PATH["anotasi"] / "label_manual.csv"), pd.read_parquet(H / "tahap08_kandidat.parquet")
u, idu = t9["uji"].merge(m, on=["review_id", "id_anotasi"], how="left", suffixes=("", "_csv")), set(t9["uji"].review_id)
# 2) Berkas yang memuat review_id uji di hasil/, log/, anotasi/ (diizinkan: sebelum pembagian Tahap 2-5, berkas uji, anotasi)
judul("9.3", "Audit B: label uji dan konsistensi", masukan=["hasil/tahap06_*, tahap09_*, tahap08_kandidat", "log/*.parquet",
      "anotasi/*.csv", "lib/wondr_latih.py (fitness)"])
izin = lambda p: p.name.startswith(("tahap02", "tahap03", "tahap04", "tahap05")) or "uji" in p.name or p.parent.name == "anotasi"
asing = []
for p in sorted([*H.glob("*.parquet"), *PATH["log"].glob("*.parquet"), *PATH["anotasi"].glob("*.csv")]):
    x = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)
    n = len(idu & set(x.review_id)) if "review_id" in x else 0
    print(f"  {rel(p):<38} id uji {fmt_id(n):>5}  {'(diizinkan)' if n and izin(p) else ''}")
    asing += [rel(p)] if n and not izin(p) else []
src = inspect.getsource(wl.fitness)
cek("uji: label_akhir = label_manual · label manual & id_anotasi lengkap · = label_manual.csv · = A bila A = B, selain itu label_final",
    [ya((u.label_akhir == u.label_manual).all()), int(u.label_manual.notna().sum()), int(u.id_anotasi.nunique()),
     ya((u.label_manual == u.label_manual_csv).all()), ya((u.label_manual == u.label_final.fillna(u.label_A)).all())],
    ["ya", len(idu), len(idu), "ya", "ya"])
cek("id uji di berkas pasca-pembagian lain (log CL, kandidat 8.3, latih/validasi) · fitness() hanya membaca tahap09 latih + validasi",
    [len(asing), ya('baca("latih"), baca("validasi")' in src and "uji" not in src)], [0, "ya"])
for b in B:
    a, d = t6[b], t9[b]
    sama = all(d[c].equals(a[c]) for c in a.columns if c != "label")
    baru = sorted(set(d.columns) - set(a.columns))
    kand = d.kandidat.equals(d[["review_id"]].merge(kd, on="review_id", how="left").kandidat.fillna(False)) if b != "uji" else True
    cek(f"{b}: n · urutan review_id · kolom Tahap 6 tidak berubah · label_awal = label Tahap 6 · kolom baru · kandidat = Tahap 8.3",
        [len(d), ya(d.review_id.equals(a.review_id)), ya(sama), ya(d.label_awal.equals(a.label)), baru, ya(kand)],
        [len(a), "ya", "ya", "ya", ["berubah", "id_anotasi", "kandidat", "label_akhir", "label_awal", "label_manual"], "ya"])
selesai()

## Tahap 9.4 — Eksplorasi data latih dan validasi (*program saja*)

**Dasar:** spesifikasi Tahap 10 di Bagian 6 (menjadi 9.4, keputusan no. 6); tidak dibahas di naskah. **Hanya data latih dan validasi**; data uji tidak dibaca. Menampilkan proporsi kelas label awal vs label akhir beserta jumlah label yang berubah, kelompok panjang × label akhir, median jumlah kata per kelas, dan 10 kata tersering per kelas pada data latih (kata = rangkaian huruf; `KATA_FUNGSI` diabaikan, hanya untuk tampilan). Acuan minimum: `DummyClassifier(strategy="most_frequent")` dilatih pada `label_akhir` latih, lalu dihitung akurasi dan *Macro F1* validasinya.

**Package:** `re`, `collections` (bawaan Python); `numpy`; `pandas`; `matplotlib`; `scikit-learn` (`DummyClassifier`, `accuracy_score`, `f1_score`); `lib.wondr_teks`.

Sel 1/2 menyimpan `bab4/tabel/tahap09_eksplorasi_{proporsi,panjang,kata}.csv`. Sel 2/2 menyimpan `bab4/gambar/tahap09_proporsi_label.png` dan `hasil/tahap09_acuan.json`.

In [ ]:
# ══ Tahap 9.4 — Eksplorasi data latih dan validasi (sel 1/2: proporsi, panjang, kata tersering; program saja) ══
# Package: re, collections (bawaan Python); pandas; lib.wondr_teks (KATA_FUNGSI, kelompok_panjang)
import re, collections
import pandas as pd
from lib.wondr_teks import KATA_FUNGSI, kelompok_panjang
mulai()
KP, NM, ya = list(CFG["kelompok_panjang"]), {0: "NP", 1: "P"}, (lambda x: "ya" if x else "tidak")
# 1) Data Tahap 9.3, hanya latih dan validasi (data uji tidak dibaca)
D = {b: pd.read_parquet(PATH["hasil"] / f"tahap09_{b}.parquet").assign(n_kata=lambda d: d.text_clean.str.split().str.len())
     for b in ("latih", "validasi")}
# 2) Proporsi label awal vs akhir + jumlah berubah; kelompok panjang x label akhir; 10 kata tersering per kelas (latih)
prop = pd.DataFrame([{"bagian": b, "n": len(d), **{f"{j}_{NM[k]}": int((d[f"label_{j}"] == k).sum()) for j in ("awal", "akhir")
                      for k in (0, 1)}, "berubah_P_ke_NP": int((d.berubah & (d.label_awal == 1)).sum()),
                      "berubah_NP_ke_P": int((d.berubah & (d.label_awal == 0)).sum())} for b, d in D.items()])
pj = pd.concat([pd.crosstab(d.kelompok_panjang, d.label_akhir).reindex(index=KP, columns=[0, 1], fill_value=0)
                .rename(columns=NM).assign(bagian=b, median_kata_NP=d.n_kata[d.label_akhir == 0].median(),
                median_kata_P=d.n_kata[d.label_akhir == 1].median()) for b, d in D.items()]).reset_index()
hit = {k: collections.Counter(w for s in D["latih"].text_clean[D["latih"].label_akhir == k] for w in re.findall(r"[^\W\d_]+", s)
                  if w not in KATA_FUNGSI).most_common(10) for k in (0, 1)}
kata = pd.DataFrame({"peringkat": range(1, 11), **{f"{c}_{NM[k]}": [x[i] for x in hit[k]] for k in (0, 1)
                     for i, c in ((0, "kata"), (1, "frek"))}})
[simpan_tabel(t, f"tahap09_eksplorasi_{nama}") for t, nama in ((prop, "proporsi"), (pj, "panjang"), (kata, "kata"))]

judul("9.4", "Eksplorasi data (sel 1/2)", masukan="hasil/tahap09_{latih,validasi}.parquet",
      keluaran="bab4/tabel/tahap09_eksplorasi_{proporsi,panjang,kata}.csv")
for r in prop.itertuples():
    print(f"{r.bagian:<9} n {fmt_id(r.n)} · Non-Positif {fmt_id(r.awal_NP)} → {fmt_id(r.akhir_NP)} ({fmt_id(100 * r.awal_NP / r.n, 1)}% → "
          f"{fmt_id(100 * r.akhir_NP / r.n, 1)}%) · berubah P→NP {fmt_id(r.berubah_P_ke_NP)}, NP→P {fmt_id(r.berubah_NP_ke_P)}")
garis(); print(f"{'Bagian':<10}{'Kelompok':<10}{'Non-Positif':>12}{'Positif':>9}{'% Non-Positif':>15}")
for r in pj.itertuples():
    print(f"{r.bagian:<10}{r.kelompok_panjang:<10}{fmt_id(r.NP):>12}{fmt_id(r.P):>9}{fmt_id(100 * r.NP / (r.NP + r.P), 1) + '%':>15}")
print("Median kata per kelas:", " · ".join(f"{b} NP {fmt_id(g.median_kata_NP.iloc[0], 1)}, P {fmt_id(g.median_kata_P.iloc[0], 1)}"
                                            for b, g in pj.groupby("bagian", sort=False)))
garis(); print(f"{'#':<4}{'Non-Positif (latih)':<26}{'Positif (latih)':<26}")
for r in kata.itertuples():
    print(f"{r.peringkat:<4}{r.kata_NP + ' (' + fmt_id(r.frek_NP) + ')':<26}{r.kata_P + ' (' + fmt_id(r.frek_P) + ')':<26}")
cek("akhir NP = awal NP + berubah P→NP − berubah NP→P (latih · validasi)", (prop.akhir_NP).tolist(),
    (prop.awal_NP + prop.berubah_P_ke_NP - prop.berubah_NP_ke_P).tolist())
cek("kelompok_panjang Tahap 6 = kelompok dari jumlah kata · total tabel panjang = n", [ya(all((d.n_kata.map(kelompok_panjang)
    == d.kelompok_panjang).all() for d in D.values())), int((pj.NP + pj.P).sum())], ["ya", int(prop.n.sum())])
selesai()

In [ ]:
# ══ Tahap 9.4 — Eksplorasi data (sel 2/2: gambar proporsi label dan acuan minimum validasi; program saja) ══
# Package: numpy; pandas; matplotlib (grafik); scikit-learn (DummyClassifier, accuracy_score, f1_score)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
mulai()
WARNA, ya = {1: "#2a78d6", 0: "#eb6834"}, (lambda x: "ya" if x else "tidak")   # warna kelas sama dengan Tahap 3
# 1) Label awal dan akhir latih + validasi (data uji tidak dibaca)
D = {b: pd.read_parquet(PATH["hasil"] / f"tahap09_{b}.parquet", columns=["label_awal", "label_akhir"]) for b in ("latih", "validasi")}
# 2) Gambar: proporsi kelas sebelum (label awal) dan sesudah anotasi (label akhir), batang bertumpuk 100%
baris = [(f"{b}: label {j}", (D[b][f"label_{j}"] == 0).mean() * 100) for b in D for j in ("awal", "akhir")]
fig, ax = plt.subplots(figsize=(6, 2.6))
for i, (nama, np_) in enumerate(baris):
    ax.barh(i, np_, color=WARNA[0], height=0.6, zorder=2)
    ax.barh(i, 100 - np_, left=np_, color=WARNA[1], height=0.6, zorder=2)
    for x, v in ((np_ / 2, np_), (np_ + (100 - np_) / 2, 100 - np_)):
        ax.text(x, i, f"{fmt_id(v, 1)}%", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
ax.set(yticks=range(len(baris)), yticklabels=[n for n, _ in baris], xlim=(0, 100), xlabel="Proporsi dokumen (%)")
ax.invert_yaxis(); ax.spines[["top", "right", "left"]].set_visible(False); ax.tick_params(colors="#52514e", length=0)
ax.legend(handles=[Patch(color=WARNA[k], label=CFG["kelas"][k]) for k in (0, 1)], frameon=False, ncol=2,
          loc="lower center", bbox_to_anchor=(0.5, 1.0))
gambar = simpan_gambar(fig, "tahap09_proporsi_label")
# 3) Acuan minimum (program saja): DummyClassifier(most_frequent) dilatih pada label_akhir latih, dinilai pada validasi
y_lat, y_val = D["latih"].label_akhir.to_numpy(), D["validasi"].label_akhir.to_numpy()
dm = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_lat), 1)), y_lat)
tebak = dm.predict(np.zeros((len(y_val), 1)))
acu = {"strategi": "most_frequent", "kelas_mayoritas_latih": int(tebak[0]), "akurasi_validasi": accuracy_score(y_val, tebak),
       "macro_f1_validasi": f1_score(y_val, tebak, average="macro", zero_division=0)}
simpan_json(acu, PATH["hasil"] / "tahap09_acuan.json")
q = (y_val == acu["kelas_mayoritas_latih"]).mean()                         # bentuk tertutup: F1 mayoritas = 2q/(1+q)
judul("9.4", "Eksplorasi data (sel 2/2)", masukan="hasil/tahap09_{latih,validasi}.parquet",
      keluaran=["bab4/gambar/tahap09_proporsi_label.png", "hasil/tahap09_acuan.json"])
print(f"Acuan minimum: selalu menebak {CFG['kelas'][acu['kelas_mayoritas_latih']]} (kelas mayoritas latih) · "
      f"akurasi validasi {fmt_id(acu['akurasi_validasi'])} · Macro F1 validasi {fmt_id(acu['macro_f1_validasi'])}")
cek("acuan = bentuk tertutup (akurasi = q · Macro F1 = q/(1+q))", [acu["akurasi_validasi"], acu["macro_f1_validasi"]],
    [q, q / (1 + q)], toleransi=1e-12, d=4)
cek("kelas mayoritas latih · gambar tersimpan", [acu["kelas_mayoritas_latih"], ya(gambar.exists())],
    [int(np.bincount(y_lat).argmax()), "ya"])
selesai()

### Tahap 9.4 — Audit akhir (tambahan, diminta pengguna)

Hanya membaca berkas. Menghitung ulang isi ketiga CSV eksplorasi dengan cara lain (hitungan per sel, median dari `text_clean`), memeriksa bahwa daftar kata memakai frekuensi kemunculan (bukan jumlah dokumen; keduanya dicetak), mencocokkan `hasil/tahap09_acuan.json` dengan sklearn dan bentuk tertutup q/(1+q), membuka PNG (ukuran dan dpi), dan membandingkan waktu ubah `tahap09_*.parquet` dengan waktu PNG dibuat. Data uji tidak dibaca.

**Package:** `re`, `collections`, `datetime` (bawaan Python); `numpy`; `pandas`; `Pillow`; `scikit-learn`; `lib.wondr_teks`.

In [ ]:
# ══ Tahap 9.4 — Audit akhir (tambahan; hanya membaca berkas, tidak menulis apa pun) ══
# Package: re, collections, datetime (bawaan Python); numpy; pandas; Pillow (PIL); scikit-learn (f1_score); lib.wondr_teks
import re, collections, datetime
import numpy as np, pandas as pd
from PIL import Image
from sklearn.metrics import f1_score
from lib.wondr_teks import KATA_FUNGSI, kelompok_panjang
mulai()
H, ya, KP = PATH["hasil"], (lambda x: "ya" if x else "tidak"), list(CFG["kelompok_panjang"])
waktu = lambda p: datetime.datetime.fromtimestamp(p.stat().st_mtime)                 # waktu terakhir berkas diubah
# 1) Keluaran 9.4 (3 CSV, json, PNG) dan masukannya (tahap09 latih + validasi); data uji tidak dibaca
D = {b: pd.read_parquet(H / f"tahap09_{b}.parquet") for b in ("latih", "validasi")}
C = {n: pd.read_csv(PATH["tabel"] / f"tahap09_eksplorasi_{n}.csv") for n in ("proporsi", "panjang", "kata")}
acu, png = baca_json(H / "tahap09_acuan.json"), PATH["gambar"] / "tahap09_proporsi_label.png"
# 2) Hitung ulang dengan cara lain (hitungan per sel; median dari text_clean; kata: kemunculan TF dan jumlah dokumen DF)
P, J = [], []
for b, d in D.items():
    nk, a, z = d.text_clean.str.split().str.len(), d.label_awal, d.label_akhir
    P.append([b, len(d), (a == 0).sum(), (a == 1).sum(), (z == 0).sum(), (z == 1).sum(), (d.berubah & (a == 1)).sum(), (d.berubah & (a == 0)).sum()])
    J += [[k, ((d.kelompok_panjang == k) & (z == 0)).sum(), ((d.kelompok_panjang == k) & (z == 1)).sum(), b, nk[z == 0].median(),
           nk[z == 1].median()] for k in KP]
P, J = pd.DataFrame(P, columns=C["proporsi"].columns), pd.DataFrame(J, columns=C["panjang"].columns)
grup_ok = all((d.text_clean.str.split().str.len().map(kelompok_panjang) == d.kelompok_panjang).all() for d in D.values())
tok = {k: [re.findall(r"[^\W\d_]+", s) for s in D["latih"].text_clean[D["latih"].label_akhir == k]] for k in (0, 1)}
TF = {k: collections.Counter(w for t in tok[k] for w in t if w not in KATA_FUNGSI) for k in (0, 1)}
DF = {k: collections.Counter(w for t in tok[k] for w in set(t) if w not in KATA_FUNGSI) for k in (0, 1)}
yv, maj = D["validasi"].label_akhir.to_numpy(), int(np.bincount(D["latih"].label_akhir).argmax())
q, img = float((yv == maj).mean()), Image.open(png)
judul("9.4", "Audit akhir", masukan=["bab4/tabel/tahap09_eksplorasi_*.csv · hasil/tahap09_acuan.json", "bab4/gambar/tahap09_proporsi_label.png ·"
      " hasil/tahap09_{latih,validasi}.parquet"])
for k, n in ((0, "NP"), (1, "P")):
    print(f"kata {n:<2} (kemunculan/dokumen): " + " · ".join(f"{w} {fmt_id(TF[k][w])}/{fmt_id(DF[k][w])}" for w in C["kata"][f"kata_{n}"]))
print(f"json: {acu}")
print(f"PNG {img.size[0]}×{img.size[1]} piksel · dpi {round(img.info.get('dpi', (0,))[0])} · disimpan {waktu(png):%d-%m %H:%M:%S} · "
      f"tahap09 latih/validasi diubah {waktu(H / 'tahap09_latih.parquet'):%d-%m %H:%M:%S} / {waktu(H / 'tahap09_validasi.parquet'):%d-%m %H:%M:%S}")
cek("CSV proporsi = hitung ulang · CSV panjang = hitung ulang · kelompok Tahap 6 = kelompok dari jumlah kata",
    [ya(C["proporsi"].equals(P)), ya(np.allclose(C["panjang"].select_dtypes("number"), J.select_dtypes("number")) and
     C["panjang"].select_dtypes("object").equals(J.select_dtypes("object"))), ya(grup_ok)], ["ya"] * 3)
cek("CSV kata: daftar = 10 teratas kemunculan · frek = kemunculan (bukan jumlah dokumen)", [ya(all(C["kata"][f"kata_{n}"].tolist() ==
    [w for w, _ in TF[k].most_common(10)] and C["kata"][f"frek_{n}"].tolist() == [TF[k][w] for w in C["kata"][f"kata_{n}"]]
    for k, n in ((0, "NP"), (1, "P"))))], ["ya"])
cek("json: mayoritas latih · akurasi validasi · Macro F1 sklearn · Macro F1 = q/(1+q)", [acu["kelas_mayoritas_latih"], acu["akurasi_validasi"],
    acu["macro_f1_validasi"], acu["macro_f1_validasi"]], [maj, q, float(f1_score(yv, np.full_like(yv, maj), average="macro",
    zero_division=0)), q / (1 + q)], toleransi=1e-12, d=4)
cek("PNG terbuka · dpi · tahap09 tidak diubah sesudah PNG dibuat", [ya(img.size[0] > 0), round(img.info.get("dpi", (0,))[0]),
    ya(max(waktu(H / f"tahap09_{b}.parquet") for b in D) < waktu(png))], ["ya", 300, "ya"])
selesai()

## Tahap 10 — *Fine-tuning* skema *default*

**Dasar naskah:** Bab III tahap 10; Tabel 3.1; pers. (2.14), (2.15), (2.18)–(2.22), (2.25). `fitness(2e-5, 0.1, 0.01)`: IndoBERT dilatih pada `tahap09_latih` (label = `label_akhir`) dengan *early stopping* berdasarkan *Macro F1* `tahap09_validasi` (*patience* 2, maksimum 4 *epoch*), untuk *seed* 42 dan 43; *fitness* = rata-rata *Macro F1* validasi terbaik kedua *seed*. Data uji tidak dibaca. GPU: perkiraan sekitar 2–3 menit pada A100.

**Package:** `importlib` (bawaan Python); `numpy`; `lib.wondr_latih` (`torch`, `transformers`, `scikit-learn`).

Menyimpan `hasil/tahap10_default.json` (konfigurasi, T dan T_w, skor dan kurva *Macro F1* per *epoch* setiap *seed*, durasi setiap pelatihan untuk tabel waktu komputasi, dan acuan minimum). Skor *seed* 42 dan 43 ini menjadi acuan cek determinisme di Tahap 13.

In [ ]:
# ══ Tahap 10 — Fine-tuning skema default ══
# Package: importlib (bawaan Python); numpy; lib.wondr_latih (fitness; torch, transformers, scikit-learn)
import importlib
import numpy as np
import lib.wondr_latih as wl
importlib.reload(wl)                                        # pakai isi lib/wondr_latih.py yang terbaru
mulai()
d, T, S, ya = CFG["default"], TARGET["10"], CFG["seed_fitness"], (lambda x: "ya" if x else "tidak")
wl.pakai_konfig(CFG)
acuan = baca_json(PATH["hasil"] / "tahap09_acuan.json")["macro_f1_validasi"]

# 1) Fitness pers. (2.25): rata-rata Macro F1 validasi terbaik seed 42 dan 43; data Tahap 9.3, label = label_akhir
r = wl.fitness(d["lr"], d["wr"], d["wd"])
H = r["hasil"]

# 2) Simpan skor, kurva per epoch, dan durasi tiap pelatihan (keputusan no. 21)
simpan_json({**d, "T": H[S[0]]["T"], "T_w": H[S[0]]["T_w"], "fitness": r["fitness"], **{f"skor{s}": H[s]["f1_terbaik"] for s in S},
             **{f"detik{s}": H[s]["detik"] for s in S}, "per_seed": {str(s): {k: H[s][k] for k in ("f1_per_epoch", "epoch_terbaik",
             "f1_terbaik", "detik", "perangkat")} for s in S}, "acuan_minimum": acuan}, PATH["hasil"] / "tahap10_default.json")

# 3) Ringkasan: T dan T_w, satu baris per seed, fitness, dan perkiraan waktu Tahap 11 (RS) + 12.1 (PSO)
n_rs, n_pso = 2 * CFG["rs_M"], 2 * CFG["pso"]["s"] * (1 + CFG["pso"]["G_max"])
menit = np.mean([H[s]["detik"] for s in S]) / 60
judul("10", "Fine-tuning skema default", masukan="hasil/tahap09_{latih,validasi}.parquet (label_akhir)",
      keluaran="hasil/tahap10_default.json")
print(f"lr {d['lr']} · wr {fmt_id(d['wr'], 2)} · wd {fmt_id(d['wd'], 2)} · T = {H[S[0]]['T']} · T_w = {H[S[0]]['T_w']}")
for s in S:
    h = H[s]
    print(f"seed {s}: Macro F1 per epoch [{' · '.join(fmt_id(f) for f in h['f1_per_epoch'])}] · epoch terbaik {h['epoch_terbaik']} · "
          f"{fmt_id(h['f1_terbaik'])} · {fmt_id(h['detik'] / 60, 1)} menit · {h['perangkat']}")
print(f"fitness = ({' + '.join(fmt_id(H[s]['f1_terbaik']) for s in S)}) / 2 = {fmt_id(r['fitness'])} · acuan minimum validasi "
      f"{fmt_id(acuan)} · selisih {fmt_id(r['fitness'] - acuan)}")
print(f"Perkiraan waktu: RS {n_rs} + PSO {n_pso} pelatihan × {fmt_id(menit, 1)} menit ≈ {fmt_id((n_rs + n_pso) * menit / 60, 1)} jam")
cek("T · T_w (n latih 3.051, rho 0,1)", (H[S[0]]["T"], H[S[0]]["T_w"]), (T["T"], T["T_w"]))
cek("fitness > acuan minimum validasi", r["fitness"], acuan, cara=">")
cek("epoch terbaik = epoch pertama dengan F1 tertinggi · berhenti sesuai patience · perangkat (seed 42 · 43)",
    [[ya(h["epoch_terbaik"] == 1 + int(np.argmax(h["f1_per_epoch"]))), ya(len(h["f1_per_epoch"]) in (min(CFG["E"],
      h["epoch_terbaik"] + CFG["p"]), CFG["E"])), h["perangkat"]] for h in (H[s] for s in S)], [["ya", "ya", "cuda"]] * len(S))
selesai()

### Tahap 11 — Persiapan: arsip hasil anggaran 30 (sekali jalan, keputusan no. 25)

Anggaran RS dan PSO dikembalikan ke 50 evaluasi (pilihan B: RS diulang dari awal). Sel ini memindahkan hasil jalan 30 evaluasi (`log/rs_state.json`, `hasil/tahap11_rs.csv`, `bab4/tabel/tahap11_rs_top5.csv`, `log/pso_state.json`, `hasil/tahap12_pso.csv`) ke `log/arsip_anggaran30/`: salin, cocokkan SHA-256, lalu hapus berkas asal. Berkas arsip yang sudah ada tidak ditimpa, dan bila asal dan arsip sama-sama ada, keduanya dibiarkan (dilaporkan sebagai bentrok). Tanpa GPU. Hanya perlu dijalankan sekali, sebelum sel 2/2 Tahap 11; pada folder kerja baru tanpa hasil anggaran 30, statusnya "tidak ditemukan".

**Package:** `hashlib`, `json`, `shutil` (bawaan Python); `pandas`.

In [ ]:
# ══ Tahap 11 — Persiapan: arsipkan hasil anggaran 30 evaluasi (keputusan no. 25; tanpa GPU) ══
# Package: hashlib, json, shutil (bawaan Python); pandas
import hashlib, json, shutil
import pandas as pd
mulai()
ARSIP, ya = PATH["log"] / "arsip_anggaran30", (lambda x: "ya" if x else "tidak")
BERKAS = [PATH["log"] / "rs_state.json", PATH["hasil"] / "tahap11_rs.csv", PATH["tabel"] / "tahap11_rs_top5.csv",
          PATH["log"] / "pso_state.json", PATH["hasil"] / "tahap12_pso.csv"]
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
def n_baris(p):                                   # CSV: jumlah baris; state: jumlah kandidat (RS) atau evaluasi (PSO)
    if p.suffix == ".csv":
        return len(pd.read_csv(p))
    st = json.loads(p.read_text(encoding="utf-8"))
    return len(st["hasil"] if "hasil" in st else st["log"])
# 1) Salin ke arsip, cocokkan SHA-256, baru hapus berkas asal; arsip yang sudah ada tidak ditimpa
ARSIP.mkdir(exist_ok=True)
hasil = []
for p in BERKAS:
    q = ARSIP / p.name
    if p.exists() and not q.exists():
        h, n = sha(p), n_baris(p)
        shutil.copy2(p, q)
        if sha(q) == h:
            p.unlink(); status = "dipindah"
        else:
            q.unlink(); status = "GAGAL: salinan berbeda, asal dibiarkan"
    elif p.exists():
        h, n, status = sha(q), n_baris(q), "BENTROK: asal dan arsip sama-sama ada, tidak diubah"
    else:
        h, n, status = (sha(q), n_baris(q), "sudah diarsipkan") if q.exists() else ("-", 0, "tidak ditemukan")
    hasil.append((rel(p), n, h[:12], status, p.exists(), q.exists()))
# 2) Ringkasan
judul("11", "Persiapan: arsip anggaran 30", masukan=[x[0] for x in hasil], keluaran=rel(ARSIP) + "/")
print(f"{'Berkas':<32}{'baris':>6}  {'SHA-256':<13}status")
for b, n, h, s, *_ in hasil:
    print(f"{b:<32}{fmt_id(n):>6}  {h:<13}{s}")
n = {b.split("/")[-1]: x for b, x, *_ in hasil}
cek("di arsip · tersisa di lokasi asal (RS dan PSO mulai dari state kosong)", [sum(x[5] for x in hasil), sum(x[4] for x in hasil)],
    [len(BERKAS), 0])
cek("isi arsip konsisten: state RS = CSV RS · top-5 · state PSO = CSV PSO", [ya(n["rs_state.json"] == n["tahap11_rs.csv"] > 0),
    n["tahap11_rs_top5.csv"], ya(n["pso_state.json"] == n["tahap12_pso.csv"] > 0)], ["ya", min(5, n["tahap11_rs.csv"]), "ya"])
selesai()

## Tahap 11 — *Random search*

**Dasar naskah:** Bab III tahap 11; pers. (2.25), (2.27); Tabel 3.1. Anggaran 50 kandidat (keputusan no. 25, sesuai naskah): `F = default_rng(2024).random((50, 3))`, didekode ke lr = 1e-5 · 5^f₁ (skala log), wr = 0,2 · f₂, wd = 0,3 · f₃, lalu `fitness()` dievaluasi berurutan k = 0–49 (100 pelatihan). *State* disimpan setelah setiap kandidat; menjalankan ulang sel melanjutkan dari kandidat terakhir. Terpilih = argmax *fitness* (seri → k terkecil). Durasi setiap pelatihan disimpan (keputusan no. 21). Data uji tidak dibaca. GPU: perkiraan ±92 menit pada A100.

**Package:** `importlib`, `json`, `os`, `pathlib` (bawaan Python); `numpy`; `pandas`; `lib.wondr_latih`; `lib.wondr_optim`.

Sel 1/2 menulis `lib/wondr_optim.py` (dekode ruang pencarian, baca/simpan *state* atomik, ringkasan *fitness*; dipakai lagi oleh PSO). Sel 2/2 menyimpan `log/rs_state.json`, `hasil/tahap11_rs.csv`, dan `bab4/tabel/tahap11_rs_top5.csv`.

In [ ]:
%%writefile lib/wondr_optim.py
# lib/wondr_optim.py -- dekode ruang pencarian dan state yang dapat dilanjutkan (Tahap 11 random search, Tahap 12 PSO).
# Isinya fungsi saja.
import json, os
from pathlib import Path
import numpy as np


def dekode(f, ruang):
    """f di [0, 1]^3 -> (lr, wr, wd). lr skala log, wr dan wd linear (pers. 2.27; Bagian 4: lr = 1e-5 * 5^f1,
    wr = 0,2 * f2, wd = 0,3 * f3 untuk ruang Tabel 3.1)."""
    f = np.asarray(f, dtype=float)
    (a, b), (c, d), (e, g) = ruang["lr"], ruang["wr"], ruang["wd"]
    return float(a * (b / a) ** f[0]), float(c + (d - c) * f[1]), float(e + (g - e) * f[2])


def baca_state(path, awal):
    """State tersimpan bila berkasnya ada; bila belum ada, salinan state awal."""
    p = Path(path)
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else json.loads(json.dumps(awal))


def simpan_state(path, state):
    """Tulis state json secara atomik (.tmp lalu os.replace) supaya tidak rusak bila Colab terputus saat menulis."""
    p = Path(path)
    tmp = p.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, ensure_ascii=False, indent=1), encoding="utf-8")
    os.replace(tmp, p)


def ringkas_fitness(r, seeds):
    """Satu baris hasil fitness(): skor, epoch terbaik, dan durasi setiap seed (keputusan no. 21) beserta fitness."""
    h = r["hasil"]
    return {"fitness": r["fitness"], **{f"skor{s}": h[s]["f1_terbaik"] for s in seeds},
            **{f"epoch{s}": h[s]["epoch_terbaik"] for s in seeds}, **{f"detik{s}": h[s]["detik"] for s in seeds}}

In [ ]:
# ══ Tahap 11 — Random search (sel 2/2) ══
# Package: importlib (bawaan Python); numpy; pandas; lib.wondr_latih (fitness); lib.wondr_optim (dekode, state)
import importlib
import numpy as np, pandas as pd
import lib.wondr_latih as wl, lib.wondr_optim as wo
importlib.reload(wl); importlib.reload(wo)                  # pakai isi lib/ yang terbaru
mulai()
wl.pakai_konfig(CFG)
M, R, S, T, LOG = CFG["rs_M"], CFG["ruang"], CFG["seed_fitness"], TARGET["11"], PATH["log"] / "rs_state.json"
ya, dft = (lambda x: "ya" if x else "tidak"), baca_json(PATH["hasil"] / "tahap10_default.json")["fitness"]
sci = lambda v: f"{v:.2e}".replace(".", ",")                 # lr dalam notasi ilmiah, koma desimal
# 1) Kandidat pers. (2.27): F = default_rng(2024).random((M, 3)), didekode ke (lr, wr, wd); state lama dilanjutkan
F = np.random.default_rng(CFG["rs_seed"]).random((M, 3))
st = wo.baca_state(LOG, {"F": F.tolist(), "hasil": []})
if not np.array_equal(np.array(st["F"]), F):
    raise ValueError("log/rs_state.json berasal dari konfigurasi lain (seed atau M berbeda); pindahkan berkas itu dulu.")
# 2) Evaluasi fitness berurutan k = 0..M-1; state disimpan setelah setiap kandidat
for k in range(len(st["hasil"]), M):
    lr, wr, wd = wo.dekode(F[k], R)
    st["hasil"].append({"k": k, "f1": F[k, 0], "f2": F[k, 1], "f3": F[k, 2], "lr": lr, "wr": wr, "wd": wd,
                        **wo.ringkas_fitness(wl.fitness(lr, wr, wd), S)})
    wo.simpan_state(LOG, st)
    x, terbaik = st["hasil"][-1], max(h["fitness"] for h in st["hasil"])
    print(f"k {k:>2} · lr {sci(lr)} · wr {fmt_id(wr)} · wd {fmt_id(wd)} · skor {fmt_id(x['skor42'])} / {fmt_id(x['skor43'])} · "
          f"fitness {fmt_id(x['fitness'])} · terbaik {fmt_id(terbaik)} · {fmt_id((x['detik42'] + x['detik43']) / 60, 1)} menit")
# 3) Ringkasan: 5 teratas; terpilih = argmax fitness (seri -> k terkecil); selisih dengan skema default
df = pd.DataFrame(st["hasil"])
df.to_csv(PATH["hasil"] / "tahap11_rs.csv", index=False)
top = df.sort_values(["fitness", "k"], ascending=[False, True]).head(5)
simpan_tabel(top, "tahap11_rs_top5"); p = top.iloc[0]
judul("11", "Random search", masukan="hasil/tahap09_{latih,validasi}.parquet · hasil/tahap10_default.json",
      keluaran=["log/rs_state.json", "hasil/tahap11_rs.csv", "bab4/tabel/tahap11_rs_top5.csv"])
print(f"{'k':>3}{'lr':>11}{'wr':>8}{'wd':>8}{'skor42':>9}{'skor43':>9}{'fitness':>9}{'epoch':>7}{'menit':>7}")
for t in top.itertuples():
    print(f"{t.k:>3}{sci(t.lr):>11}{fmt_id(t.wr):>8}{fmt_id(t.wd):>8}{fmt_id(t.skor42):>9}{fmt_id(t.skor43):>9}{fmt_id(t.fitness):>9}"
          f"{f'{t.epoch42}/{t.epoch43}':>7}{fmt_id((t.detik42 + t.detik43) / 60, 1):>7}")
print(f"Terpilih k = {int(p.k)} · fitness {fmt_id(p.fitness)} · default {fmt_id(dft)} · selisih {fmt_id(p.fitness - dft)} · "
      f"total {fmt_id((df.detik42 + df.detik43).sum() / 60, 1)} menit")
cek("kandidat · pelatihan selesai", [len(df), 2 * len(df)], [T["kandidat"], T["pelatihan"]])
cek("kandidat = dekode F default_rng(2024) · lr, wr, wd di dalam ruang", [ya(all(np.allclose(wo.dekode(F[k], R), df.loc[k, ["lr", "wr",
    "wd"]].astype(float)) for k in range(len(df)))), ya(all(df[c].between(*R[c]).all() for c in ("lr", "wr", "wd")))], ["ya", "ya"])
cek("terpilih = argmax fitness (seri → k terkecil)", int(p.k), int(df.fitness.idxmax()))
selesai()

### Tahap 11 — Pembanding RS 50 dengan arsip RS 30 (tambahan; hanya membaca berkas)

Kandidat k = 0–29 pada F 50 baris sama dengan F 30 baris, dan pelatihan deterministik, sehingga skor, *fitness*, dan *epoch* terbaik harus identik dengan `log/arsip_anggaran30/rs_state.json` pada presisi penuh (bukti determinisme antar-sesi). Sel ini juga mencocokkan *state* dengan `hasil/tahap11_rs.csv` dan `bab4/tabel/tahap11_rs_top5.csv`, serta meringkas RS 50 terhadap skema *default*. Durasi hanya deskriptif. Tanpa GPU.

**Package:** `json` (bawaan Python); `numpy`; `pandas`.

In [ ]:
# ══ Tahap 11 — Pembanding: RS 50 vs arsip RS 30 dan isi berkas (tambahan; hanya membaca berkas, tanpa GPU) ══
# Package: json (bawaan Python); numpy; pandas
import json
import numpy as np, pandas as pd
mulai()
ya, f6 = (lambda x: "ya" if x else "tidak"), (lambda v: fmt_id(v, 6))
baca = lambda p: pd.DataFrame(json.loads(p.read_text(encoding="utf-8"))["hasil"])
# 1) State baru (50), CSV, top-5, arsip (30), dan fitness default
baru, lama = baca(PATH["log"] / "rs_state.json"), baca(PATH["log"] / "arsip_anggaran30" / "rs_state.json")
C, top = pd.read_csv(PATH["hasil"] / "tahap11_rs.csv"), pd.read_csv(PATH["tabel"] / "tahap11_rs_top5.csv")
dft = baca_json(PATH["hasil"] / "tahap10_default.json")["fitness"]
# 2) k = 0..29: kandidat, skor, fitness, dan epoch harus identik (pelatihan deterministik); durasi hanya deskriptif
b, n = baru.iloc[:len(lama)].reset_index(drop=True), len(lama)
KOL = {"kandidat (k, f1–f3, lr, wr, wd)": ["k", "f1", "f2", "f3", "lr", "wr", "wd"], "skor42, skor43, fitness": ["skor42", "skor43",
       "fitness"], "epoch42, epoch43": ["epoch42", "epoch43"]}
beda = {nm: float((b[c] - lama[c]).abs().to_numpy().max()) for nm, c in KOL.items()}
rasio = (b[["detik42", "detik43"]] / lama[["detik42", "detik43"]]).to_numpy()
judul("11", "Pembanding RS 50 vs arsip RS 30", masukan=["log/rs_state.json", "log/arsip_anggaran30/rs_state.json",
      "hasil/tahap11_rs.csv · bab4/tabel/tahap11_rs_top5.csv"])
print(f"k = 0–{n - 1} (n = {n}) · maks |selisih|:", " · ".join(f"{nm} {v:.1e}".replace(".", ",") for nm, v in beda.items()),
      f"· durasi baru/lama {fmt_id(rasio.min(), 2)}–{fmt_id(rasio.max(), 2)}")
d = (baru.skor42 - baru.skor43).abs()
print(f"RS 50: fitness {fmt_id(baru.fitness.min())}–{fmt_id(baru.fitness.max())} · di atas default (presisi penuh) "
      f"{int((baru.fitness > dft).sum())} · sama dengan default {int((baru.fitness == dft).sum())} · maks |skor42 − skor43| "
      f"{fmt_id(d.max())} (k = {int(baru.k[d.idxmax()])}) · terbaik {f6(baru.fitness.max())} vs kedua "
      f"{f6(baru.fitness.nlargest(2).iloc[1])}")
cek("k 0–29 identik dengan arsip: kandidat · skor dan fitness · epoch terbaik", [ya(v == 0) for v in beda.values()], ["ya"] * 3)
num = [c for c in C.columns if c in baru.columns]
cek("state = CSV · top-5 = 5 teratas (fitness turun, seri → k kecil) · terpilih = argmax", [ya(list(C.columns) == list(baru.columns)
    and np.allclose(C[num], baru[num], rtol=0, atol=1e-12)), ya(top.k.tolist() == baru.sort_values(["fitness", "k"], ascending=[False,
    True]).k.head(5).tolist()), int(top.k.iloc[0])], ["ya", "ya", int(baru.k[baru.fitness.idxmax()])])
selesai()

## Tahap 12.1 — PSO: uji fungsi lalu jalankan

**Dasar naskah:** Bab III tahap 12; Bab II subbab 2.10, Algoritma 2.1, pers. (2.28)–(2.34), Tabel 2.17; Tabel 3.2 dengan s = 10 dan G_max = 4 → 10 × (1 + 4) = 50 evaluasi (keputusan no. 25). ω = 0,729; c₁ = c₂ = 1,49445; V_max = 1 (rentang ruang ternormalisasi [0, 1]³); posisi awal U(0, 1), kecepatan awal U(−1, 1) dari `default_rng(7)`; setiap iterasi `r1 = rng.random((10, 3))`, `r2 = rng.random((10, 3))`. Semua partikel dievaluasi lebih dulu dengan pers. (2.29) berurutan dan lebih besar ketat (keputusan no. 8), lalu kecepatan dan posisi diperbarui serentak dengan pembatasan (2.32) dan dinding penyerap (2.33). Fungsi pembaruan diuji dengan contoh Tabel 2.17. *State* (X, V, P, fP, g, iterasi, partikel, `rng.bit_generator.state`) disimpan setelah setiap evaluasi dan pembaruan; menjalankan ulang sel melanjutkan dari titik terakhir. Data uji tidak dibaca. GPU: perkiraan ±94 menit pada A100.

**Package:** `importlib` (bawaan Python); `numpy`; `pandas`; `lib.wondr_latih`; `lib.wondr_optim`; `lib.wondr_pso`.

Sel 1/2 menulis `lib/wondr_pso.py`. Sel 2/2 menyimpan `log/pso_state.json` dan `hasil/tahap12_pso.csv`.

In [ ]:
%%writefile lib/wondr_pso.py
# lib/wondr_pso.py -- particle swarm optimization (Tahap 12.1): Algoritma 2.1 dan pers. (2.29)-(2.33) Bab II;
# pembaruan serentak, dinding penyerap, state dapat dilanjutkan. Isinya fungsi saja.
import numpy as np
from lib.wondr_optim import baca_state, simpan_state


def langkah(X, V, P, g, r1, r2, omega, c1, c2, vmax, bl, bu):
    """Pembaruan serentak semua partikel: kecepatan (2.30) lalu dibatasi (2.32), posisi (2.31), dinding penyerap (2.33).
    Mengembalikan (X baru, V baru, mask dimensi yang keluar ruang sebelum dipotong)."""
    V = omega * V + c1 * r1 * (P - X) + c2 * r2 * (g - X)        # pers. (2.30)
    V = np.clip(V, -vmax, vmax)                                   # pers. (2.32)
    X = X + V                                                     # pers. (2.31)
    luar = (X < bl) | (X > bu)
    return np.clip(X, bl, bu), np.where(luar, 0.0, V), luar      # pers. (2.33)


def perbarui_terbaik(f, x, i, P, fP, g, fg):
    """Pers. (2.29) untuk satu partikel sesuai urutan evaluasi (keputusan no. 8): personal best dan global best diganti hanya
    bila fitness lebih besar ketat; bila seri, posisi lama dipertahankan. P dan fP diubah di tempat.
    Mengembalikan (g, fg, personal best diperbarui?, global best diperbarui?)."""
    p_baru, g_baru = bool(f > fP[i]), bool(f > fg)
    if p_baru:
        P[i], fP[i] = x, f
    if g_baru:
        g, fg = x.copy(), f
    return g, fg, p_baru, g_baru


def contoh_tabel_2_17():
    """Contoh Bab II 2.10.6: phi = -(l - 3)^2 + 10 pada [0, 6]; dua partikel; Vmax = 6; r1, r2 dari naskah.
    Mengembalikan posisi setelah iterasi 1 dan 2 serta fitness global best akhir."""
    phi = lambda x: -(x - 3.0) ** 2 + 10.0
    X, V = np.array([[1.0], [4.0]]), np.array([[0.5], [-0.5]])
    P, fP, g, fg, posisi = X.copy(), np.full(2, -np.inf), None, -np.inf, []
    for r1, r2 in (([0.4, 0.6], [0.95, 0.3]), ([0.7, 0.2], [0.5, 0.8])):
        if g is None:                                             # evaluasi posisi awal
            for i in range(2):
                g, fg, *_ = perbarui_terbaik(phi(X[i, 0]), X[i].copy(), i, P, fP, g, fg)
        X, V, _ = langkah(X, V, P, g, np.array(r1)[:, None], np.array(r2)[:, None], 0.729, 1.49445, 1.49445, 6.0, 0.0, 6.0)
        for i in range(2):
            g, fg, *_ = perbarui_terbaik(phi(X[i, 0]), X[i].copy(), i, P, fP, g, fg)
        posisi.append(X[:, 0].tolist())
    return posisi, fg


def jalankan(path, evaluasi, s, G, par, seed, cetak=print):
    """Algoritma 2.1 di ruang ternormalisasi [0, 1]^3. X = U(0, 1) dan V = U(-Vmax, Vmax) dari default_rng(seed); setiap
    iterasi: r1 = rng.random((s, 3)), r2 = rng.random((s, 3)). evaluasi(f) -> dict dengan kunci 'fitness'. State disimpan
    setelah setiap evaluasi dan setiap pembaruan; bila berkas state ada, proses dilanjutkan dari titik terakhir."""
    rng = np.random.default_rng(seed)
    X0, V0 = rng.uniform(0, 1, (s, 3)), rng.uniform(-par["V_max"], par["V_max"], (s, 3))
    st = baca_state(path, {"s": s, "G_max": G, "seed": seed, "X0": X0.tolist(), "t": 0, "i": 0, "X": X0.tolist(),
                           "V": V0.tolist(), "P": X0.tolist(), "fP": [None] * s, "g": None, "fg": None,
                           "dinding": [[False] * 3] * s, "rng": rng.bit_generator.state, "log": [], "iterasi": []})
    if (st["s"], st["G_max"], st["seed"]) != (s, G, seed) or not np.array_equal(np.array(st["X0"]), X0):
        raise ValueError(f"{path} berasal dari konfigurasi PSO lain (s, G_max, atau seed berbeda); pindahkan berkas itu dulu.")
    X, V, P = (np.array(st[k], dtype=float) for k in ("X", "V", "P"))
    fP = np.array([-np.inf if v is None else v for v in st["fP"]])
    g, fg = (None, -np.inf) if st["g"] is None else (np.array(st["g"]), st["fg"])
    rng.bit_generator.state = st["rng"]
    while st["t"] <= G:
        t = st["t"]
        while st["i"] < s:                                        # evaluasi semua partikel, (2.29) berurutan
            i = st["i"]
            x = X[i].copy()
            h = evaluasi(x)
            g, fg, pb, gb = perbarui_terbaik(h["fitness"], x, i, P, fP, g, fg)
            baris = {"eval": len(st["log"]), "t": t, "i": i, "f1": x[0], "f2": x[1], "f3": x[2], **h, "p_baru": pb,
                     "g_baru": gb, "dinding": bool(np.any(st["dinding"][i])), "gbest": fg}
            st["log"].append(baris)
            st.update(i=i + 1, P=P.tolist(), fP=[None if v == -np.inf else v for v in fP.tolist()], g=g.tolist(), fg=fg)
            simpan_state(path, st)
            cetak(baris)
        if t < G:                                                 # pembaruan serentak (2.30)-(2.33)
            r1, r2 = rng.random((s, 3)), rng.random((s, 3))
            X, V, luar = langkah(X, V, P, g, r1, r2, par["omega"], par["c1"], par["c2"], par["V_max"], 0.0, 1.0)
            st["iterasi"].append({"t": t + 1, "maks_abs_v": float(np.abs(V).max()), "dimensi_dinding": int(luar.sum())})
            st.update(X=X.tolist(), V=V.tolist(), dinding=luar.tolist(), rng=rng.bit_generator.state, t=t + 1, i=0)
        else:
            st["t"] = t + 1
        simpan_state(path, st)
    return st

In [ ]:
# ══ Tahap 12.1 — PSO: uji fungsi lalu jalankan (sel 2/2) ══
# Package: importlib (bawaan Python); numpy; pandas; lib.wondr_latih (fitness); lib.wondr_optim; lib.wondr_pso
import importlib
import numpy as np, pandas as pd
import lib.wondr_latih as wl, lib.wondr_optim as wo, lib.wondr_pso as wp
importlib.reload(wl); importlib.reload(wo); importlib.reload(wp)   # pakai isi lib/ yang terbaru
mulai()
wl.pakai_konfig(CFG)
p, R, S, T, LOG = CFG["pso"], CFG["ruang"], CFG["seed_fitness"], TARGET["12.1"], PATH["log"] / "pso_state.json"
ya, sci = (lambda x: "ya" if x else "-"), (lambda v: f"{v:.2e}".replace(".", ","))
# 1) Uji fungsi pembaruan dengan contoh Bab II 2.10.6 (Tabel 2.17)
pos, fg_contoh = wp.contoh_tabel_2_17()
# 2) PSO Algoritma 2.1 (s partikel, G_max iterasi pembaruan) di ruang [0, 1]^3; fitness = pers. (2.25) pada posisi terdekode
def evaluasi(f):
    lr, wr, wd = wo.dekode(f, R)
    return {"lr": lr, "wr": wr, "wd": wd, **wo.ringkas_fitness(wl.fitness(lr, wr, wd), S)}
cetak = lambda b: print(f"t {b['t']} · partikel {b['i']} · lr {sci(b['lr'])} · wr {fmt_id(b['wr'])} · wd {fmt_id(b['wd'])} · fitness "
                        f"{fmt_id(b['fitness'])} · p_i diperbarui {ya(b['p_baru'])} · dinding {ya(b['dinding'])} · gbest "
                        f"{fmt_id(b['gbest'])} · {fmt_id((b['detik42'] + b['detik43']) / 60, 1)} menit")
st = wp.jalankan(LOG, evaluasi, p["s"], p["G_max"], p, p["seed"], cetak)
# 3) Simpan log evaluasi; ringkasan satu baris per iterasi
df = pd.DataFrame(st["log"])
df.to_csv(PATH["hasil"] / "tahap12_pso.csv", index=False)
it = {x["t"]: x for x in st["iterasi"]}
judul("12.1", "PSO", masukan="hasil/tahap09_{latih,validasi}.parquet · lib/wondr_pso.py",
      keluaran=["log/pso_state.json", "hasil/tahap12_pso.csv"])
print(f"{'t':>2}{'evaluasi':>10}{'fitness maks':>14}{'rata-rata':>11}{'p_i baru':>10}{'dimensi dinding':>17}{'maks |v|':>10}{'gbest':>8}")
for t, g in df.groupby("t"):
    x = it.get(t, {})                                      # t = 0: posisi awal, belum ada pembaruan
    print(f"{t:>2}{fmt_id(len(g)):>10}{fmt_id(g.fitness.max()):>14}{fmt_id(g.fitness.mean()):>11}{fmt_id(int(g.p_baru.sum())):>10}"
          f"{(fmt_id(x['dimensi_dinding']) if x else '-'):>17}{(fmt_id(x['maks_abs_v']) if x else '-'):>10}{fmt_id(g.gbest.iloc[-1]):>8}")
gb = df.loc[df.fitness == st["fg"]].iloc[0]
print(f"global best: lr {sci(gb.lr)} · wr {fmt_id(gb.wr)} · wd {fmt_id(gb.wd)} · fitness {fmt_id(st['fg'])} (evaluasi ke-{int(gb['eval']) + 1}, "
      f"t {int(gb.t)}, partikel {int(gb.i)}) · total {fmt_id((df.detik42 + df.detik43).sum() / 60, 1)} menit")
cek("contoh Tabel 2.17: posisi iterasi 1 · iterasi 2 · fitness global best", [pos[0], pos[1], fg_contoh],
    [list(T["iter1"]), list(T["iter2"]), T["gbest_iter2"]], toleransi=T["toleransi"], d=4)
cek("evaluasi selesai · global best tidak pernah turun · baris global best = posisi g di state", [len(df), ya(df.gbest
    .is_monotonic_increasing), ya(np.allclose(gb[["f1", "f2", "f3"]].astype(float), st["g"]))], [T["evaluasi"], "ya", "ya"])
cek("posisi di [0, 1] · maks |V| ≤ V_max · lr, wr, wd = dekode posisi", [ya(df[["f1", "f2", "f3"]].stack().between(0, 1).all()),
    ya(max([x["maks_abs_v"] for x in st["iterasi"]] + [0]) <= p["V_max"]), ya(all(np.allclose(wo.dekode(r[["f1", "f2", "f3"]]
    .astype(float), R), r[["lr", "wr", "wd"]].astype(float)) for _, r in df.iterrows()))], ["ya", "ya", "ya"])
selesai()

### Tahap 12.1 — Audit akhir (tambahan, diminta pengguna)

Hanya membaca berkas; tanpa GPU dan tanpa menulis apa pun. **Audit A:** `log/pso_state.json` dibandingkan dengan `hasil/tahap12_pso.csv`, lalu Algoritma 2.1 diputar ulang secara mandiri (tanpa `lib.wondr_pso`) dari `default_rng(7)` dan nilai *fitness* yang tersimpan. Hasil putar ulang dicocokkan dengan posisi, keputusan *personal best*/*global best* (termasuk seri pada presisi penuh), tanda dinding, ringkasan iterasi, dan *state* akhir. Setiap posisi di batas ruang dijelaskan sebagai dinding (2.33), tetap (V = 0), atau tepat (|V| = V_max, mendarat tepat di batas sehingga tidak "di luar" ruang). **Audit B:** anggaran evaluasi, definisi *fitness*, dan keabsahan skor pada skema *default*, RS, dan PSO; posisi PSO yang dievaluasi ulang; perbandingan *fitness* terbaik pada presisi penuh (deskriptif, bukan uji signifikansi); dan kode sumber `fitness()`, yang hanya membaca `tahap09_latih` dan `tahap09_validasi`.

**Package:** `json`, `inspect` (bawaan Python); `numpy`; `pandas`; `lib.wondr_optim`; `lib.wondr_latih` (hanya kode sumber).

In [ ]:
# ══ Tahap 12.1 — Audit A: konsistensi evaluasi dan putar ulang PSO (tambahan; hanya membaca berkas, tanpa GPU) ══
# Package: json (bawaan Python); numpy; pandas; lib.wondr_optim (dekode)
import json, numpy as np, pandas as pd, lib.wondr_optim as wo
mulai()
p, R, s, G, ya, NM = CFG["pso"], CFG["ruang"], CFG["pso"]["s"], CFG["pso"]["G_max"], (lambda x: "ya" if x else "tidak"), ("lr", "wr", "wd")
# 1) State dan CSV PSO
st = json.loads((PATH["log"] / "pso_state.json").read_text(encoding="utf-8"))
L, C = pd.DataFrame(st["log"]), pd.read_csv(PATH["hasil"] / "tahap12_pso.csv")
# 2) Putar ulang Algoritma 2.1 secara mandiri (tanpa lib.wondr_pso) dari seed 7 dan fitness tersimpan, urut evaluasi
rng = np.random.default_rng(p["seed"])
X, V = rng.uniform(0, 1, (s, 3)), rng.uniform(-p["V_max"], p["V_max"], (s, 3))
P, fP, g, fg, kena, U, it, dd, seri, tetap = X.copy(), np.full(s, -np.inf), None, -np.inf, np.zeros((s, 3), bool), [], [], [], [], []
for t in range(G + 1):
    if t:
        r1, r2 = rng.random((s, 3)), rng.random((s, 3))
        V = np.clip(p["omega"] * V + p["c1"] * r1 * (P - X) + p["c2"] * r2 * (g - X), -p["V_max"], p["V_max"])
        X = X + V; kena, di = (X < 0) | (X > 1), (X == 0) | (X == 1)
        tetap += [(t, i, " ".join(f"{NM[j]} {'tetap' if V[i, j] == 0 else 'tepat' if abs(V[i, j]) == p['V_max'] else '?'}"
                                  for j in np.nonzero(di[i])[0])) for i in np.nonzero(~kena.any(1) & di.any(1))[0]]
        dd += [f"t{t} p{i} {NM[j]} {fmt_id(X[i, j], 3)}→{int(X[i, j] > 1)}" for i, j in zip(*np.nonzero(kena))]
        X = np.clip(X, 0, 1); V[kena] = 0; it.append({"t": t, "maks_abs_v": float(np.abs(V).max()), "dimensi_dinding": int(kena.sum())})
    for i in range(s):
        f = L.fitness[t * s + i]; pb, gb = f > fP[i], f > fg
        seri += [f"t{t} p{i}: {n} seri {fmt_id(f, 6)}" for n, v in (("p_i", fP[i]), ("g", fg)) if f == v]
        seri += [f"t{t} p{i}: {n} naik {fmt_id(v, 6)} → {fmt_id(f, 6)}" for n, v in (("p_i", fP[i]), ("g", fg)) if v < f < v + 5e-5]
        if pb: P[i], fP[i] = X[i], f
        if gb: g, fg = X[i].copy(), f
        U.append([t, i, *X[i], bool(pb), bool(gb), bool(kena[i].any()), fg])
U = pd.DataFrame(U, columns=["t", "i", "f1", "f2", "f3", "p_baru", "g_baru", "dinding", "gbest"])
F, num = L[["f1", "f2", "f3"]].to_numpy(), [c for c in C.columns if C[c].dtype != bool]
batas = ((F == 0) | (F == 1)).any(1)
judul("12.1", "Audit A: evaluasi dan putar ulang PSO", masukan=["log/pso_state.json", "hasil/tahap12_pso.csv"])
print("posisi tepat di batas:", " · ".join(f"{NM[j]} {'min' if k == 0 else 'maks'} {int((F[:, j] == k).sum())}" for j in range(3)
      for k in (0, 1)), f"| baris di batas {int(batas.sum())} = bertanda dinding {int(L.dinding.sum())} + tanpa dinding {len(tetap)}:",
      " · ".join(f"t{t} p{i} {k}" for t, i, k in tetap) or "-")
print("dinding (posisi sebelum dipotong → batas):", " · ".join(dd) or "tidak ada")
print("seri atau kenaikan < 0,00005 (presisi penuh):", "; ".join(seri) or "tidak ada")
cek("state = CSV (kolom, angka, tanda) · pasangan (t, partikel) unik · eval = t·s + partikel", [ya(list(C.columns) == list(L.columns)
    and np.allclose(C[num], L[num], rtol=0, atol=1e-12) and C.select_dtypes(bool).equals(L.select_dtypes(bool))),
    len(set(zip(L.t, L.i))), ya(L["eval"].tolist() == list(range(len(L))) and (L.t * s + L.i == L["eval"]).all())], ["ya", s * (G + 1), "ya"])
cek("putar ulang = log: posisi · p_i · g · dinding · gbest · ringkasan iterasi", [ya(np.allclose(U[["f1", "f2", "f3"]], F, rtol=0,
    atol=1e-12)), ya(U.p_baru.equals(L.p_baru)), ya(U.g_baru.equals(L.g_baru)), ya(U.dinding.equals(L.dinding)), ya(np.array_equal(
    U.gbest, L.gbest)), ya(pd.DataFrame(it).equals(pd.DataFrame(st["iterasi"])))], ["ya"] * 6)
cek("putar ulang = state akhir (X, V, P, g) · lr, wr, wd = dekode · baris di batas = dinding/tetap/tepat", [ya(all(np.allclose(
    np.array(st[k]), v, rtol=0, atol=1e-12) for k, v in (("X", X), ("V", V), ("P", P), ("g", g)))), ya(all(np.allclose(wo.dekode(r, R),
    q, rtol=1e-12) for r, q in zip(F, L[["lr", "wr", "wd"]].to_numpy()))), ya(int(batas.sum()) == int(L.dinding.sum()) + len(tetap)
    and not any("?" in k or L.dinding[t * s + i] for t, i, k in tetap))], ["ya"] * 3)
cek("global best = maks fitness · evaluasi ke- (pertama yang mencapai maks = baris g_baru terakhir)", [ya(st["fg"] == L.fitness.max()),
    int(L.fitness.to_numpy().argmax()) + 1], ["ya", int(L.index[L.g_baru][-1]) + 1])
selesai()

In [ ]:
# ══ Tahap 12.1 — Audit B: fitness dan keadilan perbandingan RS, PSO, default (tambahan; hanya membaca berkas, tanpa GPU) ══
# Package: inspect (bawaan Python); numpy; pandas; lib.wondr_latih (hanya kode sumber fitness dan latih)
import inspect, numpy as np, pandas as pd, lib.wondr_latih as wl
mulai()
ya, f6, S, sci = (lambda x: "ya" if x else "tidak"), (lambda v: fmt_id(v, 6)), CFG["seed_fitness"], (lambda v: f"{v:.3e}".replace(".", ","))
# 1) Hasil ketiga skema (presisi penuh); kem = posisi PSO yang dievaluasi lebih dari sekali
rs, ps = pd.read_csv(PATH["hasil"] / "tahap11_rs.csv"), pd.read_csv(PATH["hasil"] / "tahap12_pso.csv")
dft, src = baca_json(PATH["hasil"] / "tahap10_default.json"), inspect.getsource(wl.fitness)
br, bp, kem = rs.loc[rs.fitness.idxmax()], ps.loc[ps.fitness.idxmax()], ps[ps.duplicated(["f1", "f2", "f3"], keep=False)]
judul("12.1", "Audit B: fitness dan perbandingan", masukan=["hasil/tahap10_default.json", "hasil/tahap11_rs.csv", "hasil/tahap12_pso.csv"])
print(f"{'Skema':<9}{'evaluasi':>9}{'pelatihan':>10}{'fitness terbaik':>17}{'skor42':>10}{'skor43':>10}{'menit':>8}  konfigurasi")
for n, ev, x, mnt in (("default", 1, dft, (dft["detik42"] + dft["detik43"]) / 60), ("RS", len(rs), br, (rs.detik42 + rs.detik43).sum() / 60),
                      ("PSO", len(ps), bp, (ps.detik42 + ps.detik43).sum() / 60)):
    print(f"{n:<9}{ev:>9}{2 * ev:>10}{f6(x['fitness']):>17}{f6(x['skor42']):>10}{f6(x['skor43']):>10}{fmt_id(mnt, 1):>8}  "
          f"lr {sci(x['lr'])} · wr {fmt_id(x['wr'])} · wd {fmt_id(x['wd'])}")
print(f"selisih presisi penuh: PSO − RS {f6(bp.fitness - br.fitness)} · PSO − default {f6(bp.fitness - dft['fitness'])} · RS − default "
      f"{f6(br.fitness - dft['fitness'])} (deskriptif di validasi, bukan uji signifikansi)")
print(f"RS dengan fitness = terbaik PSO (presisi penuh): {int((rs.fitness == bp.fitness).sum())} · di atas default: RS "
      f"{int((rs.fitness > dft['fitness']).sum())}/{len(rs)} · PSO {int((ps.fitness > dft['fitness']).sum())}/{len(ps)}")
print(f"posisi PSO kembar {len(kem)} baris:", " · ".join(f"eval {[e + 1 for e in d['eval']]} skor {f6(d.skor42.iloc[0])}/"
      f"{f6(d.skor43.iloc[0])}" for _, d in kem.groupby(["f1", "f2", "f3"])) or "-")
cek("anggaran sama: evaluasi RS · PSO · pelatihan RS · PSO", [len(rs), len(ps), 2 * len(rs), 2 * len(ps)],
    [TARGET["11"]["kandidat"], TARGET["12.1"]["evaluasi"], TARGET["11"]["pelatihan"], 2 * TARGET["12.1"]["evaluasi"]])
cek("fitness = (skor42 + skor43)/2 pada default · RS · PSO", [ya(abs(dft["fitness"] - (dft["skor42"] + dft["skor43"]) / 2) < 1e-15)]
    + [ya(np.allclose(d.fitness, (d.skor42 + d.skor43) / 2, rtol=0, atol=1e-15)) for d in (rs, ps)], ["ya"] * 3)
cek("RS dan PSO: skor di (0, 1] · epoch terbaik 1–4 · durasi > 0 · posisi PSO kembar → skor sama", [ya(all(d[c].gt(0).all() and
    d[c].le(1).all() for d in (rs, ps) for c in ("skor42", "skor43"))), ya(all(d[c].isin(range(1, CFG["E"] + 1)).all() for d in (rs, ps)
    for c in ("epoch42", "epoch43"))), ya(all((d[c] > 0).all() for d in (rs, ps) for c in ("detik42", "detik43"))),
    ya(kem.groupby(["f1", "f2", "f3"])[["skor42", "skor43"]].nunique().le(1).all().all())], ["ya"] * 4)
cek("fitness() hanya membaca tahap09 latih + validasi · seed (42, 43) · early stopping bawaan", [ya('baca("latih"), baca("validasi")' in src
    and "uji" not in src), ya('_K["seed_fitness"]' in src and tuple(S) == (42, 43)), ya("early_stopping" not in src and inspect.signature(
    wl.latih).parameters["early_stopping"].default is True)], ["ya"] * 3)
selesai()

### Tahap 12.1 — Audit C (tambahan, diminta pengguna)

Hanya membaca. Jumlah evaluasi per iterasi dan baris tanpa hasil; p_i dan g dihitung ulang langsung dari riwayat; dua *global best* terakhir pada presisi penuh (t 0 p 8 0,918519334 vs t 4 p 9 0,918539609, lebih besar ketat); t 0 p 0–5 dibandingkan dengan jalan 30 evaluasi yang diarsipkan (identik pada presisi penuh); waktu ubah dan SHA-256 `lib/`.

**Catatan:** cek ketiga memberi ⚠️ karena syaratnya keliru: semua *lib* dibandingkan dengan waktu mulai RS, padahal RS tidak meng-*import* `wondr_pso.py`. Dibaca per berkas, setiap *lib* terakhir diubah sebelum jalan yang memakainya, dan SHA-256-nya sama dengan satu-satunya versi di riwayat Git (lihat CLAUDE.md, status 12.1). Sel dibiarkan seperti yang dijalankan.

**Package:** `datetime`, `hashlib`, `json` (bawaan Python); `numpy`; `pandas`.

In [ ]:
# ══ Tahap 12.1 — Audit C: per iterasi, p_i dari riwayat, dua g terakhir, reprodusibilitas, lib (tambahan; hanya membaca) ══
# Package: datetime, hashlib, json (bawaan Python); numpy; pandas
import datetime, hashlib, json
import numpy as np, pandas as pd
mulai()
ya, f9, s, G = (lambda x: "ya" if x else "tidak"), (lambda v: f"{v:.9f}".replace(".", ",")), CFG["pso"]["s"], CFG["pso"]["G_max"]
baca = lambda p: json.loads(p.read_text(encoding="utf-8"))
st, rs = baca(PATH["log"] / "pso_state.json"), baca(PATH["log"] / "rs_state.json")
L, A = pd.DataFrame(st["log"]), pd.DataFrame(baca(PATH["log"] / "arsip_anggaran30" / "pso_state.json")["log"])
SK, POS = ["skor42", "skor43", "fitness", "epoch42", "epoch43"], ["f1", "f2", "f3"]
# 1) Isi per iterasi dan baris tanpa hasil
per_t, kosong = L.groupby("t").i.nunique().reindex(range(G + 1), fill_value=0).tolist(), int(L[SK].isna().any(axis=1).sum())
# 2) p_i dan g dihitung ulang langsung dari riwayat (maks per partikel, kejadian pertama), lalu dibandingkan dengan state akhir
iP = [L.index[L.i == i][L.fitness[L.i == i].to_numpy().argmax()] for i in range(s)]
fP, P = L.fitness[iP].to_numpy(), L.loc[iP, POS].to_numpy()
# 3) Dua global best terakhir (baris g_baru) pada presisi penuh
g2 = L[L.g_baru].tail(2)
# 4) t = 0 partikel yang posisinya sama dengan jalan 30 evaluasi yang diarsipkan (pelatihan terpisah, 22-10-2026)
n0 = int((A.t == 0).sum())
b, a = L[(L.t == 0) & (L.i < n0)].reset_index(drop=True), A[A.t == 0].reset_index(drop=True)
# 5) Waktu ubah lib terhadap jendela jalan RS dan PSO (mulai ≈ waktu ubah state − total durasi pelatihan)
wkt = lambda p: datetime.datetime.fromtimestamp(p.stat().st_mtime)
dur = lambda h: datetime.timedelta(seconds=float(sum(x["detik42"] + x["detik43"] for x in h)))
mulai_rs, mulai_pso = wkt(PATH["log"] / "rs_state.json") - dur(rs["hasil"]), wkt(PATH["log"] / "pso_state.json") - dur(st["log"])
LIB = ["wondr_latih.py", "wondr_optim.py", "wondr_pso.py"]
judul("12.1", "Audit C: rincian PSO", masukan=["log/pso_state.json · log/rs_state.json", "log/arsip_anggaran30/pso_state.json · lib/"])
for (_, r), nm in zip(g2.iterrows(), ("g sebelumnya", "g akhir")):
    print(f"{nm}: t {r.t} p {r.i} · skor42 {f9(r.skor42)} · skor43 {f9(r.skor43)} · fitness {f9(r.fitness)} · epoch {r.epoch42}/{r.epoch43}"
          f" · lr {r.lr:.6e} · wr {r.wr:.6f} · wd {r.wd:.6f}".replace(".", ","))
d = float(g2.fitness.iloc[-1] - g2.fitness.iloc[0])
print(f"selisih fitness g akhir − g sebelumnya = {d:.3e}".replace(".", ","), "→ (2.29) lebih besar ketat:", ya(d > 0),
      "· bila seri, g lama dipertahankan (keputusan no. 8)")
print(f"t 0 p 0–{n0 - 1} vs arsip: maks |selisih| posisi {np.abs(b[POS] - a[POS]).to_numpy().max():.1e} · skor/fitness/epoch "
      f"{np.abs(b[SK] - a[SK]).to_numpy().max():.1e}".replace(".", ","))
for f in LIB:
    print(f"lib/{f}: diubah {wkt(PATH['lib'] / f):%d-%m %H:%M} · SHA-256 {hashlib.sha256((PATH['lib'] / f).read_bytes()).hexdigest()[:12]}")
print(f"perkiraan mulai RS {mulai_rs:%d-%m %H:%M} · mulai PSO {mulai_pso:%d-%m %H:%M}")
cek("evaluasi per iterasi t 0–4 · baris tanpa skor/fitness/epoch · kolom skor42 dan skor43 ada", [per_t, kosong,
    ya({"skor42", "skor43"} <= set(L.columns))], [[s] * (G + 1), 0, "ya"])
cek("p_i dari riwayat = state (fP · P) · g = maks seluruh fitness = fg state = posisi g", [ya(np.array_equal(fP, np.array(st["fP"], float))),
    ya(np.allclose(P, np.array(st["P"]), rtol=0, atol=0)), ya(L.fitness.max() == st["fg"] == g2.fitness.iloc[-1] and np.array_equal(
    g2[POS].iloc[-1].to_numpy(float), np.array(st["g"])))], ["ya"] * 3)
cek("t 0 vs arsip identik (posisi · skor, fitness, epoch) · lib tidak diubah selama jalan RS dan PSO", [ya(b[POS].equals(a[POS])),
    ya(b[SK].equals(a[SK])), ya(all(wkt(PATH["lib"] / f) < min(mulai_rs, mulai_pso) for f in LIB))], ["ya"] * 3)
selesai()

### Tahap 11–12.1 — Audit penutup (tambahan, diminta pengguna)

Hanya membaca. Keberadaan 19 artefak reproduksibilitas, keabsahan 50 kandidat RS dan 50 evaluasi PSO (dua *seed*, rata-rata), terbaik masing-masing pada presisi penuh, SHA-256 berkas dan isi `tahap09_latih`/`tahap09_validasi` saat ini, serta log versi.

**Catatan:** cek versi membandingkan `pertama` (sesi A100 pertama) dengan `terbaru` di `log/versi_package.json`, yaitu sesi **terakhir yang menjalankan Tahap 0.1** (ditimpa setiap kali), bukan catatan sesi pelatihan RS/PSO. Versi saat RS/PSO tidak tercatat per jalan; keterbatasan ini dicatat di CLAUDE.md (keputusan no. 26). Kesimpulan RS vs PSO ditulis sesuai keputusan no. 27.

**Package:** `hashlib`, `json` (bawaan Python); `numpy`; `pandas`.

In [ ]:
# ══ Tahap 11–12.1 — Audit penutup: kelengkapan, keterbandingan, metadata (tambahan; hanya membaca, tanpa GPU) ══
# Package: hashlib, json (bawaan Python); numpy; pandas
import hashlib, json
import numpy as np, pandas as pd
mulai()
ya, rp = (lambda x: "ya" if x else "tidak"), (lambda v: repr(float(v)).replace(".", ","))
baca, sha = (lambda p: json.loads(p.read_text(encoding="utf-8"))), (lambda b: hashlib.sha256(b).hexdigest())
# 1) Artefak yang dibutuhkan untuk reproduksibilitas (hanya dicek keberadaannya)
ART = ["log/rs_state.json", "hasil/tahap11_rs.csv", "bab4/tabel/tahap11_rs_top5.csv", "log/pso_state.json", "hasil/tahap12_pso.csv",
       "hasil/tahap10_default.json", "hasil/tahap07_tokenisasi.json", "log/versi_package.json", "hasil/tahap09_latih.parquet",
       "hasil/tahap09_validasi.parquet", "lib/bantu.py", "lib/wondr_latih.py", "lib/wondr_optim.py", "lib/wondr_pso.py"] + [
       f"log/arsip_anggaran30/{n}" for n in ("rs_state.json", "tahap11_rs.csv", "tahap11_rs_top5.csv", "pso_state.json", "tahap12_pso.csv")]
hilang = [a for a in ART if not (PATH["akar"] / a).exists()]
# 2) Evaluasi RS dan PSO dari state; sah = dua skor ada, di [0, 1], dan fitness = rata-rata keduanya
rs_st, ps_st, dft = baca(PATH["log"] / "rs_state.json"), baca(PATH["log"] / "pso_state.json"), baca_json(PATH["hasil"] / "tahap10_default.json")
rs, ps = pd.DataFrame(rs_st["hasil"]), pd.DataFrame(ps_st["log"])
sah = lambda d: int((d[["skor42", "skor43"]].notna().all(axis=1) & d.skor42.between(0, 1) & d.skor43.between(0, 1)
                     & np.isclose(d.fitness, (d.skor42 + d.skor43) / 2, rtol=0, atol=1e-15)).sum())
br, bp = rs.loc[rs.fitness.idxmax()], ps.loc[ps.fitness.idxmax()]               # idxmax = kejadian pertama (RS: k terkecil)
# 3) Metadata data latih dan validasi saat ini: SHA-256 berkas dan SHA-256 isi (hash baris pandas)
D = {b: PATH["hasil"] / f"tahap09_{b}.parquet" for b in ("latih", "validasi")}
v = baca(PATH["log"] / "versi_package.json"); K = ("torch", "transformers", "tokenizers")
judul("11–12.1", "Audit penutup RS dan PSO", masukan="log/ · hasil/ · bab4/tabel/ · lib/ (hanya membaca)", keluaran="(tidak ada)")
print(f"artefak ada {len(ART) - len(hilang)} dari {len(ART)}" + (f" · hilang: {', '.join(hilang)}" if hilang else ""))
print(f"RS : {len(rs)} kandidat · sah {sah(rs)} · terbaik k {int(br.k)} · fitness {rp(br.fitness)} · skor {rp(br.skor42)} / {rp(br.skor43)}")
print(f"PSO: {len(ps)} evaluasi · sah {sah(ps)} · terbaik t {int(bp.t)} p {int(bp.i)} · fitness {rp(bp.fitness)} · skor {rp(bp.skor42)} / "
      f"{rp(bp.skor43)}")
print(f"selisih presisi penuh: PSO − RS {rp(bp.fitness - br.fitness)} · PSO − default {rp(bp.fitness - dft['fitness'])} · RS − default "
      f"{rp(br.fitness - dft['fitness'])} (deskriptif, validasi)")
for b, p in D.items():
    d = pd.read_parquet(p)
    print(f"tahap09_{b}: {len(d)} baris · SHA-256 berkas {sha(p.read_bytes())}\n{' ' * 17}SHA-256 isi    "
          f"{sha(pd.util.hash_pandas_object(d, index=True).to_numpy().tobytes())}")
print(f"versi: pertama {v['pertama']['waktu']} · terbaru {v['terbaru']['waktu']} · "
      + " · ".join(f"{k} {v['terbaru']['package'][k]}" for k in K) + f" · {v['terbaru']['gpu']}")
cek("artefak reproduksibilitas tersedia", len(ART) - len(hilang), len(ART))
cek("RS: kandidat · k unik 0–49 · F = default_rng(2024) · evaluasi sah (2 seed, rata-rata)", [len(rs),
    ya(rs.k.tolist() == list(range(len(rs)))),
    ya(np.array_equal(np.array(rs_st["F"]), np.random.default_rng(CFG["rs_seed"]).random((CFG["rs_M"], 3)))), sah(rs)],
    [TARGET["11"]["kandidat"], "ya", "ya", TARGET["11"]["kandidat"]])
cek("PSO: evaluasi · pasangan (t, partikel) unik · evaluasi sah · gbest = maks fitness = fg", [len(ps), len(set(zip(ps.t, ps.i))), sah(ps),
    ya(bp.fitness == ps_st["fg"])], [TARGET["12.1"]["evaluasi"]] * 3 + ["ya"])
cek("versi torch · transformers · tokenizers · GPU = sesi A100 pertama", [ya(v["pertama"]["package"][k] == v["terbaru"]["package"][k])
    for k in K] + [ya(v["pertama"]["gpu"] == v["terbaru"]["gpu"])], ["ya"] * 4)
selesai()

## Tahap 12.2 — Ringkasan PSO dan konvergensi

**Dasar naskah:** Bab III tahap 12; Algoritma 2.1; Tabel 3.2 (s = 10, G_max = 4). Hanya membaca hasil Tahap 10–12.1; tanpa GPU; data uji tidak dibaca. Sel 1/2 menampilkan *global best* PSO dalam nilai asli (lr, wr, wd), selisih *fitness*-nya dengan skema *default* dan terbaik *random search* (deskriptif di data validasi, bukan uji signifikansi), serta tabel per iterasi: jumlah evaluasi kumulatif, *fitness* maksimum dan rata-rata, jumlah p_i dan g yang diperbarui, dimensi yang terkena dinding, maks |v|, *global best* akhir iterasi, dan terbaik RS pada jumlah evaluasi yang sama. Sel 2/2 menggambar kurva "terbaik sejauh ini" terhadap jumlah evaluasi untuk RS dan PSO, dengan garis mendatar skema *default*.

**Package:** `numpy`; `pandas`; `matplotlib`.

Menyimpan `bab4/tabel/tahap12_iterasi.csv` dan `bab4/gambar/tahap12_konvergensi.png` (300 dpi).

In [ ]:
# ══ Tahap 12.2 — Ringkasan PSO dan tabel per iterasi (sel 1/2) ══
# Package: numpy; pandas
import numpy as np, pandas as pd
mulai()
ya, f6, sci, R = (lambda x: "ya" if x else "tidak"), (lambda v: fmt_id(v, 6)), (lambda v: f"{v:.2e}".replace(".", ",")), CFG["ruang"]
# 1) Hasil skema default, RS (urut k), PSO (urut evaluasi), dan ringkasan pembaruan di state PSO
#    CSV dibaca persis (round_trip): pembaca bawaan bisa bergeser 1 ULP, padahal fitness dibandingkan persis dengan state (KT-16)
dft = baca_json(PATH["hasil"] / "tahap10_default.json")
rs = pd.read_csv(PATH["hasil"] / "tahap11_rs.csv", float_precision="round_trip").sort_values("k", ignore_index=True)
ps = pd.read_csv(PATH["hasil"] / "tahap12_pso.csv", float_precision="round_trip").sort_values("eval", ignore_index=True)
st = baca_json(PATH["log"] / "pso_state.json")
it = {x["t"]: x for x in st["iterasi"]}                                   # t = 0: posisi awal, tanpa pembaruan
# 2) Tabel per iterasi; RS terbaik sejauh ini pada jumlah evaluasi yang sama (pembanding anggaran setara)
tab = ps.groupby("t").agg(evaluasi=("eval", "size"), fitness_maks=("fitness", "max"), fitness_rata=("fitness", "mean"),
                          p_baru=("p_baru", "sum"), g_baru=("g_baru", "sum"), gbest=("gbest", "last")).reset_index()
tab["evaluasi_kumulatif"] = tab.evaluasi.cumsum()
tab["dimensi_dinding"] = pd.array([it[t]["dimensi_dinding"] if t in it else None for t in tab.t], dtype="Int64")
tab["maks_abs_v"] = [it[t]["maks_abs_v"] if t in it else np.nan for t in tab.t]
tab["rs_terbaik_sejauh_ini"] = rs.fitness.cummax().to_numpy()[tab.evaluasi_kumulatif - 1]
simpan_tabel(tab, "tahap12_iterasi")
gb, br = ps.loc[ps.fitness.idxmax()], rs.loc[rs.fitness.idxmax()]
# 3) Ringkasan
judul("12.2", "Ringkasan PSO dan konvergensi", masukan=["hasil/tahap12_pso.csv · log/pso_state.json", "hasil/tahap11_rs.csv · "
      "hasil/tahap10_default.json"], keluaran="bab4/tabel/tahap12_iterasi.csv")
print(f"{'t':>2}{'evaluasi':>10}{'maks':>9}{'rata-rata':>11}{'p_i baru':>10}{'g baru':>8}{'dinding':>9}{'maks |v|':>10}{'gbest':>9}"
      f"{'RS sejauh ini':>15}")
for r in tab.itertuples():
    print(f"{r.t:>2}{r.evaluasi_kumulatif:>10}{fmt_id(r.fitness_maks):>9}{fmt_id(r.fitness_rata):>11}{r.p_baru:>10}{r.g_baru:>8}"
          f"{('-' if pd.isna(r.dimensi_dinding) else int(r.dimensi_dinding)):>9}{('-' if pd.isna(r.maks_abs_v) else fmt_id(r.maks_abs_v)):>10}"
          f"{fmt_id(r.gbest):>9}{fmt_id(r.rs_terbaik_sejauh_ini):>15}")
print(f"global best PSO: lr {sci(gb.lr)} · wr {fmt_id(gb.wr)} · wd {fmt_id(gb.wd)} · skor {fmt_id(gb.skor42)} / {fmt_id(gb.skor43)} · "
      f"fitness {f6(gb.fitness)} (evaluasi ke-{int(gb['eval']) + 1}, t {int(gb.t)}, partikel {int(gb.i)})")
print(f"selisih fitness (validasi, deskriptif): PSO − default {f6(gb.fitness - dft['fitness'])} · PSO − RS {f6(gb.fitness - br.fitness)} "
      f"(RS terbaik k = {int(br.k)}: {f6(br.fitness)}; default {f6(dft['fitness'])})")
cek("iterasi · total evaluasi · gbest = maksimum kumulatif fitness · gbest akhir = fg di state", [len(tab), int(tab.evaluasi.sum()),
    ya(np.array_equal(ps.gbest, ps.fitness.cummax())), ya(tab.gbest.iloc[-1] == st["fg"] == gb.fitness)],
    [CFG["pso"]["G_max"] + 1, TARGET["12.1"]["evaluasi"], "ya", "ya"])
cek("global best di dalam ruang pencarian (lr, wr, wd) · tabel tersimpan", [ya(all(R[c][0] <= gb[c] <= R[c][1] for c in R)),
    ya((PATH["tabel"] / "tahap12_iterasi.csv").exists())], ["ya", "ya"])
selesai()

In [ ]:
# ══ Tahap 12.2 — Gambar konvergensi RS dan PSO (sel 2/2) ══
# Package: numpy; pandas; matplotlib (grafik)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
mulai()
WARNA, TEKS, GRID = {"Random search": "#2a78d6", "PSO": "#eb6834"}, "#52514e", "#e4e3df"
# 1) Terbaik sejauh ini (maksimum kumulatif fitness) menurut urutan evaluasi; skema default sebagai garis mendatar
dft = baca_json(PATH["hasil"] / "tahap10_default.json")["fitness"]
rs = pd.read_csv(PATH["hasil"] / "tahap11_rs.csv").sort_values("k")
ps = pd.read_csv(PATH["hasil"] / "tahap12_pso.csv").sort_values("eval")
kurva = {"Random search": rs.fitness.cummax().to_numpy(), "PSO": ps.fitness.cummax().to_numpy()}
n = len(ps)
# 2) Gambar: kurva tangga, titik saat terbaik naik, garis default; label langsung di ujung kanan
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.hlines(dft, 0.5, n + 0.5, color=TEKS, lw=1, zorder=2)
ax.text(n + 1, dft, f"Default {fmt_id(dft)}", va="center", fontsize=8, color=TEKS)
for nama, y in kurva.items():
    x = np.arange(1, len(y) + 1)
    naik = np.r_[True, y[1:] > y[:-1]]
    ax.step(x, y, where="post", color=WARNA[nama], lw=1.6, zorder=3, label=nama)
    ax.plot(x[naik], y[naik], "o", ms=4.5, color=WARNA[nama], mec="white", mew=1, zorder=4)
    ax.text(len(y) + 1, y[-1], f"{nama} {fmt_id(y[-1])}", va="center", fontsize=8, color=TEKS)
ax.set(xlim=(0.5, n + 12), xticks=range(0, n + 1, 10), xlabel="Jumlah evaluasi fitness",
       ylabel="Macro F1 validasi terbaik sejauh ini")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.3f}".replace(".", ",")))
ax.spines[["top", "right", "left"]].set_visible(False); ax.tick_params(colors=TEKS, length=0); ax.grid(axis="y", color=GRID, lw=0.8)
ax.legend(frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(0.45, 1.0), fontsize=8)
gambar = simpan_gambar(fig, "tahap12_konvergensi")
# 3) Ringkasan
judul("12.2", "Gambar konvergensi", masukan="hasil/tahap11_rs.csv · hasil/tahap12_pso.csv · hasil/tahap10_default.json",
      keluaran="bab4/gambar/tahap12_konvergensi.png")
for nama, y in kurva.items():
    print(f"{nama}: terbaik naik {int(np.r_[True, y[1:] > y[:-1]].sum())} kali · terakhir naik di evaluasi ke-"
          f"{int(np.flatnonzero(np.r_[True, y[1:] > y[:-1]])[-1]) + 1} · akhir {fmt_id(y[-1])} · di atas default sejak evaluasi ke-"
          f"{(int(np.argmax(y > dft)) + 1) if (y > dft).any() else '-'}")
cek("kurva tidak turun · nilai akhir = maksimum fitness (RS · PSO) · gambar tersimpan", [all(bool(np.all(np.diff(y) >= 0)) for y in
    kurva.values()), [float(kurva["Random search"][-1]), float(kurva["PSO"][-1])], gambar.exists()],
    [True, [float(rs.fitness.max()), float(ps.fitness.max())], True])
selesai()

### Tahap 12.2 — Audit koreksi CSV vs state (tambahan, diminta pengguna)

Hanya membaca. Menjelaskan ⚠️ pertama sel 1/2 ("gbest akhir = fg di state"): teks fitness di CSV sama digit demi digit dengan *state*, tetapi pembaca bawaan `pd.read_csv` dapat bergeser 1 ULP (output Colab: PSO 28 baris, RS 26 baris; maks PSO 0,9185396093525671 di *state* terbaca 0,9185396093525672), sedangkan pembaca `round_trip` persis. Posisi kenaikan terbaik-sejauh-ini dan argmax tidak berubah. Perbaikan (KT-16): sel 1/2 membaca CSV dengan `float_precision="round_trip"`.

**Package:** `json` (bawaan Python); `numpy`; `pandas`.

In [ ]:
# ══ Tahap 12.2 — Audit koreksi: fitness CSV vs state pada presisi penuh (tambahan; hanya membaca, tanpa GPU) ══
# Package: json (bawaan Python); numpy; pandas
import json
import numpy as np, pandas as pd
mulai()
ya, rp = (lambda x: "ya" if x else "tidak"), (lambda v: repr(float(v)).replace(".", ","))
baca = lambda p: json.loads(p.read_text(encoding="utf-8"))
# 1) Nilai acuan dari state (JSON, dibaca persis) dan dari CSV: teks mentah, pembaca bawaan pandas, dan pembaca round_trip
st, rst = baca(PATH["log"] / "pso_state.json"), baca(PATH["log"] / "rs_state.json")
SUMBER = {"PSO": (st["log"], PATH["hasil"] / "tahap12_pso.csv", ["fitness", "gbest"]),
          "RS": (rst["hasil"], PATH["hasil"] / "tahap11_rs.csv", ["fitness"])}   # baris CSV = urutan state (ditulis dari state)
judul("12.2", "Audit koreksi CSV vs state", masukan=["log/pso_state.json · hasil/tahap12_pso.csv", "log/rs_state.json · hasil/tahap11_rs.csv"],
      keluaran="(tidak ada; hanya membaca)")
H = {}
for n, (log, p, kol) in SUMBER.items():
    J = pd.DataFrame(log)[kol].to_numpy(float)
    teks = pd.read_csv(p, dtype=str)[kol].to_numpy()
    bawaan, rt = pd.read_csv(p)[kol].to_numpy(float), pd.read_csv(p, float_precision="round_trip")[kol].to_numpy(float)
    ulp = np.abs(bawaan - J) / np.spacing(J)
    naik = lambda x: np.flatnonzero(np.r_[True, x[1:] > x[:-1]])          # posisi terbaik-sejauh-ini naik
    H[n] = dict(teks=int((teks == np.vectorize(repr)(J)).all(axis=1).sum()), rt=int((rt == J).all(axis=1).sum()),
                beda=int((bawaan != J).any(axis=1).sum()), ulp=float(ulp.max()), maks_J=J[:, 0].max(),
                maks_C=bawaan[:, 0].max(),
                sama=ya(np.array_equal(naik(np.maximum.accumulate(J[:, 0])), naik(np.maximum.accumulate(bawaan[:, 0])))
                        and J[:, 0].argmax() == bawaan[:, 0].argmax()))
    print(f"{n:<4} baris {len(J)} · teks CSV = repr state {H[n]['teks']} · pembaca round_trip = state {H[n]['rt']} · pembaca bawaan "
          f"berbeda {H[n]['beda']} baris (maks {H[n]['ulp']:.0f} ULP) · maks state {rp(H[n]['maks_J'])} · maks CSV bawaan "
          f"{rp(H[n]['maks_C'])}")
print(f"fg state {rp(st['fg'])} · selisih maks CSV bawaan − fg {H['PSO']['maks_C'] - st['fg']:.3e}".replace(".", ",") +
      f" · ULP {np.spacing(st['fg']):.3e}".replace(".", ","))
cek("teks CSV = nilai state, digit demi digit (PSO · RS)", [H["PSO"]["teks"], H["RS"]["teks"]], [len(st["log"]), len(rst["hasil"])])
cek("pembaca round_trip = state (PSO · RS) · fg state = maks fitness state", [H["PSO"]["rt"], H["RS"]["rt"],
    ya(st["fg"] == H["PSO"]["maks_J"])], [len(st["log"]), len(rst["hasil"]), "ya"])
cek("pembaca bawaan: selisih ≤ 1 ULP · posisi kenaikan terbaik-sejauh-ini dan argmax sama (PSO · RS)", [ya(max(H["PSO"]["ulp"],
    H["RS"]["ulp"]) <= 1), H["PSO"]["sama"], H["RS"]["sama"]], ["ya"] * 3)
selesai()

## Tahap 12.3 — Ragam antar-*seed* (*program saja*)

**Dasar:** *program saja*, tidak dibahas di naskah (CLAUDE.md Bagian 6, subtahap 13.3, dengan peta nomor Bagian 10). Dari 100 konfigurasi (50 kandidat RS + 50 evaluasi PSO; keputusan no. 25): d_k = skor42 − skor43, s_k² = d_k²/2, s_gab = √(rata-rata s_k²); SE selisih dua *fitness* (masing-masing rata-rata 2 *seed*) = s_gab; zona kebetulan = ±2·SE. Tiga selisih *fitness* terbaik (RS − *default*, PSO − *default*, PSO − RS) diberi keterangan di dalam atau di luar zona. Ini pemeriksaan awal di data validasi, bukan uji signifikansi (keputusan no. 27). CSV dibaca `round_trip` (KT-16). Tanpa GPU; data uji tidak dibaca.

**Package:** `numpy`; `pandas`.

Menyimpan `hasil/tahap12_ragam.json`.

In [ ]:
# ══ Tahap 12.3 — Ragam antar-seed (program saja) ══
# Package: numpy; pandas
import numpy as np, pandas as pd
mulai()
ya, f6 = (lambda x: "ya" if x else "tidak"), (lambda v: fmt_id(v, 6))
# 1) Skor per seed dari 50 kandidat RS dan 50 evaluasi PSO (CSV dibaca persis, KT-16); fitness default dan terbaik tiap metode
rs = pd.read_csv(PATH["hasil"] / "tahap11_rs.csv", float_precision="round_trip")
ps = pd.read_csv(PATH["hasil"] / "tahap12_pso.csv", float_precision="round_trip")
dft = baca_json(PATH["hasil"] / "tahap10_default.json")["fitness"]
d = np.r_[(rs.skor42 - rs.skor43).to_numpy(), (ps.skor42 - ps.skor43).to_numpy()]
# 2) s_k^2 = d_k^2 / 2; s_gab = akar rata-rata s_k^2; SE selisih dua fitness (masing-masing rata-rata 2 seed) = s_gab; zona = ±2·SE
s_gab = float(np.sqrt(np.mean(d ** 2 / 2)))
se, zona = s_gab, 2 * s_gab
fr, fp = float(rs.fitness.max()), float(ps.fitness.max())
sel = {"RS − default": fr - dft, "PSO − default": fp - dft, "PSO − RS": fp - fr}
dalam = {k: bool(abs(v) <= zona) for k, v in sel.items()}
p = simpan_json({"n_konfigurasi": int(len(d)), "n_rs": int(len(rs)), "n_pso": int(len(ps)), "s_gab": s_gab, "se_selisih": se,
                 "zona": [-zona, zona], "fitness": {"default": dft, "rs_terbaik": fr, "pso_terbaik": fp}, "selisih": sel,
                 "di_dalam_zona": dalam, "catatan": "pemeriksaan awal di data validasi; bukan uji signifikansi"},
                PATH["hasil"] / "tahap12_ragam.json")
# 3) Ringkasan
judul("12.3", "Ragam antar-seed (program saja)", masukan="hasil/tahap11_rs.csv · hasil/tahap12_pso.csv · hasil/tahap10_default.json",
      keluaran="hasil/tahap12_ragam.json")
print(f"konfigurasi {len(d)} (RS {len(rs)} + PSO {len(ps)}) · s_gab {fmt_id(s_gab)} · SE selisih {fmt_id(se)} · "
      f"zona kebetulan ±{fmt_id(zona)}")
print(f"{'Selisih fitness':<16}{'nilai':>11}  di dalam zona?")
for k, v in sel.items():
    print(f"{k:<16}{f6(v):>11}  {ya(dalam[k])}")
print("Ini pemeriksaan awal di data validasi pada konfigurasi terbaik yang sudah terpilih, bukan uji signifikansi; keputusan "
      "statistik di Tahap 15.")
cek("konfigurasi (RS + PSO) · selisih skor per seed lengkap · json tersimpan", [len(d), ya(np.isfinite(d).all()), ya(p.exists())],
    [TARGET["11"]["kandidat"] + TARGET["12.1"]["evaluasi"], "ya", "ya"])
selesai()

### Tahap 12.3 — Audit ragam antar-*seed* (tambahan, diminta pengguna)

Hanya membaca, tanpa GPU; data uji tidak dibaca; tidak ada berkas yang ditulis. Menghitung ulang dari *state* JSON (bukan CSV) pada presisi penuh: (1) skor *seed* 42 dan 43 lengkap, di [0, 1], dan *fitness* = rata-rata keduanya untuk 50 kandidat RS dan 50 evaluasi PSO, serta CSV = *state* untuk `skor42`, `skor43`, `fitness`; (2) s_gab dengan dua cara, yaitu rumus spesifikasi d_k²/2 dan varians sampel tiap pasangan (ddof = 1, 1 df per pasangan, 100 df); (3) konfigurasi terbaik (RS argmax, PSO = fg) dan tiga selisih *fitness*; (4) `tahap12_ragam.json` = hitung ulang, dibandingkan persis. Diagnostik deskriptif, tidak mengubah metode: s per metode, rata-rata d (geseran tetap *seed* 42 vs 43), s dengan d dipusatkan, rasio selisih/SE, dan pengali pembanding t(0,975; 100) serta Bonferroni m = 3.

**Package:** `json` (bawaan Python); `numpy`; `pandas`; `scipy`.

In [ ]:
# ══ Tahap 12.3 — Audit: hitung ulang ragam antar-seed dari state (tambahan; hanya membaca, tanpa GPU) ══
# Package: json (bawaan Python); numpy; pandas; scipy
import json
import numpy as np, pandas as pd
from scipy import stats
mulai()
ya, rp = (lambda x: "ya" if x else "tidak"), (lambda v: repr(float(v)).replace(".", ","))
baca, K = (lambda p: json.loads(p.read_text(encoding="utf-8"))), ["skor42", "skor43", "fitness"]
# 1) Sumber: state RS/PSO (JSON, persis), CSV yang dibaca sel 12.3 (round_trip), json default, dan tahap12_ragam.json
st_p, dj, J = baca(PATH["log"] / "pso_state.json"), baca(PATH["hasil"] / "tahap10_default.json"), baca(PATH["hasil"] / "tahap12_ragam.json")
rs, ps = pd.DataFrame(baca(PATH["log"] / "rs_state.json")["hasil"]), pd.DataFrame(st_p["log"])
csv = [pd.read_csv(PATH["hasil"] / f, float_precision="round_trip")[K].to_numpy() for f in ("tahap11_rs.csv", "tahap12_pso.csv")]
sama_csv = [int((c == d[K].to_numpy()).all(axis=1).sum()) for c, d in zip(csv, (rs, ps))]
sah = lambda d: int((d[K].notna().all(axis=1) & d.skor42.between(0, 1) & d.skor43.between(0, 1)
                     & ((d.fitness - (d.skor42 + d.skor43) / 2).abs() <= 1e-15)).sum())   # dua skor ada; fitness = rata-rata
# 2) Hitung ulang: (a) rumus spesifikasi s_k² = d_k²/2; (b) varians sampel tiap pasangan (ddof = 1, 1 df) dirata-ratakan
S = np.r_[rs[K[:2]].to_numpy(), ps[K[:2]].to_numpy()]
d = S[:, 0] - S[:, 1]
s_a, s_b = float(np.sqrt(np.mean(d ** 2 / 2))), float(np.sqrt(np.var(S, axis=1, ddof=1).mean()))
s_m = {n: float(np.sqrt(np.mean(x ** 2 / 2))) for n, x in (("RS", d[:len(rs)]), ("PSO", d[len(rs):]))}
s_c = float(np.sqrt(np.var(d, ddof=1) / 2))       # diagnostik saja: d dipusatkan (tanpa geseran tetap seed 42 vs 43)
br, bp, fd = rs.loc[rs.fitness.idxmax()], ps.loc[ps.fitness.idxmax()], dj["fitness"]
sel = {"RS − default": br.fitness - fd, "PSO − default": bp.fitness - fd, "PSO − RS": bp.fitness - br.fitness}
H = {"n_konfigurasi": len(S), "n_rs": len(rs), "n_pso": len(ps), "s_gab": s_a, "se_selisih": s_a, "zona": [-2 * s_a, 2 * s_a],
     "fitness": {"default": fd, "rs_terbaik": br.fitness, "pso_terbaik": bp.fitness}, "selisih": sel,
     "di_dalam_zona": {k: bool(abs(v) <= 2 * s_a) for k, v in sel.items()}}
num = lambda o: np.array([o[k] for k in list(H)[:5]] + list(o["zona"]) + list(o["fitness"].values()) + list(o["selisih"].values()), float)
kunci = list(J) == list(H) + ["catatan"] and all(list(J[k]) == list(H[k]) for k in ("fitness", "selisih", "di_dalam_zona"))
# 3) Ringkasan
judul("12.3", "Audit ragam antar-seed", masukan=["log/rs_state.json · log/pso_state.json · hasil/tahap10_default.json",
      "hasil/tahap11_rs.csv · hasil/tahap12_pso.csv · hasil/tahap12_ragam.json"], keluaran="(tidak ada; hanya membaca)")
print(f"pasangan {len(S)} (RS {len(rs)} + PSO {len(ps)}) · df {len(S)} (1 per pasangan) · s_gab (a) {rp(s_a)} · (b) {rp(s_b)} · "
      f"|a − b| {abs(s_a - s_b):.1e}".replace(".", ","))
print(f"diagnostik: s RS {fmt_id(s_m['RS'], 6)} · s PSO {fmt_id(s_m['PSO'], 6)} · rata-rata d (42 − 43) {fmt_id(d.mean(), 6)} "
      f"(SE {fmt_id(d.std(ddof=1) / np.sqrt(len(d)), 6)}) · s dengan d dipusatkan {fmt_id(s_c, 6)} · maks |d| {fmt_id(np.abs(d).max(), 4)}")
print(f"terbaik: RS k {int(br.k)} {rp(br.fitness)} · PSO t {int(bp.t)} p {int(bp.i)} {rp(bp.fitness)} · default {rp(fd)}")
print(f"pengali pembanding (bukan metode): 2 · t(0,975; {len(S)}) {fmt_id(stats.t.ppf(0.975, len(S)), 3)} · Bonferroni m = 3: "
      f"t(1 − 0,05/6; {len(S)}) {fmt_id(stats.t.ppf(1 - 0.05 / 6, len(S)), 3)}")
for k, v in sel.items():
    print(f"  {k:<14}{rp(v):>24} = {fmt_id(v / s_a, 2):>5} SE · di dalam ±2 SE? {ya(abs(v) <= 2 * s_a)}")
cek("skor sah (RS · PSO) · CSV = state (RS · PSO) · RS k 0–49 · PSO (t, p) unik", [sah(rs), sah(ps), *sama_csv,
    ya(rs.k.tolist() == list(range(len(rs)))), ya(len(set(zip(ps.t, ps.i))) == len(ps))], [TARGET["11"]["kandidat"],
    TARGET["12.1"]["evaluasi"], TARGET["11"]["kandidat"], TARGET["12.1"]["evaluasi"], "ya", "ya"])
cek("|s_gab (a) − (b)| ≤ 1e-15 · default = rata-rata 2 seed · terbaik tunggal (RS · PSO) · PSO terbaik = fg", [ya(abs(s_a - s_b) <= 1e-15),
    ya(abs(fd - (dj["skor42"] + dj["skor43"]) / 2) <= 1e-15), ya((rs.fitness == br.fitness).sum() == 1),
    ya((ps.fitness == bp.fitness).sum() == 1), ya(bp.fitness == st_p["fg"])], ["ya"] * 5)
cek("json = hitung ulang: kunci sama · nilai berbeda (persis) · di_dalam_zona sama · catatan bukan uji signifikansi",
    [ya(kunci), int((num(J) != num(H)).sum()), ya(J["di_dalam_zona"] == H["di_dalam_zona"]),
     ya("bukan uji signifikansi" in J["catatan"])], ["ya", 0, "ya", "ya"])
selesai()

## Tahap 13 — Cek determinisme lingkungan sesi ini (tambahan, sebelum model akhir)

**Dasar:** Tahap 0.1 sesi 24-10-2026 memberi ⚠️ (keputusan no. 2): `transformers` 5.20.1 → 5.20.2 dan GPU A100-SXM4-40GB → A100-SXM4-80GB dibanding sesi A100 pertama. Cek determinisme Tahap 13 (CLAUDE.md Bagian 6, subtahap 14: selisih ≤ 1e-4) diuji lebih dulu dengan satu pelatihan: skema *default*, *seed* 42, data dan label sama dengan `fitness()`. Kurva *Macro F1* per *epoch* dibandingkan dengan `hasil/tahap10_default.json` pada presisi penuh. Hasil pelatihan tidak disimpan; data uji tidak dibaca. GPU ±1 menit.

**Package:** `importlib`, `importlib.metadata` (bawaan Python); `numpy`; `pandas`; `torch`; `lib.wondr_latih` (`torch`, `transformers`, `scikit-learn`).

In [ ]:
# ══ Tahap 13 — Cek determinisme di lingkungan sesi ini (tambahan; tidak menyimpan apa pun; GPU ±1 menit) ══
# Package: importlib, importlib.metadata (bawaan Python); numpy; pandas; torch; lib.wondr_latih (torch, transformers, scikit-learn)
import importlib, importlib.metadata as im
import numpy as np, pandas as pd, torch
import lib.wondr_latih as wl
importlib.reload(wl)
mulai()
ya, rp = (lambda x: "ya" if x else "tidak"), (lambda v: repr(float(v)).replace(".", ","))
wl.pakai_konfig(CFG)
d, s = CFG["default"], CFG["seed_fitness"][0]                        # skema default, seed 42
acuan = baca_json(PATH["hasil"] / "tahap10_default.json")["per_seed"][str(s)]
# 1) Satu pelatihan dengan data dan label yang sama dengan fitness() (Tahap 9.3, label = label_akhir); hasil tidak disimpan
baca = lambda b: pd.read_parquet(PATH["hasil"] / f"tahap09_{b}.parquet").assign(label=lambda x: x["label_akhir"])
h = wl.latih(baca("latih"), d["lr"], d["wr"], d["wd"], s, df_val=baca("validasi"))
# 2) Kurva Macro F1 per epoch dibandingkan dengan Tahap 10 pada presisi penuh
a, b = np.array(h["f1_per_epoch"]), np.array(acuan["f1_per_epoch"])
n_sama = len(a) == len(b)
maks = float(np.abs(a - b).max()) if n_sama else float("inf")
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada"
# 3) Ringkasan
judul("13", "Cek determinisme lingkungan sesi ini", masukan="hasil/tahap09_{latih,validasi}.parquet · hasil/tahap10_default.json",
      keluaran="(tidak ada; hasil pelatihan tidak disimpan)")
print(f"sesi ini: {gpu} · torch {torch.__version__} · transformers {im.version('transformers')} · tokenizers {im.version('tokenizers')}")
print(f"skema default (lr {d['lr']} · wr {fmt_id(d['wr'], 2)} · wd {fmt_id(d['wd'], 2)}) · seed {s} · {fmt_id(h['detik'] / 60, 1)} menit · "
      f"{h['perangkat']}")
print(f"{'epoch':<7}{'sesi ini':>22}{'Tahap 10':>22}  selisih")
for i in range(max(len(a), len(b))):
    x, y = (a[i] if i < len(a) else np.nan), (b[i] if i < len(b) else np.nan)
    print(f"{i + 1:<7}{rp(x):>22}{rp(y):>22}  {abs(x - y):.1e}".replace(".", ",").replace("nan", "-"))
print(f"epoch terbaik: sesi ini {h['epoch_terbaik']} · Tahap 10 {acuan['epoch_terbaik']}")
cek("kurva per epoch identik (presisi penuh) · selisih maks ≤ 1e-4 · epoch terbaik sama · perangkat",
    [ya(n_sama and maks == 0), ya(n_sama and maks <= 1e-4), h["epoch_terbaik"], h["perangkat"]],
    ["ya", "ya", acuan["epoch_terbaik"], "cuda"])
selesai()